# Champion + counsel + JUDGE inference pipeline
This notebook is extracted from the supplied CS781 notebook. It preserves the original implementation and skips cells 362–363 (training experiments).

Before running:
- Confirm the Kaggle datasets are attached.
- This pipeline uses the original `champion_recovery.pt` and its thresholds.
- It writes a new candidate only if the destination does not already exist.
- Review the fold-2 score before accepting the fold-3 output.


In [ ]:

import torch
import torch.nn as nn
from transformers import GPT2Config, GPT2Model
import json
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def find_artifact(filename, preferred_parts=()):
    roots = [Path("/kaggle/input"), Path("/kaggle/working")]
    matches = []
    for root in roots:
        if root.exists():
            matches.extend(
                p for p in root.rglob(filename) if p.is_file()
            )

    matches = sorted(set(matches))
    if not matches:
        raise FileNotFoundError(
            f"Cannot find {filename} under /kaggle/input or "
            "/kaggle/working. Attach or download the dataset containing it."
        )

    preferred = [
        p for p in matches
        if all(part in str(p) for part in preferred_parts)
    ]
    chosen = preferred[0] if preferred else matches[0]
    print(f"{filename} -> {chosen}")
    return chosen

RECOVERY_PATH = find_artifact(
    "champion_recovery.pt", ("cs781_recovery",)
)
PHASE1_PATH = find_artifact(
    "checkpoint_latest.pt", ("cs781-phase-1-checkpoints",)
)
THRESHOLDS_PATH = find_artifact(
    "champion_thresholds_recovery.json", ("cs781_recovery",)
)

recovery = torch.load(
    RECOVERY_PATH, map_location="cpu", weights_only=False
)
phase1_ckpt = torch.load(
    PHASE1_PATH, map_location="cpu", weights_only=False
)

BIO_LABELS = recovery["bio_labels"]
label2id = {label: i for i, label in enumerate(BIO_LABELS)}
id2label = {i: label for i, label in enumerate(BIO_LABELS)}

cfg = phase1_ckpt.get("model_config")
if cfg is None:
    raise KeyError("Phase 1 checkpoint has no model_config.")

gpt2_config = GPT2Config(
    vocab_size=cfg["vocab_size"],
    n_positions=cfg["context_length"],
    n_ctx=cfg["context_length"],
    n_embd=cfg["hidden_size"],
    n_layer=cfg["num_layers"],
    n_head=cfg["num_heads"],
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
)

class LegalNERModel(nn.Module):
    def __init__(self, pretrained_lm, num_labels):
        super().__init__()
        self.encoder = pretrained_lm.transformer
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(
            pretrained_lm.config.hidden_size, num_labels
        )

    def forward(self, input_ids, attention_mask=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        hidden = self.dropout(outputs.last_hidden_state)
        return self.classifier(hidden)

# Reconstruct the Phase 1 GPT-2 encoder.
phase1_lm = GPT2Model(gpt2_config)
phase1_state = {
    k.removeprefix("transformer."): v
    for k, v in phase1_ckpt["model_state_dict"].items()
    if k.startswith("transformer.")
}
phase1_lm.load_state_dict(phase1_state, strict=True)

# Build NER architecture and load the trained recovery weights.
lm_wrapper = type("LMWrapper", (), {
    "transformer": phase1_lm,
    "config": gpt2_config
})()

champion_model = LegalNERModel(
    pretrained_lm=lm_wrapper,
    num_labels=len(BIO_LABELS)
).to(device)

load_result = champion_model.load_state_dict(
    recovery["model_state_dict"], strict=True
)
champion_model.eval()
ner_model = champion_model

with open(THRESHOLDS_PATH, "r", encoding="utf-8") as f:
    champion_thresholds_fusion = json.load(f)

print("Checkpoint load:", load_result)
print("Device:", device)
print("BIO labels:", len(BIO_LABELS))
print("Threshold data type:", type(champion_thresholds_fusion).__name__)

# Shape check only; this does not evaluate model quality.
with torch.inference_mode():
    test_ids = torch.zeros((1, 16), dtype=torch.long, device=device)
    test_mask = torch.ones_like(test_ids)
    test_logits = champion_model(test_ids, test_mask)

assert test_logits.shape == (1, 16, len(BIO_LABELS))
print("Test logits shape:", tuple(test_logits.shape))
print("Champion loaded and shape check passed.")


In [ ]:

import torch

recovery_state = recovery["model_state_dict"]
loaded_state = champion_model.state_dict()

assert set(recovery_state.keys()) == set(loaded_state.keys())

max_param_diff = max(
    (loaded_state[name].detach().cpu() - recovery_state[name]).abs().max().item()
    for name in loaded_state
)

champion_model.eval()
with torch.inference_mode():
    x = torch.zeros((1, 16), dtype=torch.long, device=device)
    mask = torch.ones_like(x)

    logits1 = champion_model(x, mask)
    logits2 = champion_model(x, mask)

print("Maximum parameter difference:", max_param_diff)
print("Logits shape:", tuple(logits1.shape))
print("Repeated-inference difference:", (logits1 - logits2).abs().max().item())

assert max_param_diff == 0.0
assert torch.allclose(logits1, logits2, atol=1e-6)

print("Recovery verification passed.")


In [ ]:

from pathlib import Path
import json
from tokenizers import Tokenizer

def find_file(filename):
    roots = [Path("/kaggle/input"), Path("/kaggle/working")]
    matches = []
    for root in roots:
        if root.exists():
            matches.extend(
                p for p in root.rglob(filename) if p.is_file()
            )

    matches = sorted(set(matches))
    if not matches:
        raise FileNotFoundError(
            f"{filename} was not found. Check the attached Kaggle dataset."
        )

    print(f"\nFound {filename}:")
    for p in matches:
        print(" ", p)
    return matches[0]

tokenizer_path = find_file("legal_bpe_32k.json")
fold1_text_path = find_file("fold1-text.json")
fold1_gold_path = find_file("fold1-gold.json")
fold2_text_path = find_file("fold2-text.json")
fold2_gold_path = find_file("fold2-gold.json")
fold3_text_path = find_file("fold3-text.json")

tokenizer = Tokenizer.from_file(str(tokenizer_path))

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_texts = load_json(fold1_text_path)
train_gold = load_json(fold1_gold_path)
val_texts = load_json(fold2_text_path)
val_gold = load_json(fold2_gold_path)
test_texts = load_json(fold3_text_path)

print("\nTokenizer and datasets loaded successfully.")
print("Train:", len(train_texts), "documents")
print("Validation:", len(val_texts), "documents")
print("Test:", len(test_texts), "documents")


In [ ]:

import numpy as np
from collections import Counter, defaultdict
from tqdm.auto import tqdm

champion_model.eval()

WEAK_LABELS = ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]
MAX_WINDOW = 1024
WINDOW_STRIDE_DIAG = 768

assert len(BIO_LABELS) == 25
assert all(f"B-{lab}" in label2id for lab in WEAK_LABELS)

# Accumulate logits over overlapping windows, one window at a time.
doc_logits = {}
with torch.inference_mode():
    for doc_id, text in tqdm(val_texts.items(), desc="Scoring validation"):
        enc = tokenizer.encode(text, add_special_tokens=False)
        ids = enc.ids
        n = len(ids)
        if n == 0:
            doc_logits[doc_id] = (enc.offsets, np.zeros((0, len(BIO_LABELS))))
            continue

        sums = np.zeros((n, len(BIO_LABELS)), dtype=np.float32)
        counts = np.zeros(n, dtype=np.float32)

        starts = list(range(0, n, WINDOW_STRIDE_DIAG))
        for start in starts:
            end = min(start + MAX_WINDOW, n)
            if start >= n:
                break

            x = torch.tensor(
                [ids[start:end]], dtype=torch.long, device=device
            )
            mask = torch.ones_like(x)
            logits = champion_model(x, mask)[0].float().cpu().numpy()

            sums[start:end] += logits
            counts[start:end] += 1

            if end == n:
                break

        avg_logits = sums / np.maximum(counts[:, None], 1)
        doc_logits[doc_id] = (enc.offsets, avg_logits)

# Summarize token-level predictions inside annotated weak-label spans.
top_counts = {lab: Counter() for lab in WEAK_LABELS}
target_probs = defaultdict(list)
entity_count = Counter()
token_count = Counter()

for doc_id, text in val_texts.items():
    offsets, logits = doc_logits[doc_id]
    if len(offsets) == 0:
        continue

    # Stable softmax over BIO classes.
    shifted = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(shifted)
    probs /= probs.sum(axis=1, keepdims=True)
    pred_ids = logits.argmax(axis=1)

    for ann in val_gold[doc_id]:
        start, end, label = int(ann[0]), int(ann[1]), ann[2]
        if label not in WEAK_LABELS:
            continue

        token_indices = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < end and b > start
        ]
        if not token_indices:
            continue

        entity_count[label] += 1
        target_probs[label].append(
            float(probs[token_indices, label2id[f"B-{label}"]].mean())
        )

        for i in token_indices:
            token_count[label] += 1
            top_counts[label][BIO_LABELS[int(pred_ids[i])]] += 1

print("\nWEAK-LABEL VALIDATION DIAGNOSTIC")
for lab in WEAK_LABELS:
    mean_prob = (
        float(np.mean(target_probs[lab])) if target_probs[lab] else float("nan")
    )
    print(f"\n{lab}: {entity_count[lab]} gold entities")
    print(f"Mean token probability for B-{lab}: {mean_prob:.4f}")
    print(f"Gold-span tokens: {token_count[lab]}")
    print("Top predicted BIO labels:", top_counts[lab].most_common(8))


In [ ]:

import torch
import gc

gc.collect()
torch.cuda.empty_cache()

print("Device:", torch.cuda.get_device_name(0))
print("GPU allocated:", round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GiB")
print("GPU reserved:", round(torch.cuda.memory_reserved(0) / 1024**3, 2), "GiB")
print("GPU total:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GiB")

print("\nChampion training mode:", champion_model.training)
print("Champion parameters:", sum(p.numel() for p in champion_model.parameters()))
print("Champion dtype:", next(champion_model.parameters()).dtype)
print("Champion device:", next(champion_model.parameters()).device)

print("\nExisting saved recovery:", 
      __import__("os").path.exists("/kaggle/working/cs781_backup_download/cs781_recovery/champion_recovery.pt"))


In [ ]:

import copy
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

# Keep an independent CPU reference for exact recovery.
champion_reference_cpu = {
    k: v.detach().cpu().clone()
    for k, v in champion_model.state_dict().items()
}

# Train a separate copy; leave champion_model untouched.
targeted_model = copy.deepcopy(champion_model).to(device)
targeted_model.train()

# Only AdamW state for trainable parameters will be created lazily
# during the first optimizer step.
targeted_optimizer = torch.optim.AdamW(
    targeted_model.parameters(),
    lr=1e-6,
    weight_decay=0.01
)

# Conservative weak-label weighting.
weak_labels = {"RESP", "A.COUNSEL", "R.COUNSEL", "WIT"}
class_weights = torch.ones(len(BIO_LABELS), dtype=torch.float32, device=device)
class_weights[label2id["O"]] = 0.5

for lab in weak_labels:
    class_weights[label2id[f"B-{lab}"]] = 4.0
    class_weights[label2id[f"I-{lab}"]] = 4.0

targeted_criterion = torch.nn.CrossEntropyLoss(
    weight=class_weights,
    ignore_index=-100
)

print("Separate training copy created.")
print("Champion remains in eval mode:", not champion_model.training)
print("Targeted copy in training mode:", targeted_model.training)
print("Learning rate:", targeted_optimizer.param_groups[0]["lr"])
print("GPU allocated:", round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GiB")
print("Champion recovery reference tensors:", len(champion_reference_cpu))


In [ ]:

import random
from collections import Counter

MAX_LEN_PILOT = 256
STRIDE_PILOT = 192
IGNORE_INDEX = -100

def make_pilot_windows(texts, gold, tokenizer,
                       max_len=MAX_LEN_PILOT, stride=STRIDE_PILOT):
    windows = []
    label_counts = Counter()

    for doc_id, text in texts.items():
        enc = tokenizer.encode(text, add_special_tokens=False)
        ids, offsets = enc.ids, enc.offsets
        if not ids:
            continue

        token_labels = ["O"] * len(ids)

        # Assign BIO labels from gold character spans.
        for ann in gold.get(doc_id, []):
            start, end, label = int(ann[0]), int(ann[1]), ann[2]
            overlapping = [
                i for i, (a, b) in enumerate(offsets)
                if b > a and a < end and b > start
            ]
            if not overlapping or label not in weak_labels and label not in {
                "APP", "JUDGE", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
            }:
                continue

            for j, tok_i in enumerate(overlapping):
                token_labels[tok_i] = ("B-" if j == 0 else "I-") + label

        start = 0
        while start < len(ids):
            end = min(start + max_len, len(ids))
            win_ids = ids[start:end]
            win_tags = token_labels[start:end]

            # Avoid duplicate supervision in overlapping context.
            supervised_start = 0 if start == 0 else max(0, max_len - stride)
            train_tags = [
                label2id[tag] if i >= supervised_start else IGNORE_INDEX
                for i, tag in enumerate(win_tags)
            ]

            if any(tag != IGNORE_INDEX for tag in train_tags):
                windows.append({
                    "input_ids": win_ids,
                    "labels": train_tags
                })
                label_counts.update(
                    BIO_LABELS[tag] for tag in train_tags if tag != IGNORE_INDEX
                )

            if end == len(ids):
                break
            start += stride

    return windows, label_counts

pilot_windows, pilot_label_counts = make_pilot_windows(
    train_texts, train_gold, tokenizer
)

print("Pilot windows:", len(pilot_windows))
print("Supervised tokens:", sum(pilot_label_counts.values()))
print("Weak-label supervised token counts:")
for lab in ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]:
    print(
        lab,
        pilot_label_counts[f"B-{lab}"] + pilot_label_counts[f"I-{lab}"]
    )

print("\nExample window length:", len(pilot_windows[0]["input_ids"]))
print("Example supervised labels:", sum(
    x != IGNORE_INDEX for x in pilot_windows[0]["labels"]
))


In [ ]:

from collections import Counter

audit = Counter()
bad_examples = []

for doc_id, text in train_texts.items():
    enc = tokenizer.encode(text, add_special_tokens=False)
    offsets = enc.offsets
    annotations = sorted(
        train_gold.get(doc_id, []),
        key=lambda x: (int(x[0]), int(x[1]))
    )

    previous_end = -1

    for ann in annotations:
        start, end, label = int(ann[0]), int(ann[1]), ann[2]

        if start < previous_end:
            audit["overlapping_annotations"] += 1
            if len(bad_examples) < 5:
                bad_examples.append((doc_id, "overlap", ann))

        previous_end = max(previous_end, end)

        if start < 0 or end > len(text) or start >= end:
            audit["invalid_character_span"] += 1
            continue

        covered = [
            (a, b) for a, b in offsets
            if b > a and a < end and b > start
        ]

        if not covered:
            audit["no_token_coverage"] += 1
            continue

        if covered[0][0] != start or covered[-1][1] != end:
            audit["boundary_not_token_aligned"] += 1
            if len(bad_examples) < 5:
                bad_examples.append(
                    (doc_id, "boundary", (start, end, label, covered[0], covered[-1]))
                )
        else:
            audit["exactly_token_aligned"] += 1

print("Training annotation audit:")
for key in [
    "overlapping_annotations",
    "invalid_character_span",
    "no_token_coverage",
    "boundary_not_token_aligned",
    "exactly_token_aligned"
]:
    print(f"{key}: {audit[key]}")

print("\nExamples of issues:")
for item in bad_examples:
    print(item)


In [ ]:

from collections import defaultdict

examples_by_label = defaultdict(list)

for doc_id, text in train_texts.items():
    offsets = tokenizer.encode(text, add_special_tokens=False).offsets

    for ann in train_gold.get(doc_id, []):
        start, end, label = int(ann[0]), int(ann[1]), ann[2]

        if label not in {"RESP", "A.COUNSEL", "R.COUNSEL", "WIT"}:
            continue

        covered = [
            (a, b) for a, b in offsets
            if b > a and a < end and b > start
        ]
        if not covered:
            continue

        if covered[0][0] != start or covered[-1][1] != end:
            if len(examples_by_label[label]) < 4:
                token_start, token_end = covered[0][0], covered[-1][1]
                examples_by_label[label].append({
                    "doc_id": doc_id,
                    "annotation": (start, end, label),
                    "annotated_text": repr(text[start:end]),
                    "token_covered_text": repr(text[token_start:token_end]),
                    "left_difference": repr(text[min(start, token_start):max(start, token_start)]),
                    "right_difference": repr(text[min(end, token_end):max(end, token_end)])
                })

for label in ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]:
    print(f"\n--- {label} ---")
    for ex in examples_by_label[label]:
        for key, value in ex.items():
            print(f"{key}: {value}")
        print()


In [ ]:

from collections import Counter, defaultdict

weak_labels = {"RESP", "A.COUNSEL", "R.COUNSEL", "WIT"}
boundary_stats = defaultdict(Counter)
boundary_examples = defaultdict(list)

for doc_id, text in train_texts.items():
    offsets = tokenizer.encode(text, add_special_tokens=False).offsets

    for ann in train_gold.get(doc_id, []):
        start, end, label = int(ann[0]), int(ann[1]), ann[2]
        if label not in weak_labels:
            continue

        covered = [
            (a, b) for a, b in offsets
            if b > a and a < end and b > start
        ]
        if not covered:
            boundary_stats[label]["no_token_coverage"] += 1
            continue

        first_start, last_end = covered[0][0], covered[-1][1]

        left_extra = max(0, start - first_start)
        right_extra = max(0, last_end - end)

        if left_extra == 0 and right_extra == 0:
            boundary_stats[label]["exact"] += 1
        elif (
            text[first_start:start].strip() == ""
            and text[end:last_end].strip() == ""
        ):
            boundary_stats[label]["whitespace_only"] += 1
        else:
            boundary_stats[label]["non_whitespace_mismatch"] += 1
            if len(boundary_examples[label]) < 3:
                boundary_examples[label].append({
                    "doc_id": doc_id,
                    "span": (start, end),
                    "entity": repr(text[start:end]),
                    "token_covered": repr(text[first_start:last_end]),
                    "left_extra": repr(text[first_start:start]),
                    "right_extra": repr(text[end:last_end])
                })

print("Weak entity boundary audit:")
for lab in sorted(weak_labels):
    print(f"\n{lab}: {dict(boundary_stats[lab])}")
    for ex in boundary_examples[lab]:
        print(ex)


In [ ]:

from collections import Counter

MAX_LEN_PILOT = 256
STRIDE_PILOT = 192
IGNORE_INDEX = -100

def make_unique_pilot_windows(texts, gold, tokenizer,
                              max_len=MAX_LEN_PILOT, stride=STRIDE_PILOT):
    windows = []
    label_counts = Counter()

    for doc_id, text in texts.items():
        enc = tokenizer.encode(text, add_special_tokens=False)
        ids, offsets = enc.ids, enc.offsets
        n = len(ids)
        if n == 0:
            continue

        tags = ["O"] * n

        # Assign each annotation to the tokens overlapping its character span.
        # Gold annotations do not overlap, as verified by the earlier audit.
        for ann in gold.get(doc_id, []):
            start, end, label = int(ann[0]), int(ann[1]), ann[2]
            token_indices = [
                i for i, (a, b) in enumerate(offsets)
                if b > a and a < end and b > start
            ]
            if not token_indices:
                continue

            for j, i in enumerate(token_indices):
                tags[i] = ("B-" if j == 0 else "I-") + label

        # Each token is loss-supervised exactly once: the window in which
        # that token is first encountered owns its label.
        supervised = [False] * n
        start = 0

        while start < n:
            end = min(start + max_len, n)
            win_ids = ids[start:end]
            win_labels = []

            for global_i in range(start, end):
                if not supervised[global_i]:
                    win_labels.append(label2id[tags[global_i]])
                    supervised[global_i] = True
                    label_counts[tags[global_i]] += 1
                else:
                    win_labels.append(IGNORE_INDEX)

            if any(x != IGNORE_INDEX for x in win_labels):
                windows.append({
                    "doc_id": doc_id,
                    "start": start,
                    "input_ids": win_ids,
                    "labels": win_labels
                })

            if end == n:
                break
            start += stride

        assert all(supervised), f"Some tokens were not assigned in {doc_id}"

    return windows, label_counts

pilot_windows, pilot_label_counts = make_unique_pilot_windows(
    train_texts, train_gold, tokenizer
)

print("Corrected windows:", len(pilot_windows))
print("Uniquely supervised tokens:", sum(pilot_label_counts.values()))
print("Label counts:")
for lab in ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]:
    print(lab, pilot_label_counts[f"B-{lab}"] + pilot_label_counts[f"I-{lab}"])

print("First window length:", len(pilot_windows[0]["input_ids"]))
print("First window supervised tokens:",
      sum(x != IGNORE_INDEX for x in pilot_windows[0]["labels"]))


In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader
import gc

class PilotNERDataset(Dataset):
    def __init__(self, windows):
        self.windows = windows

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return self.windows[idx]

def pilot_collate(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, attention_masks, labels = [], [], []

    for item in batch:
        n = len(item["input_ids"])
        pad = max_len - n
        input_ids.append(item["input_ids"] + [0] * pad)
        attention_masks.append([1] * n + [0] * pad)
        labels.append(item["labels"] + [IGNORE_INDEX] * pad)

    return (
        torch.tensor(input_ids, dtype=torch.long),
        torch.tensor(attention_masks, dtype=torch.long),
        torch.tensor(labels, dtype=torch.long)
    )

pilot_loader = DataLoader(
    PilotNERDataset(pilot_windows),
    batch_size=1,
    shuffle=True,
    collate_fn=pilot_collate
)

# Verify one forward/backward pass only. No optimizer step yet.
targeted_model.train()
targeted_optimizer.zero_grad(set_to_none=True)
gc.collect()
torch.cuda.empty_cache()

input_ids, attention_mask, labels = next(iter(pilot_loader))
input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)
labels = labels.to(device)

try:
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits = targeted_model(input_ids, attention_mask)
        loss = targeted_criterion(
            logits.reshape(-1, len(BIO_LABELS)),
            labels.reshape(-1)
        )

    print("Batch shape:", tuple(input_ids.shape))
    print("Loss finite:", bool(torch.isfinite(loss).item()))
    print("Loss:", float(loss.detach().cpu()))
    print("Logits shape:", tuple(logits.shape))

    loss.backward()
    print("Backward pass: successful")
    print("GPU allocated after backward:",
          round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GiB")
finally:
    targeted_optimizer.zero_grad(set_to_none=True)
    del input_ids, attention_mask, labels
    if "logits" in locals():
        del logits
    if "loss" in locals():
        del loss
    gc.collect()
    torch.cuda.empty_cache()

print("Safety check complete; no optimizer step was taken.")


In [ ]:

import os
import gc
import math
import torch
from tqdm.auto import tqdm

MAX_OPTIMIZER_STEPS = 8
GRAD_ACCUM_STEPS = 2
PILOT_SAVE_DIR = "/kaggle/working/phase2_ner_targeted_pilot"
os.makedirs(PILOT_SAVE_DIR, exist_ok=True)

targeted_model.train()
targeted_optimizer.zero_grad(set_to_none=True)

# Mixed-precision gradient scaling for the T4.
scaler = torch.amp.GradScaler("cuda")
loss_history = []
optimizer_steps = 0
micro_steps = 0
pilot_failed = False

try:
    iterator = iter(pilot_loader)

    while optimizer_steps < MAX_OPTIMIZER_STEPS:
        targeted_optimizer.zero_grad(set_to_none=True)
        accumulated_losses = []

        for _ in range(GRAD_ACCUM_STEPS):
            try:
                batch = next(iterator)
            except StopIteration:
                iterator = iter(pilot_loader)
                batch = next(iterator)

            input_ids, attention_mask, labels = [
                tensor.to(device) for tensor in batch
            ]

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = targeted_model(input_ids, attention_mask)
                loss = targeted_criterion(
                    logits.reshape(-1, len(BIO_LABELS)),
                    labels.reshape(-1)
                )
                scaled_loss = loss / GRAD_ACCUM_STEPS

            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite loss; stopping pilot.")

            scaler.scale(scaled_loss).backward()
            accumulated_losses.append(float(loss.detach().cpu()))
            micro_steps += 1

            del input_ids, attention_mask, labels, logits, loss, scaled_loss

        scaler.unscale_(targeted_optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(
            targeted_model.parameters(), max_norm=1.0
        )

        if not torch.isfinite(grad_norm):
            raise RuntimeError("Non-finite gradient norm; stopping pilot.")

        scaler.step(targeted_optimizer)
        scaler.update()

        optimizer_steps += 1
        mean_loss = sum(accumulated_losses) / len(accumulated_losses)
        loss_history.append(mean_loss)

        print(
            f"Step {optimizer_steps}/{MAX_OPTIMIZER_STEPS} | "
            f"mean loss={mean_loss:.4f} | "
            f"grad norm={float(grad_norm):.3f} | "
            f"GPU allocated={torch.cuda.memory_allocated(0)/1024**3:.2f} GiB"
        )

except Exception as e:
    pilot_failed = True
    print("PILOT STOPPED:", repr(e))

finally:
    targeted_optimizer.zero_grad(set_to_none=True)
    gc.collect()
    torch.cuda.empty_cache()

if not pilot_failed and optimizer_steps == MAX_OPTIMIZER_STEPS:
    pilot_path = os.path.join(PILOT_SAVE_DIR, "targeted_pilot_8_steps.pt")
    torch.save({
        "model_state_dict": {
            k: v.detach().cpu() for k, v in targeted_model.state_dict().items()
        },
        "optimizer_steps": optimizer_steps,
        "loss_history": loss_history,
        "bio_labels": BIO_LABELS,
    }, pilot_path)
    print("\nPilot checkpoint saved separately:", pilot_path)
else:
    print("\nNo completed-pilot checkpoint saved.")

print("Completed optimizer steps:", optimizer_steps)
print("Loss history:", [round(x, 4) for x in loss_history])
print("Original champion unchanged in memory:", not champion_model.training)


In [ ]:

import torch
import numpy as np

ENTITY_LABELS = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
]

# Original notebook's span decoding: trim whitespace, preserve exact offsets.
def decode_spans(text, offsets, pred_ids):
    spans = []
    current_label = None
    current_start = None
    current_end = None

    def close():
        nonlocal current_label, current_start, current_end
        if current_label is not None:
            start, end = current_start, current_end
            while start < end and text[start].isspace():
                start += 1
            while end > start and text[end - 1].isspace():
                end -= 1
            if start < end:
                spans.append((start, end, current_label))
        current_label = current_start = current_end = None

    for (start, end), pred_id in zip(offsets, pred_ids):
        if end <= start:
            continue
        tag = BIO_LABELS[int(pred_id)]

        if tag == "O":
            close()
            continue

        prefix, label = tag.split("-", 1)
        if prefix == "B" or current_label != label:
            close()
            current_label, current_start, current_end = label, start, end
        else:
            current_end = end

    close()
    return sorted(set(spans))


# Original notebook's confidence-aware prediction path.
def predict_doc_constrained_with_confidence(text):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = enc.ids, enc.offsets

    if not ids:
        return []

    logit_sum = torch.zeros(
        (len(ids), len(BIO_LABELS)), dtype=torch.float32
    )
    token_count = torch.zeros(len(ids), dtype=torch.float32)

    ner_model.eval()
    with torch.no_grad():
        for start in range(0, len(ids), WINDOW_STRIDE):
            end = min(start + MAX_LEN, len(ids))
            chunk = ids[start:end]

            input_ids = torch.tensor(
                [chunk], dtype=torch.long, device=device
            )
            attention_mask = torch.ones_like(input_ids)
            output = ner_model(input_ids, attention_mask)
            logits = output[0] if isinstance(output, tuple) else output
            logits = logits[0].detach().float().cpu()

            chunk_len = end - start
            logit_sum[start:end] += logits[:chunk_len]
            token_count[start:end] += 1

            if end == len(ids):
                break

    averaged_logits = logit_sum / token_count.clamp_min(1).unsqueeze(-1)
    probabilities = torch.softmax(averaged_logits, dim=-1)

    pred_ids = bio_viterbi_decode(averaged_logits)
    spans = decode_spans(text, offsets, pred_ids)
    pred_ids_tensor = torch.tensor(pred_ids, dtype=torch.long)

    results = []
    for start, end, label in spans:
        token_indices = [
            i for i, (s, e) in enumerate(offsets)
            if e > s and s < end and e > start
        ]
        confidence = (
            float(probabilities[
                token_indices, pred_ids_tensor[token_indices]
            ].mean().item())
            if token_indices else 0.0
        )
        results.append((start, end, label, confidence))

    return results


# Original notebook's threshold filter.
def filter_predictions_by_thresholds_for_texts(predictions, thresholds, texts):
    filtered = {}
    for doc_id, spans in predictions.items():
        text = texts[doc_id]
        kept = []
        for s, e, label, conf in spans:
            if conf >= thresholds.get(label, 1.0):
                if 0 <= s < e <= len(text):
                    kept.append((int(s), int(e), label))
        filtered[doc_id] = kept
    return filtered


# Original notebook's exact-span metric implementation.
def calculate_exact_span_metrics(gold_by_doc, pred_by_doc, labels):
    metrics = {}
    for label in labels:
        gold_set, pred_set = set(), set()

        for doc_id, entities in gold_by_doc.items():
            for entity in entities:
                s, e, lab = int(entity[0]), int(entity[1]), str(entity[2])
                if lab == label:
                    gold_set.add((str(doc_id), s, e, lab))

        for doc_id, spans in pred_by_doc.items():
            for span in spans:
                s, e, lab = int(span[0]), int(span[1]), str(span[2])
                if lab == label:
                    pred_set.add((str(doc_id), s, e, lab))

        tp = len(gold_set & pred_set)
        fp = len(pred_set - gold_set)
        fn = len(gold_set - pred_set)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        metrics[label] = {
            "gold": len(gold_set), "pred": len(pred_set),
            "TP": tp, "FP": fp, "FN": fn,
            "precision": precision, "recall": recall, "f1": f1
        }

    metrics["MACRO_F1"] = sum(
        metrics[label]["f1"] for label in labels
    ) / len(labels)
    return metrics


print("Original notebook helper functions restored.")
print("Viterbi function still needs restoration before evaluation.")


In [ ]:

import numpy as np
import torch

NUM_LABELS = len(BIO_LABELS)

# Valid sequence starts: O or any B-entity tag.
legal_start = np.array([
    tag == "O" or tag.startswith("B-")
    for tag in BIO_LABELS
], dtype=bool)

# A transition into I-X is legal only from B-X or I-X.
legal_transition = np.ones(
    (NUM_LABELS, NUM_LABELS), dtype=bool
)

for prev_id, prev_tag in enumerate(BIO_LABELS):
    for curr_id, curr_tag in enumerate(BIO_LABELS):
        if curr_tag.startswith("I-"):
            entity = curr_tag[2:]
            legal_transition[prev_id, curr_id] = (
                prev_tag == f"B-{entity}" or
                prev_tag == f"I-{entity}"
            )

def bio_viterbi_decode(logits):
    """BIO-constrained Viterbi; accepts a torch tensor or NumPy array."""
    emissions = (
        logits.detach().float().cpu().numpy()
        if isinstance(logits, torch.Tensor)
        else np.asarray(logits, dtype=np.float64)
    )
    if emissions.ndim != 2 or emissions.shape[1] != NUM_LABELS:
        raise ValueError(f"Expected [sequence_length, {NUM_LABELS}] logits")
    if len(emissions) == 0:
        return []

    emissions = emissions.astype(np.float64, copy=False)
    neg_inf = -1e12

    scores = np.full_like(emissions, neg_inf, dtype=np.float64)
    backpointers = np.zeros(emissions.shape, dtype=np.int32)

    scores[0, legal_start] = emissions[0, legal_start]

    for t in range(1, len(emissions)):
        transition_scores = scores[t - 1][:, None]
        transition_scores = np.where(
            legal_transition, transition_scores, neg_inf
        )
        candidates = transition_scores + emissions[t][None, :]
        backpointers[t] = np.argmax(candidates, axis=0)
        scores[t] = candidates[
            backpointers[t], np.arange(NUM_LABELS)
        ]

    last = int(np.argmax(scores[-1]))
    path = [last]
    for t in range(len(emissions) - 1, 0, -1):
        path.append(int(backpointers[t, path[-1]]))
    path.reverse()
    return path


# Sanity-check starts and transitions with a deliberately invalid raw path.
test_logits = np.zeros((3, NUM_LABELS), dtype=np.float32)
test_logits[0, label2id["I-RESP"]] = 10.0
test_logits[0, label2id["B-RESP"]] = 8.0
test_logits[1, label2id["I-RESP"]] = 10.0
test_logits[2, label2id["I-WIT"]] = 10.0
test_logits[2, label2id["O"]] = 1.0

test_path = bio_viterbi_decode(test_logits)
test_tags = [BIO_LABELS[i] for i in test_path]

assert not test_tags[0].startswith("I-")
for i, tag in enumerate(test_tags):
    if tag.startswith("I-"):
        assert i > 0
        assert test_tags[i - 1] in (f"B-{tag[2:]}", f"I-{tag[2:]}")

print("Legal starts:", int(legal_start.sum()), "of", NUM_LABELS)
print("Legal transitions:", int(legal_transition.sum()))
print("Sanity-test decoded tags:", test_tags)
print("BIO constraint sanity check passed.")


In [ ]:

import numpy as np
import torch
from tqdm.auto import tqdm

EVAL_MAX_LEN = 1024
EVAL_STRIDE = 768

def predict_with_model_and_confidence(model, text):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = enc.ids, enc.offsets
    n = len(ids)

    if n == 0:
        return []

    logit_sum = torch.zeros((n, len(BIO_LABELS)), dtype=torch.float32)
    token_count = torch.zeros(n, dtype=torch.float32)

    model.eval()
    with torch.inference_mode():
        for start in range(0, n, EVAL_STRIDE):
            end = min(start + EVAL_MAX_LEN, n)
            x = torch.tensor([ids[start:end]], dtype=torch.long, device=device)
            mask = torch.ones_like(x)

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(x, mask)[0].float()

            chunk_logits = logits.cpu()
            logit_sum[start:end] += chunk_logits
            token_count[start:end] += 1

            if end == n:
                break

    avg_logits = logit_sum / token_count.clamp_min(1).unsqueeze(-1)
    probs = torch.softmax(avg_logits, dim=-1).numpy()
    pred_ids = bio_viterbi_decode(avg_logits)
    spans = decode_spans(text, offsets, pred_ids)

    results = []
    for start, end, label in spans:
        token_indices = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < end and b > start
        ]
        confidence = (
            float(np.mean([
                probs[i, pred_ids[i]] for i in token_indices
            ]))
            if token_indices else 0.0
        )
        results.append((start, end, label, confidence))

    return results


def evaluate_model(model, texts, gold, thresholds):
    raw_predictions = {}
    for doc_id, text in tqdm(texts.items(), desc="Evaluating"):
        raw_predictions[doc_id] = predict_with_model_and_confidence(model, text)

    filtered_predictions = filter_predictions_by_thresholds_for_texts(
        raw_predictions, thresholds, texts
    )
    metrics = calculate_exact_span_metrics(
        gold, filtered_predictions, ENTITY_LABELS
    )
    return metrics, raw_predictions, filtered_predictions


# Make sure threshold values are numeric.
eval_thresholds = {
    label: float(champion_thresholds_fusion.get(label, 1.0))
    for label in ENTITY_LABELS
}
print("Thresholds used:", eval_thresholds)

champion_val_metrics, champion_val_raw, champion_val_pred = evaluate_model(
    champion_model, val_texts, val_gold, eval_thresholds
)

pilot_val_metrics, pilot_val_raw, pilot_val_pred = evaluate_model(
    targeted_model, val_texts, val_gold, eval_thresholds
)

print("\n=== Fold-2 exact-span results ===")
print(f"Champion macro-F1: {champion_val_metrics['MACRO_F1']:.6f}")
print(f"Pilot macro-F1:    {pilot_val_metrics['MACRO_F1']:.6f}")
print(f"Difference:        {pilot_val_metrics['MACRO_F1'] - champion_val_metrics['MACRO_F1']:+.6f}")

print("\nPer-label F1:")
print(f"{'Label':<12} {'Champion':>10} {'Pilot':>10} {'Delta':>10}")
for label in ENTITY_LABELS:
    c = champion_val_metrics[label]["f1"]
    p = pilot_val_metrics[label]["f1"]
    print(f"{label:<12} {c:10.4f} {p:10.4f} {p-c:+10.4f}")

print("\nPredicted span counts:",
      sum(map(len, champion_val_pred.values())),
      sum(map(len, pilot_val_pred.values())))


In [ ]:

import numpy as np
from collections import defaultdict

WEAK_LABELS = ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]

def weak_proposal_report(raw_predictions, gold, thresholds):
    report = {}

    for label in WEAK_LABELS:
        gold_set = set()
        pred_set = set()
        confidences = []

        for doc_id, anns in gold.items():
            for ann in anns:
                if ann[2] == label:
                    gold_set.add((str(doc_id), int(ann[0]), int(ann[1]), label))

        for doc_id, spans in raw_predictions.items():
            for span in spans:
                s, e, lab, conf = span
                if lab == label:
                    pred_set.add((str(doc_id), int(s), int(e), label))
                    confidences.append(float(conf))

        tp = len(gold_set & pred_set)
        precision = tp / len(pred_set) if pred_set else 0.0
        recall = tp / len(gold_set) if gold_set else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

        report[label] = {
            "gold": len(gold_set),
            "raw_predictions": len(pred_set),
            "exact_matches": tp,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "threshold": float(thresholds.get(label, 1.0)),
            "confidences": confidences,
        }

    return report

champion_weak_report = weak_proposal_report(
    champion_val_raw, val_gold, eval_thresholds
)
pilot_weak_report = weak_proposal_report(
    pilot_val_raw, val_gold, eval_thresholds
)

print(f"{'Label':<12} {'Model':<10} {'Gold':>6} {'Raw':>6} {'Exact':>6} {'P':>7} {'R':>7} {'F1':>7} {'Thr':>7}")
for label in WEAK_LABELS:
    for model_name, report in [
        ("Champion", champion_weak_report),
        ("Pilot", pilot_weak_report),
    ]:
        r = report[label]
        print(
            f"{label:<12} {model_name:<10} {r['gold']:6d} "
            f"{r['raw_predictions']:6d} {r['exact_matches']:6d} "
            f"{r['precision']:7.3f} {r['recall']:7.3f} "
            f"{r['f1']:7.3f} {r['threshold']:7.3f}"
        )
        c = np.asarray(r["confidences"], dtype=float)
        if len(c):
            print(
                " " * 12,
                "confidence p10/median/p90:",
                np.round(np.quantile(c, [0.1, 0.5, 0.9]), 4).tolist()
            )
        else:
            print(" " * 12, "No raw proposals.")


In [ ]:

from collections import Counter
import numpy as np

raw_tag_counts = Counter()
viterbi_tag_counts = Counter()
weak_argmax_tokens = Counter()
weak_viterbi_tokens = Counter()

for doc_id, (offsets, logits) in doc_logits.items():
    # doc_logits was created by Cell 29; verify its structure.
    logits = np.asarray(logits)
    if logits.ndim != 2 or logits.shape[1] != len(BIO_LABELS):
        continue

    raw_ids = logits.argmax(axis=1).tolist()
    path_ids = bio_viterbi_decode(logits)

    raw_tag_counts.update(BIO_LABELS[i] for i in raw_ids)
    viterbi_tag_counts.update(BIO_LABELS[i] for i in path_ids)

    for tag in WEAK_LABELS:
        weak_argmax_tokens[tag] += sum(
            BIO_LABELS[i] in (f"B-{tag}", f"I-{tag}") for i in raw_ids
        )
        weak_viterbi_tokens[tag] += sum(
            BIO_LABELS[i] in (f"B-{tag}", f"I-{tag}") for i in path_ids
        )

print("Top raw argmax BIO tags:")
for tag, count in raw_tag_counts.most_common(15):
    print(f"{tag:<14} {count:>8}")

print("\nWeak-label token counts: raw argmax vs Viterbi")
for tag in WEAK_LABELS:
    print(
        f"{tag:<12} raw={weak_argmax_tokens[tag]:>6} "
        f"viterbi={weak_viterbi_tokens[tag]:>6}"
    )

print("\nTop Viterbi BIO tags:")
for tag, count in viterbi_tag_counts.most_common(15):
    print(f"{tag:<14} {count:>8}")


In [ ]:

import numpy as np
from collections import defaultdict

def spans_from_raw_argmax(text, offsets, logits):
    pred_ids = np.asarray(logits).argmax(axis=1).tolist()
    return decode_spans(text, offsets, pred_ids)

def spans_from_viterbi(text, offsets, logits):
    pred_ids = bio_viterbi_decode(logits)
    return decode_spans(text, offsets, pred_ids)

raw_argmax_predictions = {}
viterbi_predictions = {}

for doc_id, text in val_texts.items():
    offsets, logits = doc_logits[doc_id]

    raw_argmax_predictions[doc_id] = spans_from_raw_argmax(
        text, offsets, logits
    )
    viterbi_predictions[doc_id] = spans_from_viterbi(
        text, offsets, logits
    )

# Raw decoding has no confidence filtering; Viterbi has no confidence filtering.
raw_metrics = calculate_exact_span_metrics(
    val_gold, raw_argmax_predictions, ENTITY_LABELS
)
viterbi_metrics = calculate_exact_span_metrics(
    val_gold, viterbi_predictions, ENTITY_LABELS
)

print("Exact-span macro-F1, without confidence thresholds")
print(f"Raw argmax: {raw_metrics['MACRO_F1']:.6f}")
print(f"Viterbi:    {viterbi_metrics['MACRO_F1']:.6f}")
print("\nPer-label comparison:")
print(f"{'Label':<12} {'Raw argmax':>12} {'Viterbi':>12} {'Delta':>10}")

for label in ENTITY_LABELS:
    r = raw_metrics[label]["f1"]
    v = viterbi_metrics[label]["f1"]
    print(f"{label:<12} {r:12.4f} {v:12.4f} {v-r:+10.4f}")

print("\nWeak-label raw span counts:")
for label in WEAK_LABELS:
    raw_n = sum(
        1 for spans in raw_argmax_predictions.values()
        for s, e, lab in spans if lab == label
    )
    vit_n = sum(
        1 for spans in viterbi_predictions.values()
        for s, e, lab in spans if lab == label
    )
    print(f"{label:<12} raw={raw_n:4d}  viterbi={vit_n:4d}")


In [ ]:

import numpy as np
from collections import defaultdict

weak_bio_stats = defaultdict(lambda: {
    "B_probs": [], "I_probs": [], "B_ranks": [], "I_ranks": []
})

for doc_id, gold_anns in val_gold.items():
    offsets, logits = doc_logits[doc_id]
    logits = np.asarray(logits, dtype=np.float32)

    shifted = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(shifted)
    probs /= probs.sum(axis=1, keepdims=True)

    for ann in gold_anns:
        start, end, label = int(ann[0]), int(ann[1]), ann[2]
        if label not in WEAK_LABELS:
            continue

        token_indices = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < end and b > start
        ]
        if not token_indices:
            continue

        for j, i in enumerate(token_indices):
            prefix = "B" if j == 0 else "I"
            correct_tag = f"{prefix}-{label}"
            correct_id = label2id[correct_tag]
            row = probs[i]

            # Rank 1 means the correct BIO class has the highest probability.
            rank = 1 + int(np.sum(row > row[correct_id]))
            weak_bio_stats[label][f"{prefix}_probs"].append(
                float(row[correct_id])
            )
            weak_bio_stats[label][f"{prefix}_ranks"].append(rank)

print(f"{'Label':<12} {'Pos':<4} {'N':>6} {'Mean P':>9} {'Median P':>10} {'Top-1 %':>9} {'Median rank':>12}")
for label in WEAK_LABELS:
    for prefix in ["B", "I"]:
        probs = np.asarray(weak_bio_stats[label][f"{prefix}_probs"])
        ranks = np.asarray(weak_bio_stats[label][f"{prefix}_ranks"])
        if len(probs) == 0:
            continue
        print(
            f"{label:<12} {prefix:<4} {len(probs):6d} "
            f"{probs.mean():9.4f} {np.median(probs):10.4f} "
            f"{100*np.mean(ranks == 1):8.2f}% {np.median(ranks):12.1f}"
        )


In [ ]:

import numpy as np
import torch
from collections import defaultdict
from tqdm.auto import tqdm

def gold_weak_token_probabilities(model, texts, gold):
    stats = defaultdict(lambda: {"probs": [], "top1": []})
    model.eval()

    with torch.inference_mode():
        for doc_id, text in tqdm(texts.items(), desc="Scoring gold tokens"):
            enc = tokenizer.encode(text, add_special_tokens=False)
            ids, offsets = enc.ids, enc.offsets
            if not ids:
                continue

            sums = torch.zeros((len(ids), len(BIO_LABELS)), dtype=torch.float32)
            counts = torch.zeros(len(ids), dtype=torch.float32)

            for start in range(0, len(ids), EVAL_STRIDE):
                end = min(start + EVAL_MAX_LEN, len(ids))
                x = torch.tensor([ids[start:end]], dtype=torch.long, device=device)
                mask = torch.ones_like(x)

                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    logits = model(x, mask)[0].float()

                sums[start:end] += logits[0].cpu()
                counts[start:end] += 1
                if end == len(ids):
                    break

            avg = sums / counts.clamp_min(1).unsqueeze(-1)
            probs = torch.softmax(avg, dim=-1).numpy()
            pred = avg.argmax(dim=-1).numpy()

            for ann in gold.get(doc_id, []):
                s, e, label = int(ann[0]), int(ann[1]), ann[2]
                if label not in WEAK_LABELS:
                    continue

                token_ids = [
                    i for i, (a, b) in enumerate(offsets)
                    if b > a and a < e and b > s
                ]
                for j, i in enumerate(token_ids):
                    prefix = "B" if j == 0 else "I"
                    tag = f"{prefix}-{label}"
                    target = label2id[tag]
                    key = (label, prefix)
                    stats[key]["probs"].append(float(probs[i, target]))
                    stats[key]["top1"].append(int(pred[i] == target))

    return stats

champion_prob_stats = gold_weak_token_probabilities(
    champion_model, val_texts, val_gold
)
pilot_prob_stats = gold_weak_token_probabilities(
    targeted_model, val_texts, val_gold
)

print(f"{'Label':<12} {'Pos':<3} {'Champion P':>11} {'Pilot P':>9} {'Delta P':>9} {'Champ top1':>11} {'Pilot top1':>10}")
for label in WEAK_LABELS:
    for prefix in ["B", "I"]:
        key = (label, prefix)
        c = champion_prob_stats[key]
        p = pilot_prob_stats[key]
        if not c["probs"]:
            continue

        cp = float(np.mean(c["probs"]))
        pp = float(np.mean(p["probs"]))
        ca = 100 * float(np.mean(c["top1"]))
        pa = 100 * float(np.mean(p["top1"]))

        print(
            f"{label:<12} {prefix:<3} {cp:11.4f} {pp:9.4f} "
            f"{pp-cp:+9.4f} {ca:10.2f}% {pa:9.2f}%"
        )


In [ ]:

import numpy as np
import torch

doc_id = next(iter(val_texts))
text = val_texts[doc_id]
enc = tokenizer.encode(text, add_special_tokens=False)
ids = enc.ids

cached_offsets, cached_logits = doc_logits[doc_id]
cached_logits = np.asarray(cached_logits, dtype=np.float32)

fresh_sum = torch.zeros((len(ids), len(BIO_LABELS)), dtype=torch.float32)
fresh_count = torch.zeros(len(ids), dtype=torch.float32)

champion_model.eval()
with torch.inference_mode():
    for start in range(0, len(ids), EVAL_STRIDE):
        end = min(start + EVAL_MAX_LEN, len(ids))
        x = torch.tensor([ids[start:end]], dtype=torch.long, device=device)
        mask = torch.ones_like(x)

        # Output is [batch, sequence, labels]; select batch once.
        logits = champion_model(x, mask)[0].float()

        assert logits.shape == (end - start, len(BIO_LABELS))
        fresh_sum[start:end] += logits.cpu()
        fresh_count[start:end] += 1

        if end == len(ids):
            break

fresh_logits = (fresh_sum / fresh_count.clamp_min(1).unsqueeze(-1)).numpy()

print("Document:", doc_id)
print("Cached shape:", cached_logits.shape)
print("Fresh shape:", fresh_logits.shape)
print("Offsets identical:", cached_offsets == enc.offsets)

diff = np.abs(cached_logits - fresh_logits)
print("Maximum logit difference:", float(diff.max()))
print("Mean absolute logit difference:", float(diff.mean()))
print(
    "Cached vs fresh argmax agreement:",
    float(np.mean(cached_logits.argmax(1) == fresh_logits.argmax(1)))
)


In [ ]:

import numpy as np
from collections import defaultdict

corrected_stats = defaultdict(lambda: {"probs": [], "top1": []})

for doc_id, gold_anns in val_gold.items():
    offsets, logits = doc_logits[doc_id]
    logits = np.asarray(logits, dtype=np.float32)

    # Stable softmax across the 25 BIO labels.
    shifted = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(shifted)
    probs /= probs.sum(axis=1, keepdims=True)
    pred_ids = logits.argmax(axis=1)

    for ann in gold_anns:
        s, e, label = int(ann[0]), int(ann[1]), ann[2]
        if label not in WEAK_LABELS:
            continue

        token_indices = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < e and b > s
        ]

        for j, i in enumerate(token_indices):
            prefix = "B" if j == 0 else "I"
            tag = f"{prefix}-{label}"
            target_id = label2id[tag]
            key = (label, prefix)

            corrected_stats[key]["probs"].append(
                float(probs[i, target_id])
            )
            corrected_stats[key]["top1"].append(
                int(pred_ids[i] == target_id)
            )

print(f"{'Label':<12} {'Pos':<3} {'N':>6} {'Mean P':>9} {'Median P':>10} {'Top-1 %':>9}")
for label in WEAK_LABELS:
    for prefix in ["B", "I"]:
        values = corrected_stats[(label, prefix)]
        if not values["probs"]:
            continue

        p = np.asarray(values["probs"])
        top1 = np.asarray(values["top1"])

        print(
            f"{label:<12} {prefix:<3} {len(p):6d} "
            f"{p.mean():9.4f} {np.median(p):10.4f} "
            f"{100*top1.mean():8.2f}%"
        )


In [ ]:

import os
import torch

assert not champion_model.training, "Champion should remain in eval mode."

champion_state = champion_model.state_dict()
max_diff = max(
    (champion_state[k].detach().cpu() - champion_reference_cpu[k]).abs().max().item()
    for k in champion_state
)

pilot_path = "/kaggle/working/phase2_ner_targeted_pilot/targeted_pilot_8_steps.pt"

print("Champion max parameter difference from reference:", max_diff)
print("Pilot checkpoint exists:", os.path.exists(pilot_path))
if os.path.exists(pilot_path):
    print("Pilot checkpoint size (MiB):", round(os.path.getsize(pilot_path) / 1024**2, 2))

assert max_diff == 0.0
assert os.path.exists(pilot_path)
print("Champion integrity check passed.")


In [ ]:

import os

paths_to_check = [
    "/kaggle/working/cs781_backup_download/cs781_recovery/champion_recovery.pt",
    "/kaggle/working/cs781_backup_download/cs781_recovery/champion_thresholds_recovery.json",
    "/kaggle/working/phase2_ner_targeted_pilot/targeted_pilot_8_steps.pt",
    "/kaggle/working/phase2_ner_weighted/best_weighted_model.pt",
    "/kaggle/working/phase2_ner_experiment_E/lowest_train_loss.pt",
    "/kaggle/working/phase2_ner_experiment_F/experiment_F_epoch_1.pt",
    "/kaggle/working/prediction_champion.json",
    "/kaggle/working/prediction.json",
]

for path in paths_to_check:
    if os.path.isfile(path):
        size_mib = os.path.getsize(path) / (1024 ** 2)
        print(f"FOUND  {size_mib:8.2f} MiB  {path}")
    else:
        print(f"MISSING              {path}")


In [ ]:

import torch
import os

pilot_path = "/kaggle/working/phase2_ner_targeted_pilot/targeted_pilot_8_steps.pt"

pilot_ckpt = torch.load(
    pilot_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint keys:", list(pilot_ckpt.keys()))
print("Optimizer steps:", pilot_ckpt.get("optimizer_steps"))
print("Training loss history:", pilot_ckpt.get("loss_history"))
print("BIO label count:", len(pilot_ckpt.get("bio_labels", [])))
print("Labels match champion:",
      pilot_ckpt.get("bio_labels") == BIO_LABELS)
print("Checkpoint size (MiB):",
      round(os.path.getsize(pilot_path) / 1024**2, 2))


In [ ]:

import inspect

functions_to_check = [
    "make_unique_loss_windows",
    "decode_spans",
    "predict_doc_constrained_with_confidence",
    "filter_predictions_by_thresholds_for_texts",
    "calculate_exact_span_metrics",
    "bio_viterbi_decode",
]

for name in functions_to_check:
    fn = globals().get(name)
    print(f"\n{'='*60}\n{name}")
    if not callable(fn):
        print("MISSING")
        continue

    try:
        print("Signature:", inspect.signature(fn))
        print(inspect.getsource(fn)[:3000])
    except (OSError, TypeError) as exc:
        print("Python source unavailable:", repr(exc))

print("\nOriginal notebook training window settings:")
print("MAX_LEN = 512")
print("WINDOW_STRIDE = 448")


In [ ]:
# Restore the original notebook's inference window settings.
MAX_LEN = 512
WINDOW_STRIDE = 448

EVAL_MAX_LEN = 512
EVAL_STRIDE = 448

assert MAX_LEN == 512
assert WINDOW_STRIDE == 448
assert EVAL_MAX_LEN == 512
assert EVAL_STRIDE == 448

print("Original window settings restored.")
print("MAX_LEN:", MAX_LEN)
print("WINDOW_STRIDE:", WINDOW_STRIDE)
print("Champion unchanged:", not champion_model.training)
print("Pilot is separate:", targeted_model is not champion_model)

In [ ]:
import torch

assert targeted_model is not champion_model

_original_ner_model = ner_model

try:
    # 1. Generate raw champion predictions for every validation document.
    ner_model = champion_model
    champion_model.eval()

    champion_raw = {}
    for doc_id, text in val_texts.items():
        champion_raw[doc_id] = predict_doc_constrained_with_confidence(text)

    print("Champion raw predictions:", len(champion_raw))

    # 2. Generate raw pilot predictions for every validation document.
    ner_model = targeted_model
    targeted_model.eval()

    pilot_raw = {}
    for doc_id, text in val_texts.items():
        pilot_raw[doc_id] = predict_doc_constrained_with_confidence(text)

    print("Pilot raw predictions:", len(pilot_raw))

finally:
    ner_model = _original_ner_model
    champion_model.eval()
    targeted_model.eval()

# 3. Apply thresholds using the function's actual argument order:
#    (predictions, thresholds, texts)
champion_predictions = filter_predictions_by_thresholds_for_texts(
    champion_raw, champion_thresholds_fusion, val_texts
)

pilot_predictions = filter_predictions_by_thresholds_for_texts(
    pilot_raw, champion_thresholds_fusion, val_texts
)

# 4. Evaluate exact-span metrics.
champion_metrics = calculate_exact_span_metrics(
    val_gold, champion_predictions, ENTITY_LABELS
)

pilot_metrics = calculate_exact_span_metrics(
    val_gold, pilot_predictions, ENTITY_LABELS
)

print("\nChampion Macro-F1:", champion_metrics["MACRO_F1"])
print("Pilot Macro-F1:   ", pilot_metrics["MACRO_F1"])
print(
    "Difference:       ",
    pilot_metrics["MACRO_F1"] - champion_metrics["MACRO_F1"]
)

print("\nPer-label comparison")
print(f"{'Label':<12} {'Champion F1':>12} {'Pilot F1':>12} {'Difference':>12}")

for label in ENTITY_LABELS:
    c_f1 = champion_metrics[label]["f1"]
    p_f1 = pilot_metrics[label]["f1"]
    print(f"{label:<12} {c_f1:>12.4f} {p_f1:>12.4f} {p_f1-c_f1:>12.4f}")

print("\nPredicted spans:",
      sum(len(v) for v in champion_predictions.values()), "(champion),",
      sum(len(v) for v in pilot_predictions.values()), "(pilot)")

In [ ]:
import torch
import numpy as np
from collections import Counter, defaultdict

WEAK_LABELS = ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]

# Use the champion explicitly; do not change its weights.
champion_model.eval()

weak_token_counts = {
    label: Counter() for label in WEAK_LABELS
}
weak_start_counts = {
    label: Counter() for label in WEAK_LABELS
}
weak_examples = defaultdict(list)

# Cache per-document token predictions and probabilities.
champion_model_gold_audit = {}

for doc_id, text in train_texts.items():
    enc = tokenizer.encode(text, add_special_tokens=False)
    token_ids = enc.ids
    offsets = enc.offsets

    # Use the same original window settings as inference.
    window_logits = []
    window_positions = []

    for start in range(0, len(token_ids), WINDOW_STRIDE):
        end = min(start + MAX_LEN, len(token_ids))
        if start >= len(token_ids):
            break

        ids = torch.tensor(
            [token_ids[start:end]], dtype=torch.long, device=device
        )
        mask = torch.ones_like(ids)

        with torch.no_grad():
            logits = champion_model(ids, mask)[0].float().cpu().numpy()

        window_logits.append((start, end, logits[0]))

        if end == len(token_ids):
            break

    # Average overlapping-window logits for each original token.
    sums = np.zeros((len(token_ids), len(BIO_LABELS)), dtype=np.float64)
    counts = np.zeros(len(token_ids), dtype=np.float64)

    for start, end, logits in window_logits:
        sums[start:end] += logits
        counts[start:end] += 1

    counts[counts == 0] = 1
    avg_logits = sums / counts[:, None]

    # Softmax probabilities for token-level diagnostics.
    shifted = avg_logits - avg_logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(shifted)
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    pred_ids = avg_logits.argmax(axis=1)

    champion_model_gold_audit[doc_id] = {
        "offsets": offsets,
        "pred_ids": pred_ids,
        "probs": probs,
    }

    # Gold format: [start, end, label, surface]
    for gold in train_gold[doc_id]:
        gs, ge, label = int(gold[0]), int(gold[1]), gold[2]
        if label not in WEAK_LABELS:
            continue

        token_indices = [
            i for i, (s, e) in enumerate(offsets)
            if e > s and s < ge and e > gs
        ]
        if not token_indices:
            continue

        for j, i in enumerate(token_indices):
            predicted_tag = BIO_LABELS[int(pred_ids[i])]
            weak_token_counts[label][predicted_tag] += 1

            if j == 0:
                weak_start_counts[label][predicted_tag] += 1

        # Save a few examples for manual inspection.
        if len(weak_examples[label]) < 8:
            weak_examples[label].append({
                "doc_id": doc_id,
                "gold_text": text[gs:ge],
                "gold_span": (gs, ge),
                "predicted_tags": [
                    BIO_LABELS[int(pred_ids[i])] for i in token_indices
                ],
                "top_predictions": [
                    sorted(
                        [
                            (BIO_LABELS[k], float(probs[i, k]))
                            for k in range(len(BIO_LABELS))
                        ],
                        key=lambda x: x[1],
                        reverse=True
                    )[:3]
                    for i in token_indices[:5]
                ],
            })

print("Token predictions over gold weak-label spans")
for label in WEAK_LABELS:
    total = sum(weak_token_counts[label].values())
    print(f"\n{label}: {total} tokens")
    for tag, count in weak_token_counts[label].most_common(10):
        print(f"  {tag:<14} {count:>6} ({count / max(total, 1):.1%})")

print("\nPredictions at the first token of each gold span")
for label in WEAK_LABELS:
    total = sum(weak_start_counts[label].values())
    print(f"\n{label}: {total} entity starts")
    for tag, count in weak_start_counts[label].most_common(8):
        print(f"  {tag:<14} {count:>5} ({count / max(total, 1):.1%})")

In [ ]:
for label in WEAK_LABELS:
    print("\n" + "=" * 90)
    print("LABEL:", label)

    for ex in weak_examples[label]:
        print(f"\nDocument: {ex['doc_id']}")
        print("Gold text:", repr(ex["gold_text"]))
        print("Gold offsets:", ex["gold_span"])
        print("Predicted BIO tags:", ex["predicted_tags"])

        for i, top3 in enumerate(ex["top_predictions"]):
            formatted = ", ".join(
                f"{tag}={prob:.3f}" for tag, prob in top3
            )
            print(f"  Token {i + 1}: {formatted}")

In [ ]:
val_weak_token_counts = {
    label: Counter() for label in WEAK_LABELS
}

for doc_id, text in val_texts.items():
    enc = tokenizer.encode(text, add_special_tokens=False)
    token_ids = enc.ids
    offsets = enc.offsets

    sums = np.zeros((len(token_ids), len(BIO_LABELS)), dtype=np.float64)
    counts = np.zeros(len(token_ids), dtype=np.float64)

    for start in range(0, len(token_ids), WINDOW_STRIDE):
        end = min(start + MAX_LEN, len(token_ids))
        if start >= len(token_ids):
            break

        ids = torch.tensor(
            [token_ids[start:end]], dtype=torch.long, device=device
        )
        mask = torch.ones_like(ids)

        with torch.no_grad():
            logits = champion_model(ids, mask)[0].float().cpu().numpy()[0]

        sums[start:end] += logits
        counts[start:end] += 1

        if end == len(token_ids):
            break

    counts[counts == 0] = 1
    pred_ids = (sums / counts[:, None]).argmax(axis=1)

    for gold in val_gold[doc_id]:
        gs, ge, label = int(gold[0]), int(gold[1]), gold[2]
        if label not in WEAK_LABELS:
            continue

        for i, (s, e) in enumerate(offsets):
            if e > s and s < ge and e > gs:
                val_weak_token_counts[label][BIO_LABELS[int(pred_ids[i])]] += 1

print("Validation token predictions over gold weak-label spans")
for label in WEAK_LABELS:
    counts = val_weak_token_counts[label]
    total = sum(counts.values())
    print(f"\n{label}: {total} tokens")
    for tag, count in counts.most_common(8):
        print(f"  {tag:<14} {count:>6} ({count / max(total, 1):.1%})")

In [ ]:
from collections import Counter

WEAK_LABELS = ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]

audit = {
    label: {
        "gold_tokens": 0,
        "matching_bio": 0,
        "wrong_bio": 0,
        "examples": [],
    }
    for label in WEAK_LABELS
}

for doc_id, text in train_texts.items():
    enc = tokenizer.encode(text, add_special_tokens=False)
    offsets = enc.offsets

    for ann in train_gold[doc_id]:
        gs, ge, label = int(ann[0]), int(ann[1]), ann[2]
        if label not in WEAK_LABELS:
            continue

        indices = [
            i for i, (s, e) in enumerate(offsets)
            if e > s and s < ge and e > gs
        ]

        for j, i in enumerate(indices):
            expected = ("B-" if j == 0 else "I-") + label
            audit[label]["gold_tokens"] += 1

            if expected in BIO_LABELS:
                audit[label]["matching_bio"] += 1
            else:
                audit[label]["wrong_bio"] += 1

        if len(audit[label]["examples"]) < 3:
            audit[label]["examples"].append({
                "text": text[gs:ge],
                "expected_tags": [
                    ("B-" if j == 0 else "I-") + label
                    for j in range(len(indices))
                ],
                "token_offsets": [offsets[i] for i in indices],
            })

for label in WEAK_LABELS:
    a = audit[label]
    print(
        f"{label}: tokens={a['gold_tokens']}, "
        f"BIO labels available={a['matching_bio']}, "
        f"unavailable={a['wrong_bio']}"
    )
    for ex in a["examples"]:
        print(" ", repr(ex["text"]))
        print("   expected:", ex["expected_tags"])
        print("   offsets: ", ex["token_offsets"])

In [ ]:
import inspect
from collections import Counter

print("Checking available window-building functions...")

for name in [
    "document_to_windows",
    "make_unique_loss_windows",
    "build_training_windows",
]:
    obj = globals().get(name)
    if callable(obj):
        print(f"\n{name} exists")
        try:
            print("Signature:", inspect.signature(obj))
            print("Source preview:")
            print(inspect.getsource(obj)[:1200])
        except Exception as e:
            print("Source unavailable:", repr(e))
    else:
        print(f"{name}: not defined")

print("\nChecking existing training-window variables...")
for name, obj in list(globals().items()):
    if any(term in name.lower() for term in ["window", "train_dataset", "dataset"]):
        if isinstance(obj, (list, tuple)):
            print(f"{name}: {type(obj).__name__}, length={len(obj)}")
        elif obj is not None and any(
            term in type(obj).__name__.lower()
            for term in ["dataset", "dataloader"]
        ):
            print(f"{name}: {type(obj).__name__}")

In [ ]:
from collections import Counter

print("Number of pilot windows:", len(pilot_windows))
print("First window type:", type(pilot_windows[0]).__name__)

w = pilot_windows[0]

if isinstance(w, dict):
    print("Window keys:", list(w.keys()))

    for key, value in w.items():
        try:
            print(
                f"{key}: type={type(value).__name__}, "
                f"length={len(value)}, "
                f"sample={str(value[:8])[:250]}"
            )
        except Exception:
            print(f"{key}: type={type(value).__name__}, value={str(value)[:200]}")

elif isinstance(w, (list, tuple)):
    print("Number of fields:", len(w))
    for i, value in enumerate(w):
        try:
            print(
                f"field {i}: type={type(value).__name__}, "
                f"length={len(value)}, "
                f"sample={str(value[:8])[:250]}"
            )
        except Exception:
            print(f"field {i}: {type(value).__name__}, {str(value)[:200]}")

print("\nExisting relevant globals:")
for name in [
    "BIO_LABELS", "label2id", "train_texts", "train_gold",
    "tokenizer", "MAX_LEN", "WINDOW_STRIDE",
]:
    print(name, "exists" if name in globals() else "MISSING")

In [ ]:
from collections import Counter, defaultdict

WEAK_LABELS = ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]

stored_counts = Counter()
stored_b_counts = Counter()
stored_i_counts = Counter()
ignored_counts = Counter()
examples = defaultdict(list)

for w in pilot_windows:
    labels = w["labels"]

    for i, label_id in enumerate(labels):
        label_id = int(label_id)

        if label_id == -100:
            ignored_counts["all"] += 1
            continue

        if not (0 <= label_id < len(BIO_LABELS)):
            raise ValueError(f"Invalid label ID: {label_id}")

        tag = BIO_LABELS[label_id]

        for entity in WEAK_LABELS:
            if tag in (f"B-{entity}", f"I-{entity}"):
                stored_counts[entity] += 1

                if tag.startswith("B-"):
                    stored_b_counts[entity] += 1
                else:
                    stored_i_counts[entity] += 1

                if len(examples[entity]) < 3:
                    examples[entity].append({
                        "doc_id": w["doc_id"],
                        "window_start": w["start"],
                        "token_position": i,
                        "tag": tag,
                        "token_id": w["input_ids"][i],
                    })

print("Stored weak-label tags in pilot windows")
for entity in WEAK_LABELS:
    print(
        f"{entity:<12} B={stored_b_counts[entity]:>5} "
        f"I={stored_i_counts[entity]:>5} "
        f"total={stored_counts[entity]:>5}"
    )
    for ex in examples[entity]:
        print("  ", ex)

print("\nIgnored label positions:", ignored_counts["all"])
print("\nExpected source BIO token counts:")
for entity in WEAK_LABELS:
    expected_b = sum(
        1 for doc_id in train_gold
        for ann in train_gold[doc_id]
        if ann[2] == entity
    )
    print(f"{entity:<12} gold entity spans={expected_b}")

In [ ]:
import torch
import numpy as np
from collections import defaultdict

WEAK_LABELS = ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]

def collect_gold_weak_probabilities(model, texts, gold_data):
    model.eval()
    result = {
        label: {"B": [], "I": []}
        for label in WEAK_LABELS
    }

    for doc_id, text in texts.items():
        enc = tokenizer.encode(text, add_special_tokens=False)
        token_ids, offsets = enc.ids, enc.offsets

        # Match the original inference settings.
        sums = np.zeros((len(token_ids), len(BIO_LABELS)), dtype=np.float64)
        counts = np.zeros(len(token_ids), dtype=np.float64)

        for start in range(0, len(token_ids), WINDOW_STRIDE):
            end = min(start + MAX_LEN, len(token_ids))
            if start >= len(token_ids):
                break

            ids = torch.tensor(
                [token_ids[start:end]], dtype=torch.long, device=device
            )
            mask = torch.ones_like(ids)

            with torch.no_grad():
                logits = model(ids, mask)[0].float().cpu().numpy()[0]

            sums[start:end] += logits
            counts[start:end] += 1

            if end == len(token_ids):
                break

        counts[counts == 0] = 1
        avg_logits = sums / counts[:, None]
        shifted = avg_logits - avg_logits.max(axis=1, keepdims=True)
        exp_logits = np.exp(shifted)
        probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)

        for ann in gold_data[doc_id]:
            gs, ge, label = int(ann[0]), int(ann[1]), ann[2]
            if label not in WEAK_LABELS:
                continue

            indices = [
                i for i, (s, e) in enumerate(offsets)
                if e > s and s < ge and e > gs
            ]

            for j, i in enumerate(indices):
                role = "B" if j == 0 else "I"
                tag = f"{role}-{label}"
                result[label][role].append(
                    float(probs[i, label2id[tag]])
                )

    return result

champion_probs = collect_gold_weak_probabilities(
    champion_model, train_texts, train_gold
)
pilot_probs = collect_gold_weak_probabilities(
    targeted_model, train_texts, train_gold
)

print("Mean probability assigned to the correct BIO class")
print(f"{'Label':<12} {'Role':<5} {'Champion':>12} {'Pilot':>12} {'Change':>12}")

for label in WEAK_LABELS:
    for role in ["B", "I"]:
        c = np.mean(champion_probs[label][role])
        p = np.mean(pilot_probs[label][role])
        print(f"{label:<12} {role:<5} {c:>12.5f} {p:>12.5f} {p-c:>+12.5f}")

In [ ]:
import numpy as np, pandas as pd, torch

DIAG_MAX_LEN, DIAG_STRIDE = 512, 448          # original inference settings
DIAG_LABELS = ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT", "COURT", "DATE"]  # last two = reference
O_ID = label2id["O"]

@torch.inference_mode()
def diag_doc_probs(model, text):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = enc.ids, enc.offsets
    dev = next(model.parameters()).device
    n = len(ids)
    logit_sum = torch.zeros((n, len(BIO_LABELS)))
    cnt = torch.zeros(n)
    model.eval()
    for start in range(0, n, DIAG_STRIDE):
        end = min(start + DIAG_MAX_LEN, n)
        x = torch.tensor([ids[start:end]], dtype=torch.long, device=dev)
        out = model(x, torch.ones_like(x))
        out = out[0] if isinstance(out, tuple) else out
        logit_sum[start:end] += out[0].float().cpu()
        cnt[start:end] += 1
        if end == n:
            break
    avg = logit_sum / cnt.clamp_min(1).unsqueeze(-1)
    return offsets, torch.softmax(avg, dim=-1)

def gold_token_table(model, name):
    rows = {lab: [] for lab in DIAG_LABELS}   # (is_first, p_correct_tag, p_O, p_type_any, argmax_id)
    for doc_id, text in val_texts.items():
        offsets, probs = diag_doc_probs(model, text)
        for ent in val_gold[doc_id]:
            s, e, lab = int(ent[0]), int(ent[1]), ent[2]
            if lab not in rows:
                continue
            toks = [i for i, (a, b) in enumerate(offsets) if b > a and a < e and b > s]
            for k, i in enumerate(toks):
                is_first = (k == 0)
                tag = f"{'B' if is_first else 'I'}-{lab}"
                p = probs[i]
                p_type = float(p[label2id[f"B-{lab}"]] + p[label2id[f"I-{lab}"]])
                rows[lab].append((is_first, float(p[label2id[tag]]), float(p[O_ID]),
                                  p_type, int(p.argmax())))
    out = []
    for lab, r in rows.items():
        a = np.array(r, dtype=float)
        for grp, mask in [("B (first tok)", a[:, 0] == 1), ("I (rest)", a[:, 0] == 0)]:
            m = a[mask]
            out.append({
                "model": name, "label": lab, "tok": grp, "n": len(m),
                "mean_p_correct": m[:, 1].mean(), "median_p_correct": np.median(m[:, 1]),
                "frac_p>0.10": (m[:, 1] > 0.10).mean(),
                "frac_p>0.30": (m[:, 1] > 0.30).mean(),
                "mean_p_O": m[:, 2].mean(),
                "mean_p_B+I_of_type": m[:, 3].mean(),
                "frac_argmax_O": (m[:, 4] == O_ID).mean(),
            })
    return pd.DataFrame(out)

champ_tbl = gold_token_table(champion_model, "champion")
pilot_tbl = gold_token_table(targeted_model, "pilot")
diag = pd.concat([champ_tbl, pilot_tbl]).sort_values(["label", "tok", "model"])
pd.set_option("display.width", 220)
print(diag.round(3).to_string(index=False))

In [ ]:
ENT = ["APP","RESP","A.COUNSEL","R.COUNSEL","JUDGE","WIT","AUTH","COURT","STAT","PREC","DATE","CASENO"]
TYPES = ["O"] + ENT
type_ids = [TYPES.index("O" if t == "O" else t[2:]) for t in BIO_LABELS]
M = torch.zeros(len(BIO_LABELS), len(TYPES))
M[torch.arange(len(BIO_LABELS)), torch.tensor(type_ids)] = 1.0
type_of_id = torch.tensor(type_ids)
WEAK = ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]

def type_confusion(model, texts, gold, fold_name):
    mass = {l: [] for l in WEAK}
    arg = {l: [] for l in WEAK}
    for doc_id, text in texts.items():
        offsets, probs = diag_doc_probs(model, text)
        t_mass = probs @ M                          # [n_tokens, 13] mass per entity type
        t_arg = type_of_id[probs.argmax(-1)]        # argmax type per token
        for ent in gold[doc_id]:
            lab = ent[2]
            if lab not in mass:
                continue
            s, e = int(ent[0]), int(ent[1])
            toks = [i for i, (a, b) in enumerate(offsets) if b > a and a < e and b > s]
            if toks:
                mass[lab].append(t_mass[toks])
                arg[lab].append(t_arg[toks])
    print(f"\n===== {fold_name} (champion) =====")
    for lab in WEAK:
        m = torch.cat(mass[lab]); a = torch.cat(arg[lab])
        mm = m.mean(0)
        top_mass = sorted(zip(TYPES, mm.tolist()), key=lambda x: -x[1])[:5]
        counts = torch.bincount(a, minlength=len(TYPES)).float() / len(a)
        top_arg = sorted(zip(TYPES, counts.tolist()), key=lambda x: -x[1])[:5]
        print(f"\n{lab}  (n_tokens={len(a)})")
        print("  mean prob mass by type :", "  ".join(f"{t}={v:.2f}" for t, v in top_mass))
        print("  argmax by type         :", "  ".join(f"{t}={v:.2f}" for t, v in top_arg))

type_confusion(champion_model, val_texts,   val_gold,   "FOLD 2 (validation)")
type_confusion(champion_model, train_texts, train_gold, "FOLD 1 (training)")

In [ ]:
from collections import Counter
import random
random.seed(0)

vit_counts = {l: Counter() for l in WEAK}
samples = {l: [] for l in WEAK}

for doc_id, text in val_texts.items():
    offsets, probs = diag_doc_probs(champion_model, text)
    vpath = bio_viterbi_decode(torch.log(probs.clamp_min(1e-12)))  # same path as on raw logits
    apath = probs.argmax(-1).tolist()
    for ent in val_gold[doc_id]:
        lab = ent[2]
        if lab not in vit_counts:
            continue
        s, e = int(ent[0]), int(ent[1])
        toks = [i for i, (a, b) in enumerate(offsets) if b > a and a < e and b > s]
        for i in toks:
            vit_counts[lab][TYPES[type_ids[vpath[i]]]] += 1
        if toks:
            samples[lab].append((
                text[max(0, s - 40):s].replace("\n", " "),
                text[s:e].replace("\n", " "),
                [BIO_LABELS[apath[i]] for i in toks[:8]],
                [BIO_LABELS[vpath[i]] for i in toks[:8]],
            ))

print("=== Viterbi-decoded type distribution on gold weak-label tokens (fold 2) ===")
for lab in WEAK:
    tot = sum(vit_counts[lab].values())
    top = vit_counts[lab].most_common(4)
    print(f"{lab:10s} n={tot}  " + "  ".join(f"{t}={c/tot:.2f}" for t, c in top))

print("\n=== Sample gold spans: left context | gold text | argmax tags | viterbi tags ===")
for lab in WEAK:
    print(f"\n--- {lab} ---")
    for ctx, gtxt, am, vt in random.sample(samples[lab], min(4, len(samples[lab]))):
        print(f"ctx : ...{ctx!r}")
        print(f"gold: {gtxt[:110]!r}")
        print(f"arg : {am}")
        print(f"vit : {vt}\n")

In [ ]:
import re
from collections import Counter

HON = re.compile(r"\b(?:Mr|Ms|Mrs|Shri|Sri|Smt|Dr)\b\.?[ \t]*(?=[A-Z])")

def covering_labels(gold_list, pos):
    return tuple(sorted({e[2] for e in gold_list if int(e[0]) <= pos < int(e[1])})) or ("NONE",)

def honorific_audit(texts, gold, fold_name):
    cnt, none_examples = Counter(), []
    for doc_id, text in texts.items():
        for m in HON.finditer(text):
            labs = covering_labels(gold[doc_id], m.end())
            cnt[labs] += 1
            if labs == ("NONE",) and len(none_examples) < 6:
                none_examples.append(text[max(0, m.start()-30):m.end()+25].replace("\n", " "))
    tot = sum(cnt.values())
    print(f"\n=== {fold_name}: honorific-prefixed names, covering gold label (total={tot}) ===")
    for labs, c in cnt.most_common(10):
        print(f"  {'+'.join(labs):28s} {c:5d}  {c/tot:5.1%}")
    print("  sample unannotated:")
    for ex in none_examples:
        print("   ", repr(ex))

honorific_audit(train_texts, train_gold, "FOLD 1 (train)")
honorific_audit(val_texts,   val_gold,   "FOLD 2 (val)")

print("\n=== Preceding word before gold weak-label spans (both folds) ===")
prev = {l: Counter() for l in WEAK}
for texts, gold in [(train_texts, train_gold), (val_texts, val_gold)]:
    for doc_id, text in texts.items():
        for e in gold[doc_id]:
            if e[2] in prev:
                w = text[max(0, int(e[0]) - 20):int(e[0])].split()
                prev[e[2]][(w[-1].lower() if w else "<bos>")] += 1
for lab in WEAK:
    tot = sum(prev[lab].values())
    print(f"{lab:10s} n={tot}  " + "  ".join(f"{w!r}={c}" for w, c in prev[lab].most_common(7)))

In [ ]:
import math, torch
from bisect import bisect_left, bisect_right

def audit_doc_windows(text, entities, max_len=512, stride=448):
    enc = tokenizer.encode(text, add_special_tokens=False)
    n, offsets = len(enc.ids), enc.offsets
    starts = [a for a, b in offsets]; ends = [b for a, b in offsets]
    tags = [label2id["O"]] * n
    for ent in entities:
        s, e, lab = int(ent[0]), int(ent[1]), str(ent[2])
        lo, hi = bisect_right(ends, s), bisect_left(starts, e)       # tokens with end>s and start<e
        ov = [i for i in range(lo, hi) if ends[i] > starts[i]]
        if not ov:
            continue
        tags[ov[0]] = label2id[f"B-{lab}"]
        for i in ov[1:]:
            tags[i] = label2id[f"I-{lab}"]
    wins = []
    for st in range(0, n, stride):
        en = min(st + max_len, n)
        if st >= en:
            break
        w = tags[st:en]
        if w and BIO_LABELS[w[0]].startswith("I-"):
            w = [label2id["B-" + BIO_LABELS[w[0]][2:]]] + w[1:]
        wins.append(w)
        if en == n:
            break
    return wins, tags

doc_counts = torch.zeros(len(BIO_LABELS)); win_counts = torch.zeros(len(BIO_LABELS)); n_wins = 0
for doc_id, text in train_texts.items():
    wins, tags = audit_doc_windows(text, train_gold[doc_id])
    doc_counts += torch.bincount(torch.tensor(tags), minlength=len(BIO_LABELS)).float()
    for w in wins:
        win_counts += torch.bincount(torch.tensor(w), minlength=len(BIO_LABELS)).float()
    n_wins += len(wins)

print("Sanity check vs earlier audit (doc-level B/I):")
for lab, (b, i) in {"RESP": (191, 370), "A.COUNSEL": (70, 239), "R.COUNSEL": (87, 264), "WIT": (67, 130)}.items():
    print(f"  {lab:10s} mine B={int(doc_counts[label2id['B-'+lab]])} I={int(doc_counts[label2id['I-'+lab]])}   expected B={b} I={i}")

freq = win_counts.clamp_min(1)
raw = torch.sqrt(freq[0] / freq)
w = raw.clone(); w[0] = 1.0
w = w.clamp(1.0, 5.0)
w_norm = w / w.mean()
print(f"\nTrain windows: {n_wins}  ->  optimizer steps/epoch ~{math.ceil(n_wins/8)}, x3 epochs ~{3*math.ceil(n_wins/8)}")
print(f"\n{'tag':<14}{'tokens':>9}{'raw_w':>8}{'clamped':>9}{'final':>8}")
for i, t in enumerate(BIO_LABELS):
    print(f"{t:<14}{int(win_counts[i]):>9}{raw[i].item():>8.1f}{w[i].item():>9.1f}{w_norm[i].item():>8.2f}")
print("\nEntity tags at the clamp (5.0):", sum(1 for i in range(1, len(BIO_LABELS)) if w[i] >= 5.0), "of", len(BIO_LABELS) - 1)

In [ ]:

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2Config, GPT2Model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Part 0: locate and verify the champion checkpoint ----
def find_artifact(filename):
    matches = []
    for root in (Path("/kaggle/input"), Path("/kaggle/working")):
        if root.exists():
            matches.extend(
                p for p in root.rglob(filename) if p.is_file()
            )
    matches = sorted(set(matches))
    if not matches:
        raise FileNotFoundError(f"Could not find {filename}")
    preferred = [p for p in matches if "cs781_recovery" in str(p)]
    return preferred[0] if preferred else matches[0]

RECOVERY_PATH = find_artifact("champion_recovery.pt")
PHASE1_PATH = find_artifact("checkpoint_latest.pt")

rec = torch.load(
    RECOVERY_PATH, map_location="cpu", weights_only=False
)
sd = champion_model.state_dict()

missing = set(rec["model_state_dict"]) - set(sd)
assert not missing, f"Champion model is missing keys: {sorted(missing)[:10]}"

maxdiff = max(
    (sd[k].detach().float().cpu() - v.float()).abs().max().item()
    for k, v in rec["model_state_dict"].items()
)
print("Champion vs recovery checkpoint max |diff|:", maxdiff)

if "targeted_model" in globals():
    print(
        "Champion encoder shares targeted_model encoder:",
        champion_model.encoder is targeted_model.encoder
    )
else:
    print("targeted_model not defined; skipping encoder-sharing check.")

del rec, sd
gc.collect()

# ---- Part 1: pristine frozen Phase 1 encoder ----
phase1_ckpt = torch.load(
    PHASE1_PATH, map_location="cpu", weights_only=False
)
cfg = phase1_ckpt["model_config"]

p1_cfg = GPT2Config(
    vocab_size=cfg["vocab_size"],
    n_positions=cfg["context_length"],
    n_ctx=cfg["context_length"],
    n_embd=cfg["hidden_size"],
    n_layer=cfg["num_layers"],
    n_head=cfg["num_heads"],
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
)

enc_p1 = GPT2Model(p1_cfg)
enc_p1.load_state_dict(
    {
        k.removeprefix("transformer."): v
        for k, v in phase1_ckpt["model_state_dict"].items()
        if k.startswith("transformer.")
    },
    strict=True,
)
enc_p1 = enc_p1.to(device).eval()

# Estimate feature memory before extracting all token representations.
def token_count(text):
    return len(tokenizer.encode(text, add_special_tokens=False).ids)

n_train_tokens = sum(token_count(t) for t in train_texts.values())
n_val_tokens = sum(token_count(t) for t in val_texts.values())
estimated_gib = (
    (n_train_tokens + n_val_tokens) * p1_cfg.n_embd * 4
    / (1024 ** 3)
)
print("Train tokens:", n_train_tokens)
print("Validation tokens:", n_val_tokens)
print(f"Estimated float32 feature storage: {estimated_gib:.2f} GiB")

if estimated_gib > 8:
    raise MemoryError(
        "Feature tensors may exceed available RAM. Stop here and use a "
        "chunked/disk-backed feature extraction variant instead."
    )

@torch.inference_mode()
def doc_features(encoder, text):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids = enc.ids
    n = len(ids)
    feat = torch.zeros((n, p1_cfg.n_embd), dtype=torch.float32)
    cnt = torch.zeros(n, dtype=torch.float32)

    for st in range(0, n, 448):
        en = min(st + 512, n)
        x = torch.tensor([ids[st:en]], dtype=torch.long, device=device)
        h = encoder(
            input_ids=x,
            attention_mask=torch.ones_like(x)
        ).last_hidden_state[0].float().cpu()

        feat[st:en] += h
        cnt[st:en] += 1
        if en == n:
            break

    return feat / cnt.clamp_min(1).unsqueeze(-1)

def build(texts, gold, want_champion=False):
    X, Y, C = [], [], []
    for doc_id, text in texts.items():
        _, tags = audit_doc_windows(text, gold[str(doc_id)] if str(doc_id) in gold else gold[doc_id])
        X.append(doc_features(enc_p1, text))
        Y.append(torch.tensor(tags, dtype=torch.long))

        if want_champion:
            _, probs = diag_doc_probs(champion_model, text)
            C.append(probs.argmax(-1).cpu())

    X = torch.cat(X)
    Y = torch.cat(Y)
    C = torch.cat(C) if want_champion else None
    assert len(X) == len(Y), "Feature and BIO tag lengths do not match."
    if C is not None:
        assert len(C) == len(Y), "Champion prediction and gold lengths do not match."
    return X, Y, C

Xtr, Ytr, _ = build(train_texts, train_gold)
Xva, Yva, Cva = build(val_texts, val_gold, want_champion=True)
print("Feature shapes:", tuple(Xtr.shape), tuple(Xva.shape))

# ---- Part 2: train the linear probe ----
Xtr_g = Xtr.to(device)
Ytr_g = Ytr.to(device)

mu = Xtr_g.mean(0)
sdv = Xtr_g.std(0) + 1e-6

counts = torch.bincount(
    Ytr, minlength=len(BIO_LABELS)
).float().clamp_min(1)

w = torch.sqrt(counts[0] / counts)
w[0] = 1.0

head = nn.Linear(p1_cfg.n_embd, len(BIO_LABELS)).to(device)
opt = torch.optim.AdamW(
    head.parameters(), lr=1e-2, weight_decay=1e-4
)

for step in range(1, 401):
    logits = head((Xtr_g - mu) / sdv)
    loss = F.cross_entropy(logits, Ytr_g, weight=w.to(device))
    opt.zero_grad()
    loss.backward()
    opt.step()

    if step % 100 == 0:
        print(f"step {step} loss {loss.item():.4f}")

with torch.no_grad():
    P_tr = head((Xtr_g - mu) / sdv).argmax(-1).cpu()
    P_va = head((Xva.to(device) - mu) / sdv).argmax(-1).cpu()

# ---- Part 3: token-level type P/R/F1 (B/I merged) ----
def prf(gold_ids, pred_ids):
    g_t = type_of_id[gold_ids]
    p_t = type_of_id[pred_ids]
    out = {}

    for t in ENT:
        j = TYPES.index(t)
        g = g_t == j
        p = p_t == j
        tp = (g & p).sum().item()
        P = tp / max(p.sum().item(), 1)
        R = tp / max(g.sum().item(), 1)
        F1 = 2 * P * R / (P + R) if P + R else 0.0
        out[t] = (P, R, F1)

    return out

probe_va = prf(Yva, P_va)
champ_va = prf(Yva, Cva)
probe_tr = prf(Ytr, P_tr)

rows = [{
    "label": t,
    "probe_val_P": probe_va[t][0],
    "probe_val_R": probe_va[t][1],
    "probe_val_F1": probe_va[t][2],
    "champ_val_P": champ_va[t][0],
    "champ_val_R": champ_va[t][1],
    "champ_val_F1": champ_va[t][2],
    "probe_train_F1": probe_tr[t][2],
} for t in ENT]

pd.set_option("display.width", 220)
print(pd.DataFrame(rows).round(3).to_string(index=False))

del Xtr_g, Ytr_g, Xtr, Xva
gc.collect()
torch.cuda.empty_cache()
print("\nLinear-probe experiment complete; no prediction files were written.")


In [ ]:

# FIX — Define the champion counsel baseline before calculating gain

if "c_f_thr" not in globals():
    if "champion_model" not in globals() or "val_texts" not in globals():
        raise NameError(
            "Need champion_model and val_texts. Run the champion-loading "
            "and validation setup cells first."
        )

    if "diag_doc_probs" not in globals():
        raise NameError(
            "diag_doc_probs is not defined. Run the original inference-helper "
            "cell before calculating the champion baseline."
        )

    # Use the notebook's original inference helper to recover champion
    # token predictions, then calculate the counsel baseline with its
    # existing evaluation function.
    print(
        "c_f_thr is missing. Do not calculate gain yet: first run the "
        "cell that evaluates champion predictions and defines c_f_thr."
    )
else:
    gain = (
        r_va["A.COUNSEL"] + r_va["R.COUNSEL"]
        - c_f_thr["A.COUNSEL"] - c_f_thr["R.COUNSEL"]
    ) / 12

    print(
        f"Macro-F1 delta on validation if counsel predictions are replaced: "
        f"{gain:+.4f}"
    )


In [ ]:

# FIX — Correct counsel-rule comparison
r_va = globals()["r_va"]

assert isinstance(r_va, dict), "r_va should contain per-label rule F1 scores."
assert all(k in r_va for k in ("A.COUNSEL", "R.COUNSEL")), (
    f"Unexpected r_va labels: {list(r_va.keys())}"
)

print("Fold 2 counsel-rule results:")
for lab in ("A.COUNSEL", "R.COUNSEL"):
    print(f"  {lab}: F1={r_va[lab]:.4f}")

print(f"\nMean F1 across the two counsel labels: "
      f"{(r_va['A.COUNSEL'] + r_va['R.COUNSEL']) / 2:.4f}")

print("\nChampion overall macro-F1 currently stored in c_f1:", c_f1)
print(
    "These values are not directly comparable. To calculate the true gain, "
    "we must evaluate the champion per-label on fold 2 with the same "
    "evaluation function used for the rules."
)


In [ ]:

# Inspect the notebook's evaluation helpers to find the correct baseline method
import inspect

for name in ("eval_rule", "score", "as_scored", "merge_policy",
             "predict_texts", "diag_doc_probs"):
    obj = globals().get(name)
    print(f"\n{name}: {type(obj).__name__ if obj is not None else 'MISSING'}")
    if callable(obj):
        try:
            print(inspect.signature(obj))
            print(inspect.getsource(obj)[:1800])
        except Exception as e:
            print("Source unavailable:", e)


In [ ]:

# CELL — Exact-span counsel baseline for the champion on fold 2
from collections import defaultdict, Counter
import torch

assert "champion_model" in globals()
assert "val_texts" in globals() and "val_gold" in globals()
assert "tokenizer" in globals() and "BIO_LABELS" in globals()
assert "prf_sets" in globals(), "Missing original prf_sets evaluation helper."

# Gold spans for the two counsel labels.
G = defaultdict(set)
for d, ents in val_gold.items():
    for e in ents:
        if len(e) >= 3 and e[2] in ("A.COUNSEL", "R.COUNSEL"):
            G[e[2]].add((d, int(e[0]), int(e[1])))

P = defaultdict(set)
id_to_bio = {i: lab for i, lab in enumerate(BIO_LABELS)}

def decode_bio_spans(text, offsets, pred_ids):
    spans = []
    current_label = None
    current_start = None
    current_end = None

    for (start, end), idx in zip(offsets, pred_ids):
        bio = id_to_bio[int(idx)]
        if start == end:
            continue

        if bio == "O":
            if current_label is not None:
                spans.append((current_start, current_end, current_label))
                current_label = None
            continue

        if "-" not in bio:
            continue

        prefix, label = bio.split("-", 1)
        if prefix == "B" or label != current_label:
            if current_label is not None:
                spans.append((current_start, current_end, current_label))
            current_label = label
            current_start, current_end = start, end
        elif prefix == "I" and current_label == label:
            current_end = end

    if current_label is not None:
        spans.append((current_start, current_end, current_label))

    return spans

for d, text in val_texts.items():
    offsets, probs = diag_doc_probs(champion_model, text)
    pred_ids = probs.argmax(-1).tolist()

    for s, e, lab in decode_bio_spans(text, offsets, pred_ids):
        if lab in ("A.COUNSEL", "R.COUNSEL"):
            P[lab].add((d, int(s), int(e)))

for lab in ("A.COUNSEL", "R.COUNSEL"):
    precision, recall, f1, n_pred = prf_sets(G[lab], P[lab])
    print(
        f"{lab:10s} champion P={precision:.3f} "
        f"R={recall:.3f} F1={f1:.3f} pred={n_pred}"
    )

champion_counsel_f1 = {}
for lab in ("A.COUNSEL", "R.COUNSEL"):
    champion_counsel_f1[lab] = prf_sets(G[lab], P[lab])[2]

print("\nRule F1:", r_va)
print("Champion F1:", champion_counsel_f1)

for lab in ("A.COUNSEL", "R.COUNSEL"):
    print(
        f"{lab}: rule-minus-champion F1 = "
        f"{r_va[lab] - champion_counsel_f1[lab]:+.4f}"
    )


In [ ]:
from collections import Counter

print("=== DATA ALIGNMENT ===")
print("val_texts:", len(val_texts), "documents")
print("val_gold:", len(val_gold), "documents")
print("text ID examples:", [(repr(k), type(k).__name__) for k in list(val_texts)[:5]])
print("gold ID examples:", [(repr(k), type(k).__name__) for k in list(val_gold)[:5]])

text_ids = set(val_texts)
gold_ids = set(val_gold)

print("IDs shared directly:", len(text_ids & gold_ids))
print("Text IDs missing gold:", len(text_ids - gold_ids))
print("Gold IDs missing text:", len(gold_ids - text_ids))

print("\n=== GOLD COUNTS ===")
for lab in ("A.COUNSEL", "R.COUNSEL"):
    spans = [
        (d, int(e[0]), int(e[1]))
        for d, ents in val_gold.items()
        for e in ents
        if len(e) >= 3 and e[2] == lab
    ]
    print(lab, "gold spans:", len(spans), "examples:", spans[:5])

print("\n=== OFFSET SANITY CHECK ===")
checked = 0
for d, text in list(val_texts.items())[:3]:
    enc = tokenizer.encode(text, add_special_tokens=False)
    print(
        "doc:", repr(d),
        "text length:", len(text),
        "token count:", len(enc.ids),
        "first offsets:", enc.offsets[:8],
        "text at first offsets:",
        [text[s:e] for s, e in enc.offsets[:8] if e > s]
    )
    checked += 1

print("\n=== GOLD SPAN BOUNDS ===")
for lab in ("A.COUNSEL", "R.COUNSEL"):
    valid = invalid = 0
    for d, ents in val_gold.items():
        if d not in val_texts:
            continue
        for e in ents:
            if len(e) >= 3 and e[2] == lab:
                s, end = int(e[0]), int(e[1])
                if 0 <= s < end <= len(val_texts[d]):
                    valid += 1
                else:
                    invalid += 1
    print(lab, "valid:", valid, "invalid:", invalid)

In [ ]:
from collections import Counter, defaultdict
import torch

# Check that the expected objects exist
assert "champion_model" in globals()
assert "diag_doc_probs" in globals()
assert "BIO_LABELS" in globals()

id_to_bio = {i: label for i, label in enumerate(BIO_LABELS)}

def decode_bio_debug(offsets, pred_ids):
    spans = []
    current_label = None
    current_start = None
    current_end = None

    for (start, end), idx in zip(offsets, pred_ids):
        if start == end:
            continue

        bio = id_to_bio[int(idx)]

        if bio == "O" or "-" not in bio:
            if current_label is not None:
                spans.append((current_start, current_end, current_label))
                current_label = None
            continue

        prefix, label = bio.split("-", 1)

        if prefix == "B" or label != current_label:
            if current_label is not None:
                spans.append((current_start, current_end, current_label))
            current_label = label
            current_start, current_end = start, end
        elif prefix == "I" and label == current_label:
            current_end = end

    if current_label is not None:
        spans.append((current_start, current_end, current_label))

    return spans

pred_counts = Counter()
pred_examples = defaultdict(list)

for doc_id, text in val_texts.items():
    offsets, probs = diag_doc_probs(champion_model, text)
    pred_ids = probs.argmax(dim=-1).tolist()

    for start, end, label in decode_bio_debug(offsets, pred_ids):
        pred_counts[label] += 1

        if label in ("A.COUNSEL", "R.COUNSEL") and len(pred_examples[label]) < 10:
            pred_examples[label].append({
                "doc_id": doc_id,
                "start": start,
                "end": end,
                "text": text[start:end]
            })

print("=== Predicted BIO label counts ===")
print(pred_counts.most_common())

print("\n=== Counsel prediction examples ===")
for label in ("A.COUNSEL", "R.COUNSEL"):
    print(f"\n{label}: {pred_counts[label]} predicted spans")
    for example in pred_examples[label]:
        print(example)

print("\n=== Gold examples ===")
for label in ("A.COUNSEL", "R.COUNSEL"):
    shown = 0
    for doc_id, entities in val_gold.items():
        text = val_texts[doc_id]
        for entity in entities:
            if len(entity) >= 3 and entity[2] == label:
                start, end = int(entity[0]), int(entity[1])
                print(label, repr(text[start:end]), (start, end))
                shown += 1
                if shown >= 5:
                    break
        if shown >= 5:
            break

In [ ]:
import re
import random
from collections import defaultdict

random.seed(1)

# Define the honorific pattern used by the original diagnostic.
HON_BEFORE = re.compile(
    r"(?:Mr|Ms|Mrs|Shri|Sri|Smt)\.?[ \t]*$"
)

def side_state(text, s):
    ctx = text[max(0, s - 600):s]

    a = [m.start() for m in SIDE_A.finditer(ctx)]
    r = [m.start() for m in SIDE_R.finditer(ctx)]

    if not a and not r:
        return "none"
    if a and not r:
        return "A-cue only"
    if r and not a:
        return "R-cue only"

    return "both (nearer wins)"


def side_error_report(texts, gold, name, show=6):
    acc = defaultdict(lambda: [0, 0])
    wrong, missed = [], []

    for d, text in texts.items():
        gl = {
            (int(e[0]), int(e[1])): e[2]
            for e in gold[d]
            if len(e) >= 3 and e[2] in ("A.COUNSEL", "R.COUNSEL")
        }

        rule = set(rule_spans(text))

        # Side accuracy, but only on spans whose boundaries match gold.
        for s, e in rule:
            if (s, e) in gl:
                st = side_state(text, s)
                pred = side(text, s)
                ok = pred == gl[(s, e)]

                acc[st][0] += int(ok)
                acc[st][1] += 1

                if not ok:
                    wrong.append((
                        text[max(0, s - 160):s].replace("\n", " "),
                        text[s:e],
                        gl[(s, e)],
                        pred,
                        st
                    ))

        # Gold spans not proposed by the rules.
        for (s, e), lab in gl.items():
            if (s, e) not in rule:
                left = text[max(0, s - 40):s]
                missed.append((
                    left.replace("\n", " "),
                    text[s:e].replace("\n", " "),
                    lab,
                    bool(HON_BEFORE.search(left))
                ))

    print(f"\n=== {name}: side accuracy by cue state ===")
    for st, (ok, n) in sorted(acc.items()):
        print(f"  {st:20s} {ok:3d}/{n:3d} = {ok/max(n, 1):.3f}")

    print(f"\n--- Wrong-side examples (up to {show}) ---")
    for ctx, nm, gl_lab, pred, st in random.sample(
        wrong, min(show, len(wrong))
    ):
        print(
            f"[{st}] gold={gl_lab} pred={pred}\n"
            f"   context: ...{ctx!r}\n"
            f"   span: {nm!r}"
        )

    n_hon = sum(1 for item in missed if item[3])
    print(
        f"\nMissed gold spans: {len(missed)} total; "
        f"{n_hon} follow an honorific, "
        f"{len(missed) - n_hon} do not."
    )

    print(f"\n--- Missed gold examples (up to {show}) ---")
    for left, nm, lab, hon in random.sample(
        missed, min(show, len(missed))
    ):
        print(
            f"gold={lab} honorific_before={hon}\n"
            f"   left: ...{left!r}\n"
            f"   gold: {nm!r}"
        )

print("Definitions restored.")
print("SIDE_A available:", "SIDE_A" in globals())
print("SIDE_R available:", "SIDE_R" in globals())
print("rule_spans available:", "rule_spans" in globals())
print("side available:", "side" in globals())

In [ ]:
# Check that the original notebook's required objects exist.
required = [
    "side_error_report", "train_texts", "train_gold",
    "val_texts", "val_gold", "rule_spans", "side",
    "SIDE_A", "SIDE_R", "HON_BEFORE"
]
missing = [name for name in required if name not in globals()]

if missing:
    print("Missing variables/functions:", missing)
    print("Run the original notebook cells defining these objects first.")
else:
    side_error_report(
        train_texts, train_gold,
        "FOLD 1",
        show=10
    )

    side_error_report(
        val_texts, val_gold,
        "FOLD 2",
        show=10
    )

In [ ]:
from collections import defaultdict

def counsel_rule_diagnostics(texts, gold, name):
    gold_by_label = defaultdict(set)
    pred_by_label = defaultdict(set)

    for doc_id, entities in gold.items():
        for ent in entities:
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL"):
                gold_by_label[ent[2]].add(
                    (doc_id, int(ent[0]), int(ent[1]))
                )

    for doc_id, text in texts.items():
        for start, end in rule_spans(text):
            label = side(text, start)
            if label in ("A.COUNSEL", "R.COUNSEL"):
                pred_by_label[label].add(
                    (doc_id, int(start), int(end))
                )

    print(f"\n=== {name}: exact-span counsel rule evaluation ===")
    for label in ("A.COUNSEL", "R.COUNSEL"):
        g = gold_by_label[label]
        p = pred_by_label[label]
        correct = len(g & p)
        precision = correct / len(p) if p else 0.0
        recall = correct / len(g) if g else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        print(
            f"{label:10s} gold={len(g):4d} predicted={len(p):4d} "
            f"matched={correct:4d} P={precision:.3f} "
            f"R={recall:.3f} F1={f1:.3f}"
        )

counsel_rule_diagnostics(train_texts, train_gold, "FOLD 1")
counsel_rule_diagnostics(val_texts, val_gold, "FOLD 2")

In [ ]:
import re
from collections import defaultdict

# Candidate patterns only: this does not assign labels or alter model weights.
INITIAL = r"(?:[A-Z]\.?\s*){1,4}"
NAME_WORD = r"[A-Z][A-Za-z'’-]+"
NAME = rf"(?:{INITIAL})?{NAME_WORD}(?:\s+{NAME_WORD}){{0,3}}"

HONORIFIC = (
    r"(?:Mr|Mrs|Ms|Miss|Dr|Shri|Sri|Smt)\.?"
)

# A name after an honorific, or a name followed by common legal-role cues.
HON_NAME_RE = re.compile(
    rf"(?<!\w){HONORIFIC}\s+({NAME})",
    flags=re.IGNORECASE
)

ROLE_CUE_RE = re.compile(
    r"\b(?:counsel\s+for|appearing\s+for|for\s+the\s+"
    r"(?:appellant|respondent|petitioner|applicant|state))\b",
    flags=re.IGNORECASE
)

def expanded_candidates(text):
    spans = set()

    for m in HON_NAME_RE.finditer(text):
        # Capture group contains only the candidate name, not the honorific.
        s, e = m.span(1)
        spans.add((s, e))

    return spans

def candidate_recall_report(texts, gold, name):
    gold_spans = set()
    old_spans = set()
    new_spans = set()

    for doc_id, text in texts.items():
        for ent in gold[doc_id]:
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL"):
                gold_spans.add((doc_id, int(ent[0]), int(ent[1])))

        for s, e in rule_spans(text):
            old_spans.add((doc_id, int(s), int(e)))

        for s, e in expanded_candidates(text):
            new_spans.add((doc_id, int(s), int(e)))

    old_hits = len(gold_spans & old_spans)
    new_hits = len(gold_spans & new_spans)
    union_hits = len(gold_spans & (old_spans | new_spans))

    print(f"\n{name}")
    print("Gold counsel spans:", len(gold_spans))
    print("Original-rule matches:", old_hits)
    print("Honorific-candidate matches:", new_hits)
    print("Union matches:", union_hits)
    print("Additional gold spans recovered:", union_hits - old_hits)

candidate_recall_report(train_texts, train_gold, "FOLD 1")

In [ ]:
def inspect_new_candidates(texts, gold, max_examples=30):
    shown = 0

    for doc_id, text in texts.items():
        gold_map = {
            (int(e[0]), int(e[1])): e[2]
            for e in gold[doc_id]
            if len(e) >= 3 and e[2] in ("A.COUNSEL", "R.COUNSEL")
        }
        old = set(rule_spans(text))

        for s, e in sorted(expanded_candidates(text)):
            if (s, e) in old:
                continue

            snippet = text[max(0, s-45):min(len(text), e+45)]
            label = gold_map.get((s, e), "NOT EXACT GOLD")

            print(
                f"doc={doc_id} gold={label}\n"
                f"  candidate={text[s:e]!r}\n"
                f"  context={snippet!r}"
            )
            shown += 1
            if shown >= max_examples:
                return

inspect_new_candidates(train_texts, train_gold)

In [ ]:
import re
from collections import Counter, defaultdict

def inspect_missed_contexts(texts, gold, show=50):
    missed = []

    for doc_id, text in texts.items():
        existing = set(rule_spans(text))

        for ent in gold[doc_id]:
            if len(ent) < 3 or ent[2] not in ("A.COUNSEL", "R.COUNSEL"):
                continue

            s, e, label = int(ent[0]), int(ent[1]), ent[2]

            if (s, e) in existing:
                continue

            left = text[max(0, s - 140):s].replace("\n", " ")
            name = text[s:e].replace("\n", " ")
            right = text[e:min(len(text), e + 70)].replace("\n", " ")

            # Broad contextual indicators, not predictions.
            cue = bool(re.search(
                r"\b(counsel|advocate|appellant|respondent|petitioner|"
                r"appearing|for the state|for the applicant|"
                r"senior counsel|learned counsel)\b",
                left,
                re.IGNORECASE
            ))

            missed.append({
                "doc": doc_id,
                "label": label,
                "name": name,
                "left": left,
                "right": right,
                "cue": cue
            })

    print("Missed gold spans:", len(missed))
    print("With counsel/party context:", sum(x["cue"] for x in missed))
    print("Without those context cues:", sum(not x["cue"] for x in missed))

    print("\n--- Missed spans with relevant context ---")
    selected = [x for x in missed if x["cue"]]
    for x in selected[:show]:
        print(
            f"\n[{x['label']}] {x['name']!r} | doc={x['doc']}\n"
            f"LEFT:  ...{x['left']!r}\n"
            f"RIGHT: {x['right']!r}"
        )

inspect_missed_contexts(train_texts, train_gold, show=30)

In [ ]:
def counsel_context_counts(texts, gold):
    patterns = {
        "for the appellant/respondent": (
            r"\bfor the (?:learned )?(?:appellant|respondent|"
            r"petitioner|applicant|complainant|accused|state)\b"
        ),
        "counsel/advocate": r"\b(?:counsel|advocate)s?\b",
        "appearing for": r"\bappearing for\b",
        "senior counsel": r"\bsenior counsel\b",
        "party heading": (
            r"\b(?:for the appellant|for the respondent|"
            r"for the petitioners?|for the state)\s*:"
        ),
        "learned counsel": r"\blearned counsel\b",
    }

    counts = Counter()
    total = Counter()

    for doc_id, text in texts.items():
        for ent in gold[doc_id]:
            if len(ent) < 3 or ent[2] not in ("A.COUNSEL", "R.COUNSEL"):
                continue

            s = int(ent[0])
            ctx = text[max(0, s - 220):s]

            for name, pattern in patterns.items():
                if re.search(pattern, ctx, re.IGNORECASE):
                    counts[name] += 1
                total[name] += 1

    print("Gold counsel spans preceded by each cue:")
    for name, count in counts.most_common():
        print(f"{name:32s} {count}")

counsel_context_counts(train_texts, train_gold)

In [ ]:
import re
from collections import defaultdict

# Candidate name: supports initials, multi-word names, and common punctuation.
INITIAL = r"(?:[A-Z]\.?\s*){1,4}"
WORD = r"[A-Z][A-Za-z'’\-]+"
NAME_PATTERN = rf"(?:{INITIAL})?{WORD}(?:\s+{WORD}){{0,3}}"

# Explicit legal representation contexts.
SECTION_RE = re.compile(
    r"\b(?:"
    r"counsel\s+for\s+(?:the\s+)?(?:appellant|respondent|petitioner|"
    r"applicant|complainant|accused|state|opposite\s+party)s?"
    r"|for\s+(?:the\s+)?(?:appellant|respondent|petitioner|applicant|"
    r"complainant|accused|state|opposite\s+party)s?"
    r"|appearing\s+for"
    r"|agent\s+for\s+(?:the\s+)?(?:appellant|respondent|petitioner|"
    r"applicant|respondent)s?"
    r")\s*[:\-]?\s*",
    re.IGNORECASE
)

# Stop at a clear section boundary to avoid collecting unrelated prose.
SECTION_STOP_RE = re.compile(
    r"\b(?:Hon['’]?ble|JUDGMENT|ORDER|CORAM|"
    r"the\s+following\s+order|judgment\s+was\s+delivered)\b",
    re.IGNORECASE
)

def counsel_list_candidates(text):
    spans = set()

    for section_match in SECTION_RE.finditer(text):
        start = section_match.end()
        limit = min(len(text), start + 500)
        region = text[start:limit]

        stop = SECTION_STOP_RE.search(region)
        if stop:
            region = region[:stop.start()]

        # Lists commonly end at a line break, a long gap, or a new heading.
        # Keep this deliberately conservative.
        region = re.split(
            r"\n\s*\n|(?:\s{3,})|"
            r"\b(?:versus|JUDGMENT|ORDER)\b",
            region,
            maxsplit=1,
            flags=re.IGNORECASE
        )[0]

        for match in re.finditer(NAME_PATTERN, region):
            s = start + match.start()
            e = start + match.end()
            candidate = text[s:e].strip()

            if not candidate:
                continue

            # Reject obvious generic/legal role words.
            if candidate.lower() in {
                "appellant", "respondent", "petitioner", "applicant",
                "state", "court", "justice", "judge", "counsel",
                "advocate", "senior counsel", "learned counsel"
            }:
                continue

            spans.add((s, e))

    return spans

In [ ]:
def evaluate_candidate_expansion(texts, gold, name):
    gold_spans = set()
    old_spans = set()
    extra_spans = set()

    for doc_id, text in texts.items():
        for ent in gold[doc_id]:
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL"):
                gold_spans.add((doc_id, int(ent[0]), int(ent[1])))

        for s, e in rule_spans(text):
            old_spans.add((doc_id, int(s), int(e)))

        for s, e in counsel_list_candidates(text):
            extra_spans.add((doc_id, int(s), int(e)))

    combined = old_spans | extra_spans

    def report(label, pred):
        tp = len(gold_spans & pred)
        p = tp / len(pred) if pred else 0.0
        r = tp / len(gold_spans) if gold_spans else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        print(
            f"{label:22s} TP={tp:3d} pred={len(pred):4d} "
            f"P={p:.3f} R={r:.3f} F1={f:.3f}"
        )

    print(f"\n=== {name} ===")
    report("Original rule", old_spans)
    report("List candidates", extra_spans)
    report("Union candidates", combined)

    print("\nExamples of newly recovered gold spans:")
    recovered = sorted((gold_spans & combined) - old_spans)
    for doc_id, s, e in recovered[:20]:
        print(doc_id, repr(texts[doc_id][s:e]))

    return {
        "gold": gold_spans,
        "original": old_spans,
        "expanded": extra_spans,
        "combined": combined,
    }

fold1_candidate_eval = evaluate_candidate_expansion(
    train_texts, train_gold, "FOLD 1"
)

In [ ]:
import re

# Cues that strongly associate a nearby name with legal representation.
AFTER_NAME_CUE = re.compile(
    r"^\s*(?:,?\s*\([^)]{0,50}\s+with\s+him\)|"
    r"\s*,?\s*(?:Sr\.?\s+Adv\.?|Adv\.?|Advocate|"
    r"Solicitor[- ]General|Attorney[- ]General))?"
    r"\s*,?\s*(?:for\s+(?:the\s+)?"
    r"(?:appellant|respondent|petitioner|applicant|"
    r"complainant|accused|state|intervener)s?\b)",
    re.IGNORECASE
)

def filtered_counsel_candidates(text):
    accepted = set()

    for s, e in counsel_list_candidates(text):
        # Inspect a short window after the name.
        after = text[e:min(len(text), e + 100)]

        # The original list-candidate function is deliberately broad;
        # accept only candidates followed by a representation cue.
        if AFTER_NAME_CUE.search(after):
            accepted.add((s, e))

    return accepted


def evaluate_filtered_expansion(texts, gold, name):
    gold_spans = set()
    original = set()
    filtered = set()

    for doc_id, text in texts.items():
        for ent in gold[doc_id]:
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL"):
                gold_spans.add((doc_id, int(ent[0]), int(ent[1])))

        original.update(
            (doc_id, int(s), int(e)) for s, e in rule_spans(text)
        )
        filtered.update(
            (doc_id, int(s), int(e))
            for s, e in filtered_counsel_candidates(text)
        )

    combined = original | filtered

    def score(pred):
        tp = len(gold_spans & pred)
        p = tp / len(pred) if pred else 0.0
        r = tp / len(gold_spans) if gold_spans else 0.0
        f = 2*p*r/(p+r) if p+r else 0.0
        return tp, len(pred), p, r, f

    for label, pred in [
        ("Original", original),
        ("Filtered only", filtered),
        ("Combined", combined)
    ]:
        tp, n, p, r, f = score(pred)
        print(
            f"{label:15s} TP={tp:3d} pred={n:4d} "
            f"P={p:.3f} R={r:.3f} F1={f:.3f}"
        )

evaluate_filtered_expansion(train_texts, train_gold, "FOLD 1")

In [ ]:
# Inspect new candidates introduced by the filtered expansion.
shown = 0

for doc_id, text in train_texts.items():
    old = {(int(s), int(e)) for s, e in rule_spans(text)}
    new = filtered_counsel_candidates(text)

    gold_map = {
        (int(ent[0]), int(ent[1])): ent[2]
        for ent in train_gold[doc_id]
        if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL")
    }

    for s, e in sorted(new - old):
        print(
            f"doc={doc_id} "
            f"gold={gold_map.get((s, e), 'NOT EXACT GOLD')}\n"
            f"  candidate={text[s:e]!r}\n"
            f"  context={text[max(0, s-90):min(len(text), e+100)]!r}"
        )
        shown += 1
        if shown >= 30:
            break

    if shown >= 30:
        break

if shown == 0:
    print("No additional candidates were introduced.")

In [ ]:
import re

BAD_EXACT = {
    "india", "agent", "panel lawyer", "advocate", "counsel",
    "senior counsel", "learned counsel", "solicitor general",
    "attorney general", "justice", "judge", "court", "state",
    "appellant", "respondent", "petitioner", "applicant",
    "intervener", "for the appellant", "for the respondent",
}

BAD_TRAILING = re.compile(
    r"\s+(?:counsel|advocate|lawyer|justice|judge|court|"
    r"submitted|contended|argued|appearing|for|in|on|also|"
    r"were|was|has|have|and|or|the|his|her|their)\s*$",
    re.IGNORECASE
)

def clean_candidate_span(text, s, e):
    candidate = text[s:e].strip()

    if candidate.lower() in BAD_EXACT:
        return None

    # Remove a trailing generic/legal word only if it is at the end.
    while candidate and BAD_TRAILING.search(candidate):
        candidate = BAD_TRAILING.sub("", candidate).rstrip()

    # Avoid candidates that are clearly not personal names.
    if not candidate or candidate.lower() in BAD_EXACT:
        return None

    # Require at least one alphabetic character.
    if not re.search(r"[A-Za-z]", candidate):
        return None

    # Recover the trimmed character offsets.
    left_trim = len(text[s:e]) - len(text[s:e].lstrip())
    new_s = s + left_trim
    new_e = new_s + len(candidate)

    return new_s, new_e


def cleaned_counsel_candidates(text):
    cleaned = set()

    for s, e in filtered_counsel_candidates(text):
        span = clean_candidate_span(text, s, e)
        if span is not None:
            cleaned.add(span)

    return cleaned


def evaluate_cleaned_expansion(texts, gold, name):
    gold_spans = set()
    original = set()
    cleaned = set()

    for doc_id, text in texts.items():
        for ent in gold[doc_id]:
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL"):
                gold_spans.add((doc_id, int(ent[0]), int(ent[1])))

        original.update(
            (doc_id, int(s), int(e)) for s, e in rule_spans(text)
        )
        cleaned.update(
            (doc_id, int(s), int(e))
            for s, e in cleaned_counsel_candidates(text)
        )

    def report(label, pred):
        tp = len(gold_spans & pred)
        p = tp / len(pred) if pred else 0.0
        r = tp / len(gold_spans) if gold_spans else 0.0
        f1 = 2*p*r/(p+r) if p+r else 0.0
        print(
            f"{label:18s} TP={tp:3d} pred={len(pred):4d} "
            f"P={p:.3f} R={r:.3f} F1={f1:.3f}"
        )

    combined = original | cleaned

    print(f"\n=== {name} ===")
    report("Original", original)
    report("Cleaned candidates", cleaned)
    report("Combined", combined)

evaluate_cleaned_expansion(
    train_texts, train_gold, "FOLD 1"
)

In [ ]:
def evaluate_rule_union(texts, gold, name):
    gold_by_label = defaultdict(set)
    original_by_label = defaultdict(set)
    combined_by_label = defaultdict(set)

    for doc_id, text in texts.items():
        for ent in gold[doc_id]:
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL"):
                gold_by_label[ent[2]].add(
                    (doc_id, int(ent[0]), int(ent[1]))
                )

        # Existing rules, with the original side assignment.
        for s, e in rule_spans(text):
            lab = side(text, s)
            if lab in ("A.COUNSEL", "R.COUNSEL"):
                original_by_label[lab].add((doc_id, int(s), int(e)))
                combined_by_label[lab].add((doc_id, int(s), int(e)))

        # Add only cleaned candidates not already captured by the original rules.
        for s, e in cleaned_counsel_candidates(text):
            lab = side(text, s)
            if lab in ("A.COUNSEL", "R.COUNSEL"):
                combined_by_label[lab].add((doc_id, int(s), int(e)))

    def report(title, pred_by_label):
        # Micro-aggregate exact spans across both counsel labels.
        g = gold_by_label["A.COUNSEL"] | gold_by_label["R.COUNSEL"]
        p = pred_by_label["A.COUNSEL"] | pred_by_label["R.COUNSEL"]
        tp = len(g & p)
        precision = tp / len(p) if p else 0.0
        recall = tp / len(g) if g else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )
        print(
            f"{title:18s} TP={tp:3d} pred={len(p):4d} "
            f"P={precision:.3f} R={recall:.3f} F1={f1:.3f}"
        )

        for lab in ("A.COUNSEL", "R.COUNSEL"):
            gl = gold_by_label[lab]
            pl = pred_by_label[lab]
            hit = len(gl & pl)
            pp = hit / len(pl) if pl else 0.0
            rr = hit / len(gl) if gl else 0.0
            ff = 2 * pp * rr / (pp + rr) if pp + rr else 0.0
            print(
                f"  {lab:10s} TP={hit:3d} pred={len(pl):4d} "
                f"P={pp:.3f} R={rr:.3f} F1={ff:.3f}"
            )

    print(f"\n=== {name} ===")
    report("Original", original_by_label)
    report("Cleaned + original", combined_by_label)

evaluate_rule_union(val_texts, val_gold, "FOLD 2 HELD-OUT")

In [ ]:
from collections import defaultdict

def compare_labeled_counsel(texts, gold, name):
    gold_labeled = set()
    original_labeled = set()
    combined_labeled = set()

    for doc_id, text in texts.items():
        for ent in gold[doc_id]:
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL"):
                gold_labeled.add(
                    (doc_id, int(ent[0]), int(ent[1]), ent[2])
                )

        for s, e in rule_spans(text):
            lab = side(text, s)
            if lab in ("A.COUNSEL", "R.COUNSEL"):
                item = (doc_id, int(s), int(e), lab)
                original_labeled.add(item)
                combined_labeled.add(item)

        for s, e in cleaned_counsel_candidates(text):
            lab = side(text, s)
            if lab in ("A.COUNSEL", "R.COUNSEL"):
                combined_labeled.add((doc_id, int(s), int(e), lab))

    def show(label, pred):
        tp = len(gold_labeled & pred)
        p = tp / len(pred) if pred else 0.0
        r = tp / len(gold_labeled) if gold_labeled else 0.0
        f1 = 2*p*r/(p+r) if p+r else 0.0
        print(
            f"{label:18s} exact labeled TP={tp:3d} "
            f"pred={len(pred):4d} P={p:.3f} R={r:.3f} F1={f1:.3f}"
        )

    print(f"\n=== {name}: exact labeled spans ===")
    show("Original", original_labeled)
    show("Combined", combined_labeled)
    print("New labeled predictions:", len(combined_labeled - original_labeled))
    print("New exact labeled matches:", len(
        (gold_labeled & combined_labeled) -
        (gold_labeled & original_labeled)
    ))

compare_labeled_counsel(val_texts, val_gold, "FOLD 2")

In [ ]:
def inspect_new_labeled_candidates(texts, gold, limit=40):
    shown = 0

    for doc_id, text in texts.items():
        old = {(int(s), int(e)) for s, e in rule_spans(text)}

        gold_map = {
            (int(ent[0]), int(ent[1])): ent[2]
            for ent in gold[doc_id]
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL")
        }

        for s, e in sorted(cleaned_counsel_candidates(text)):
            if (s, e) in old:
                continue

            pred = side(text, s)
            gold_label = gold_map.get((s, e), "NO EXACT MATCH")

            print(
                f"doc={doc_id} gold={gold_label} pred={pred}\n"
                f"  span={text[s:e]!r}\n"
                f"  context={text[max(0, s-180):min(len(text), e+120)]!r}\n"
            )

            shown += 1
            if shown >= limit:
                return

inspect_new_labeled_candidates(val_texts, val_gold, limit=40)

In [ ]:
import re
from collections import defaultdict

# Explicit representation phrases.
EXPLICIT_A = re.compile(
    r"\bfor\s+(?:the\s+)?(?:appellant|petitioner|applicant|"
    r"complainant|plaintiff)s?\b",
    re.IGNORECASE
)

EXPLICIT_R = re.compile(
    r"\bfor\s+(?:the\s+)?(?:respondent|accused|defendant|"
    r"opposite\s+party|state)s?\b",
    re.IGNORECASE
)

def explicit_side(text, start, end):
    """
    Look for a side cue immediately following the candidate, allowing
    other names in the same list to precede the cue.
    Returns A.COUNSEL, R.COUNSEL, or None.
    """
    # First inspect the next 100 chars, where list-ending cues commonly occur.
    right = text[end:min(len(text), end + 100)]

    # If a side cue occurs before a strong sentence/section boundary, use it.
    boundary = re.search(r"[.;\n]| {3,}", right)
    if boundary:
        right = right[:boundary.start()]

    a = EXPLICIT_A.search(right)
    r = EXPLICIT_R.search(right)

    if a and not r:
        return "A.COUNSEL"
    if r and not a:
        return "R.COUNSEL"

    # No unambiguous local explicit cue: preserve the original heuristic.
    return side(text, start)


def evaluate_side_variant(texts, gold, name):
    gold_labeled = set()
    original_labeled = set()
    explicit_labeled = set()

    for doc_id, text in texts.items():
        for ent in gold[doc_id]:
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL"):
                gold_labeled.add(
                    (doc_id, int(ent[0]), int(ent[1]), ent[2])
                )

        candidates = set(rule_spans(text)) | set(cleaned_counsel_candidates(text))

        for s, e in candidates:
            original_label = side(text, s)
            if original_label in ("A.COUNSEL", "R.COUNSEL"):
                original_labeled.add((doc_id, int(s), int(e), original_label))

            new_label = explicit_side(text, s, e)
            if new_label in ("A.COUNSEL", "R.COUNSEL"):
                explicit_labeled.add((doc_id, int(s), int(e), new_label))

    def report(label, pred):
        tp = len(gold_labeled & pred)
        p = tp / len(pred) if pred else 0.0
        r = tp / len(gold_labeled) if gold_labeled else 0.0
        f1 = 2*p*r/(p+r) if p+r else 0.0
        print(
            f"{label:20s} TP={tp:3d} pred={len(pred):4d} "
            f"P={p:.3f} R={r:.3f} F1={f1:.3f}"
        )

    print(f"\n=== {name}: labeled counsel spans ===")
    report("Original side()", original_labeled)
    report("Explicit side cue", explicit_labeled)


evaluate_side_variant(train_texts, train_gold, "FOLD 1")
evaluate_side_variant(val_texts, val_gold, "FOLD 2")

In [ ]:
def counsel_side_ablation(texts, gold, name):
    gold_labeled = set()
    original_candidates = set()
    expanded_candidates_set = set()

    for doc_id, entities in gold.items():
        for ent in entities:
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL"):
                gold_labeled.add(
                    (doc_id, int(ent[0]), int(ent[1]), ent[2])
                )

    for doc_id, text in texts.items():
        for s, e in rule_spans(text):
            original_candidates.add((doc_id, int(s), int(e)))

        for s, e in cleaned_counsel_candidates(text):
            expanded_candidates_set.add((doc_id, int(s), int(e)))

    def make_labeled(candidates, side_fn):
        pred = set()

        for doc_id, s, e in candidates:
            text = texts[doc_id]
            lab = side_fn(text, s, e)

            if lab in ("A.COUNSEL", "R.COUNSEL"):
                pred.add((doc_id, s, e, lab))

        return pred

    def score(pred):
        tp = len(gold_labeled & pred)
        p = tp / len(pred) if pred else 0.0
        r = tp / len(gold_labeled) if gold_labeled else 0.0
        f1 = 2*p*r/(p+r) if p+r else 0.0
        return tp, len(pred), p, r, f1

    combined_candidates = original_candidates | expanded_candidates_set

    variants = [
        ("Original candidates + side", original_candidates,
         lambda t, s, e: side(t, s)),
        ("Original candidates + explicit", original_candidates,
         explicit_side),
        ("Expanded candidates + side", combined_candidates,
         lambda t, s, e: side(t, s)),
        ("Expanded + explicit", combined_candidates,
         explicit_side),
    ]

    print(f"\n=== {name}: candidate/side ablation ===")
    for label, candidates, fn in variants:
        tp, n, p, r, f1 = score(make_labeled(candidates, fn))
        print(
            f"{label:32s} TP={tp:3d} pred={n:4d} "
            f"P={p:.3f} R={r:.3f} F1={f1:.3f}"
        )

counsel_side_ablation(train_texts, train_gold, "FOLD 1")
counsel_side_ablation(val_texts, val_gold, "FOLD 2")

In [ ]:
def report_explicit_side_by_label(texts, gold, name):
    gold_by_label = defaultdict(set)
    pred_by_label = defaultdict(set)

    for doc_id, entities in gold.items():
        for ent in entities:
            if len(ent) >= 3 and ent[2] in ("A.COUNSEL", "R.COUNSEL"):
                gold_by_label[ent[2]].add(
                    (doc_id, int(ent[0]), int(ent[1]))
                )

    for doc_id, text in texts.items():
        # Use the original candidate generator only.
        for s, e in rule_spans(text):
            label = explicit_side(text, s, e)
            if label in ("A.COUNSEL", "R.COUNSEL"):
                pred_by_label[label].add((doc_id, int(s), int(e)))

    print(f"\n=== {name}: original candidates + explicit side ===")
    for label in ("A.COUNSEL", "R.COUNSEL"):
        g, p = gold_by_label[label], pred_by_label[label]
        tp = len(g & p)
        precision = tp / len(p) if p else 0.0
        recall = tp / len(g) if g else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )
        print(
            f"{label:10s} gold={len(g):3d} pred={len(p):3d} "
            f"TP={tp:3d} P={precision:.3f} R={recall:.3f} F1={f1:.3f}"
        )

report_explicit_side_by_label(train_texts, train_gold, "FOLD 1")
report_explicit_side_by_label(val_texts, val_gold, "FOLD 2")

In [ ]:
import os
import json
import glob
from collections import defaultdict
from sklearn.metrics import f1_score

# 1. Locate the saved champion threshold file
threshold_files = []
for root in ("/kaggle/working", "/kaggle/input"):
    for path in glob.glob(
        os.path.join(root, "**", "*threshold*.json"),
        recursive=True
    ):
        threshold_files.append(path)

threshold_files = sorted(set(threshold_files))
print("Threshold JSON files found:")
for path in threshold_files:
    print(" ", path)

# 2. Load the recovery thresholds if champ_thr is missing
if "champ_thr" not in globals():
    candidates = [
        p for p in threshold_files
        if "champion" in os.path.basename(p).lower()
    ]

    if not candidates:
        raise FileNotFoundError(
            "No champion threshold JSON found. "
            "Check that the recovery dataset is attached to this Kaggle notebook."
        )

    print("\nInspecting candidate threshold files:")
    for path in candidates:
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        print("\nFile:", path)
        print("JSON type:", type(obj).__name__)
        if isinstance(obj, dict):
            print("Keys:", list(obj.keys())[:40])

    # Prefer a directly usable label -> threshold dictionary.
    loaded = None
    for path in candidates:
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        if isinstance(obj, dict):
            for key in ("champ_thr", "thresholds", "label_thresholds"):
                if key in obj and isinstance(obj[key], dict):
                    loaded = obj[key]
                    break

            if loaded is None and obj and all(
                isinstance(v, (int, float)) for v in obj.values()
            ):
                loaded = obj

        if loaded is not None:
            print("\nUsing thresholds from:", path)
            break

    if loaded is None:
        raise ValueError(
            "Found threshold JSON files, but couldn't safely identify "
            "a label-to-threshold mapping. Share the printed keys."
        )

    champ_thr = loaded

print("\nChampion thresholds loaded:")
print(champ_thr)

# 3. Define an exact-span macro-F1 evaluator.
# Gold entities are expected to be (start, end, label) or similar triples.
def score(predictions, gold=None):
    if gold is None or not gold:
        if "val_gold" not in globals():
            raise RuntimeError("val_gold is not available in this kernel.")
        gold = val_gold

    labels = list(BIO_LABELS) if "BIO_LABELS" in globals() else None
    if labels is None:
        labels = sorted({
            str(ent[2])
            for ents in gold.values()
            for ent in ents
            if len(ent) >= 3
        })

    gold_set = set()
    pred_set = set()

    for doc_id, entities in gold.items():
        for ent in entities:
            if len(ent) >= 3:
                gold_set.add((doc_id, int(ent[0]), int(ent[1]), str(ent[2])))

    for doc_id, entities in predictions.items():
        for ent in entities:
            if len(ent) >= 3:
                pred_set.add((doc_id, int(ent[0]), int(ent[1]), str(ent[2])))

    # Convert span sets to per-label exact-match counts.
    per_label = {}
    for label in labels:
        g = {x for x in gold_set if x[3] == label}
        p = {x for x in pred_set if x[3] == label}
        tp = len(g & p)
        precision = tp / len(p) if p else 0.0
        recall = tp / len(g) if g else 0.0
        per_label[label] = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

    macro = sum(per_label.values()) / len(per_label) if per_label else 0.0
    return macro, per_label

print("\nEvaluator defined. No predictions or model weights were changed.")

In [ ]:
import json
import os

threshold_path = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-persistent-ner-checkpoints/champion_thresholds.json"
)
recovery_path = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-working-backup-20261010-215243/"
    "cs781_recovery/champion_thresholds_recovery.json"
)

with open(threshold_path, "r", encoding="utf-8") as f:
    champion_config = json.load(f)

print("=== CHAMPION CONFIGURATION ===")
print("Checkpoint:", champion_config.get("checkpoint"))
print("Decoder:", champion_config.get("decoder"))
print("Saved validation macro-F1:", champion_config.get("validation_macro_f1"))
print("Thresholds match loaded champ_thr:",
      champion_config.get("thresholds") == champ_thr)

if os.path.exists(recovery_path):
    with open(recovery_path, "r", encoding="utf-8") as f:
        recovery_thresholds = json.load(f)
    print("\n=== RECOVERY THRESHOLDS ===")
    print(recovery_thresholds)
    print("Recovery thresholds match champion thresholds:",
          recovery_thresholds == champ_thr)

print("\n=== REQUIRED INFERENCE FUNCTIONS ===")
for name in (
    "diag_doc_probs", "bio_viterbi_decode", "decode_spans",
    "predict_texts", "apply_thr", "as_scored",
    "score", "val_texts", "val_gold",
    "champion_model", "champ_thr"
):
    print(f"{name:24s}", "AVAILABLE" if name in globals() else "MISSING")

print("\nNo model weights or prediction files were modified.")

In [ ]:
import json
import pandas as pd

with open(
    "/kaggle/input/datasets/oipranesh/"
    "cs781-persistent-ner-checkpoints/champion_thresholds.json",
    "r",
    encoding="utf-8"
) as f:
    champion_config = json.load(f)

with open(
    "/kaggle/input/datasets/oipranesh/"
    "cs781-working-backup-20261010-215243/"
    "cs781_recovery/champion_thresholds_recovery.json",
    "r",
    encoding="utf-8"
) as f:
    recovery_thr = json.load(f)

champion_thr = champion_config["thresholds"]

comparison = pd.DataFrame({
    "champion_threshold": champion_thr,
    "recovery_threshold": recovery_thr
})
comparison["difference"] = (
    comparison["recovery_threshold"] - comparison["champion_threshold"]
)
comparison["changed"] = (
    comparison["champion_threshold"] != comparison["recovery_threshold"]
)

print("Saved champion validation macro-F1:",
      champion_config.get("validation_macro_f1"))
print("\nThreshold comparison:")
print(comparison.round(3).to_string())
print("\nLabels with different thresholds:",
      comparison.index[comparison["changed"]].tolist())
print("\nNo thresholds, model weights, or prediction files were changed.")

In [ ]:
import torch
import pandas as pd
from bisect import bisect_left, bisect_right

required = [
    "champion_model", "diag_doc_probs", "bio_viterbi_decode",
    "decode_spans", "val_texts", "val_gold", "champ_thr"
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError("Missing definitions: " + ", ".join(missing))

# Recovery thresholds were loaded in the previous cell.
if "recovery_thr" not in globals():
    raise RuntimeError(
        "recovery_thr is missing. Rerun the threshold-comparison cell first."
    )

@torch.inference_mode()
def infer_spans_for_eval(model, texts):
    result = {}
    model.eval()

    for doc_id, text in texts.items():
        offsets, probs = diag_doc_probs(model, text)

        if len(offsets) == 0:
            result[doc_id] = []
            continue

        pred_ids = torch.as_tensor(
            bio_viterbi_decode(torch.log(probs.clamp_min(1e-12))),
            dtype=torch.long
        ).tolist()

        starts = [int(a) for a, b in offsets]
        ends = [int(b) for a, b in offsets]
        spans = []

        for s, e, label in decode_spans(text, offsets, pred_ids):
            token_ids = [
                i for i in range(
                    bisect_right(ends, int(s)),
                    bisect_left(starts, int(e))
                )
                if ends[i] > starts[i]
            ]

            confidence = (
                float(probs[token_ids, torch.as_tensor(pred_ids)[token_ids]].mean())
                if token_ids else 0.0
            )
            spans.append((int(s), int(e), str(label), confidence))

        result[doc_id] = spans

    return result


def threshold_spans(raw, thresholds):
    return {
        doc_id: [
            (s, e, label)
            for s, e, label, confidence in entities
            if confidence >= float(thresholds.get(label, 0.0))
        ]
        for doc_id, entities in raw.items()
    }


def exact_span_report(predictions, gold):
    labels = sorted({
        str(ent[2])
        for entities in gold.values()
        for ent in entities
        if len(ent) >= 3
    } | {
        str(ent[2])
        for entities in predictions.values()
        for ent in entities
        if len(ent) >= 3
    })

    gold_set = {
        (doc_id, int(ent[0]), int(ent[1]), str(ent[2]))
        for doc_id, entities in gold.items()
        for ent in entities if len(ent) >= 3
    }
    pred_set = {
        (doc_id, int(ent[0]), int(ent[1]), str(ent[2]))
        for doc_id, entities in predictions.items()
        for ent in entities if len(ent) >= 3
    }

    rows = []
    for label in labels:
        g = {x for x in gold_set if x[3] == label}
        p = {x for x in pred_set if x[3] == label}
        tp = len(g & p)
        precision = tp / len(p) if p else 0.0
        recall = tp / len(g) if g else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )
        rows.append({
            "label": label,
            "gold": len(g),
            "predicted": len(p),
            "TP": tp,
            "precision": precision,
            "recall": recall,
            "F1": f1
        })

    report = pd.DataFrame(rows).set_index("label")
    return float(report["F1"].mean()), report


# Run model inference once; threshold comparisons reuse these outputs.
raw_val_predictions = infer_spans_for_eval(champion_model, val_texts)

champion_predictions = threshold_spans(raw_val_predictions, champ_thr)
recovery_predictions = threshold_spans(raw_val_predictions, recovery_thr)

champion_macro_eval, champion_report_eval = exact_span_report(
    champion_predictions, val_gold
)
recovery_macro_eval, recovery_report_eval = exact_span_report(
    recovery_predictions, val_gold
)

print("=== EXACT-SPAN EVALUATION ON val_gold ===")
print(f"Champion thresholds macro-F1: {champion_macro_eval:.4f}")
print(f"Recovery thresholds macro-F1: {recovery_macro_eval:.4f}")
print(f"Difference (recovery - champion): "
      f"{recovery_macro_eval - champion_macro_eval:+.4f}")

print("\n=== PER-LABEL F1 ===")
comparison = pd.DataFrame({
    "champion_F1": champion_report_eval["F1"],
    "recovery_F1": recovery_report_eval["F1"],
    "champion_predicted": champion_report_eval["predicted"],
    "recovery_predicted": recovery_report_eval["predicted"],
}).round(3)
print(comparison.to_string())

print("\nNo training, threshold-file edits, or prediction-file writes occurred.")

In [ ]:
# CS781 — Restore V5 counsel helpers from the latest notebook
# No training, checkpoint edits, or file writes.

import re
import pandas as pd
import torch
from bisect import bisect_left, bisect_right

required = [
    "champion_model", "champ_thr", "diag_doc_probs",
    "bio_viterbi_decode", "decode_spans",
    "val_texts", "val_gold", "train_texts", "train_gold",
    "rule_spans", "score"
]
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(
        "Still missing foundational definitions: " + ", ".join(missing)
    )

# ---- Exact V5 side-assignment setup from the latest notebook ----
PARTY_A = r"(?:petitioners?|appellants?|applicants?|plaintiffs?|complainants?|prosecution|revisionists?)"
PARTY_R = r"(?:respondents?|defendants?|accused|non-\s*applicants?|opposite\s+part(?:y|ies))"
FOR = r"(?:for|on\s+behalf\s+of|appearing\s+for|representing)"
MODS = r"(?:(?:learned|first|second|third|\d+\w*|all|other)\s+)*"

CUE_A = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_A}", re.I)
CUE_R = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_R}", re.I)
HDR_A = re.compile(rf"{PARTY_A}s?[\s\.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HDR_R = re.compile(rf"{PARTY_R}s?[\s\.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HON_TAIL = re.compile(r"(?:Mr|Ms|Mrs|Shri|Sri|Smt)\.?[ \t]*$")

A_, R_ = "A.COUNSEL", "R.COUNSEL"
ORDER = ["hdr", "right", "left", "inherit"]

def _cues(text, lo, hi):
    out = [(m.start(), m.end(), A_) for m in CUE_A.finditer(text, lo, hi)]
    out += [(m.start(), m.end(), R_) for m in CUE_R.finditer(text, lo, hi)]
    return out

def assign_side(text, s, e, order, prev, W_L=300, W_R=200):
    for step in order:
        if step == "right":
            cues = _cues(text, e, min(len(text), e + W_R))
            if cues:
                return min(cues)[2], "right"
        elif step == "left":
            cues = _cues(text, max(0, s - W_L), s)
            cues = [x for x in cues if x[1] <= s]
            if cues:
                return max(cues, key=lambda x: x[1])[2], "left"
        elif step == "hdr":
            tail = HON_TAIL.sub("", text[max(0, s - 80):s])
            if HDR_A.search(tail):
                return A_, "hdr"
            if HDR_R.search(tail):
                return R_, "hdr"
        elif step == "inherit":
            if prev and s - prev[0] <= 200 and prev[2] != "default":
                return prev[1], "inherit"
    return R_, "default"

def counsel_spans(text, policy):
    out, prev = [], None
    for s, e in sorted(rule_spans(text)):
        lab, src = assign_side(text, s, e, ORDER, prev)
        prev = (e, lab, src)
        if src == "default":
            if policy == "drop":
                continue
            if policy == "A":
                lab = A_
        out.append((int(s), int(e), lab))
    return out

def counsel_f1(texts, gold, policy):
    G = {A_: set(), R_: set()}
    P = {A_: set(), R_: set()}

    for d, ents in gold.items():
        for ent in ents:
            if len(ent) >= 3 and ent[2] in G:
                G[ent[2]].add((d, int(ent[0]), int(ent[1])))

    for d, text in texts.items():
        for s, e, lab in counsel_spans(text, policy):
            P[lab].add((d, s, e))

    result = {}
    for lab in (A_, R_):
        tp = len(G[lab] & P[lab])
        p = tp / max(len(P[lab]), 1)
        r = tp / max(len(G[lab]), 1)
        result[lab] = 2*p*r/(p+r) if p+r else 0.0
    return result

# Choose fallback policy using fold 1 only.
fold1_policy_scores = {
    p: counsel_f1(train_texts, train_gold, p)
    for p in ("R", "A", "drop")
}
chosen = max(
    fold1_policy_scores,
    key=lambda p: (
        fold1_policy_scores[p][A_] + fold1_policy_scores[p][R_],
        p == "R"
    )
)

print("Fold-1 counsel policy selection:")
for p, f in fold1_policy_scores.items():
    print(f"{p:5s} A={f[A_]:.3f} R={f[R_]:.3f} "
          f"mean={(f[A_]+f[R_])/2:.3f}")
print("Selected policy:", chosen)

# Reconstruct champion inference helpers.
@torch.inference_mode()
def predict_texts(model, texts):
    result = {}
    model.eval()

    for doc_id, text in texts.items():
        offsets, probs = diag_doc_probs(model, text)
        if len(offsets) == 0:
            result[doc_id] = []
            continue

        pred_ids = torch.as_tensor(
            bio_viterbi_decode(torch.log(probs.clamp_min(1e-12))),
            dtype=torch.long
        )
        starts = [a for a, b in offsets]
        ends = [b for a, b in offsets]
        spans = []

        for s, e, lab in decode_spans(text, offsets, pred_ids.tolist()):
            ids = [
                i for i in range(bisect_right(ends, s), bisect_left(starts, e))
                if ends[i] > starts[i]
            ]
            conf = float(probs[ids, pred_ids[ids]].mean()) if ids else 0.0
            spans.append((int(s), int(e), lab, conf))

        result[doc_id] = spans
    return result

def apply_thr(raw, thresholds):
    return {
        d: [(s, e, lab) for s, e, lab, conf in spans
            if conf >= thresholds.get(lab, 0.0)]
        for d, spans in raw.items()
    }

def as_scored(predictions):
    return {
        d: [(s, e, lab, 1.0) for s, e, lab in spans]
        for d, spans in predictions.items()
    }

def merge_policy(texts, champion_filtered, policy):
    result = {}
    for d, text in texts.items():
        rules = counsel_spans(text, policy)
        kept = [
            ent for ent in champion_filtered[d]
            if ent[2] not in (A_, R_)
        ]
        kept = [
            ent for ent in kept
            if not any(
                ent[0] < re_ and ent[1] > rs
                for rs, re_, _ in rules
            )
        ]
        result[d] = sorted(set(kept + rules))
    return result

# Evaluate once on fold 2.
raw_fold2 = predict_texts(champion_model, val_texts)
c_filt = apply_thr(raw_fold2, champ_thr)

base_macro, base_f1 = score(as_scored(c_filt), {})
counsel_predictions = merge_policy(val_texts, c_filt, chosen)
counsel_macro, counsel_f1 = score(as_scored(counsel_predictions), {})

print("\nFOLD-2 FULL-NER RESULT")
print(f"Champion:         {base_macro:.4f}")
print(f"Champion + V5:    {counsel_macro:.4f}")
print(f"Delta:            {counsel_macro - base_macro:+.4f}")
print("\nPer-label F1:")
print(pd.DataFrame({
    "champion": base_f1,
    "champion_plus_V5": counsel_f1
}).round(3).to_string())

print("\nNo model weights or prediction files were modified.")

In [ ]:
# CS781 — Diagnose why every BIO-label F1 is zero

def inspect_sample(name, obj):
    print(f"\n=== {name} ===")
    print("Container type:", type(obj).__name__)

    if not isinstance(obj, dict):
        print("Sample:", repr(obj)[:800])
        return

    print("Number of documents:", len(obj))
    for doc_id, entities in list(obj.items())[:2]:
        print("Document ID:", repr(doc_id))
        print("Value type:", type(entities).__name__)
        print("Value sample:", repr(entities)[:1000])

inspect_sample("val_gold", val_gold)
inspect_sample("champion filtered predictions", c_filt)
inspect_sample("counsel predictions", counsel_predictions)

print("\n=== SCORE FUNCTION SOURCE ===")
import inspect
try:
    print(inspect.getsource(score))
except (OSError, TypeError):
    print("Python source unavailable for score().")

print("\n=== SCORE FUNCTION MODULE ===")
print(getattr(score, "__module__", None))
print("\nNo files or model weights were modified.")

In [ ]:
# CS781 — Correct exact-span evaluator for entity-level labels
# Uses val_gold's entity labels, not BIO token tags.

import pandas as pd

def score_entity_spans(predictions, gold):
    labels = sorted({
        str(ent[2])
        for entities in gold.values()
        for ent in entities
        if len(ent) >= 3
    })

    gold_set = {
        (str(doc_id), int(ent[0]), int(ent[1]), str(ent[2]))
        for doc_id, entities in gold.items()
        for ent in entities
        if len(ent) >= 3
    }

    pred_set = {
        (str(doc_id), int(ent[0]), int(ent[1]), str(ent[2]))
        for doc_id, entities in predictions.items()
        for ent in entities
        if len(ent) >= 3
    }

    rows = []
    for label in labels:
        g = {x for x in gold_set if x[3] == label}
        p = {x for x in pred_set if x[3] == label}
        tp = len(g & p)
        precision = tp / len(p) if p else 0.0
        recall = tp / len(g) if g else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )
        rows.append({
            "label": label,
            "gold": len(g),
            "predicted": len(p),
            "TP": tp,
            "precision": precision,
            "recall": recall,
            "F1": f1,
        })

    report = pd.DataFrame(rows).set_index("label")
    macro = float(report["F1"].mean()) if len(report) else 0.0
    return macro, report


# Re-evaluate existing predictions with the correct entity label set.
base_macro_fixed, base_report_fixed = score_entity_spans(c_filt, val_gold)
counsel_macro_fixed, counsel_report_fixed = score_entity_spans(
    counsel_predictions, val_gold
)

print("=== CORRECTED ENTITY-LEVEL FOLD-2 RESULTS ===")
print(f"Champion:         {base_macro_fixed:.4f}")
print(f"Champion + V5:    {counsel_macro_fixed:.4f}")
print(f"Delta:            {counsel_macro_fixed - base_macro_fixed:+.4f}")

print("\n=== PER-LABEL F1 ===")
print(pd.DataFrame({
    "champion_F1": base_report_fixed["F1"],
    "V5_F1": counsel_report_fixed["F1"],
    "champion_predicted": base_report_fixed["predicted"],
    "V5_predicted": counsel_report_fixed["predicted"],
}).round(3).to_string())

print("\nNo model weights or prediction files were modified.")

In [ ]:

# CS781 — Fold-1 cross-validation for V5
# Exact entity-span scoring; no training or checkpoint modification.

import inspect
import pandas as pd

# 1. Locate fold-1 texts and gold spans.
def first_existing(names):
    for name in names:
        if name in globals() and globals()[name] is not None:
            return name, globals()[name]
    return None, None

text_name, fold1_texts = first_existing([
    "fold1_texts", "fold1_val_texts", "val_texts_fold1",
    "texts_fold1", "f1_texts", "train_texts_fold1"
])

gold_name, fold1_gold = first_existing([
    "fold1_gold", "fold1_val_gold", "val_gold_fold1",
    "gold_fold1", "f1_gold", "train_gold_fold1"
])

print("Fold-1 text variable:", text_name)
print("Fold-1 gold variable:", gold_name)

if fold1_texts is None or fold1_gold is None:
    print("\nCould not safely identify both fold-1 inputs.")
    print("Available likely variables:")
    for name in sorted(globals()):
        if any(k in name.lower() for k in ("fold1", "fold_1", "gold", "text")):
            obj = globals()[name]
            if isinstance(obj, (dict, list, tuple)):
                print(f"  {name}: {type(obj).__name__}, len={len(obj)}")
    raise RuntimeError(
        "No evaluation was run. Use the printed variable names to map "
        "the actual fold-1 texts and gold spans, then rerun this cell."
    )

if not isinstance(fold1_texts, dict) or not isinstance(fold1_gold, dict):
    raise TypeError(
        "Expected fold-1 texts and gold spans as dictionaries keyed by document ID. "
        f"Got texts={type(fold1_texts).__name__}, "
        f"gold={type(fold1_gold).__name__}."
    )

if set(fold1_texts) != set(fold1_gold):
    raise ValueError(
        "Fold-1 text and gold document IDs do not match. "
        f"texts={len(fold1_texts)}, gold={len(fold1_gold)}."
    )

# 2. Exact-span scorer using entity labels (not B-/I- BIO tags).
def exact_entity_score(predictions, gold):
    labels = sorted({
        str(e[2])
        for ents in gold.values()
        for e in ents
        if len(e) >= 3
    })

    gs = {
        (str(doc), int(e[0]), int(e[1]), str(e[2]))
        for doc, ents in gold.items()
        for e in ents if len(e) >= 3
    }
    ps = {
        (str(doc), int(e[0]), int(e[1]), str(e[2]))
        for doc, ents in predictions.items()
        for e in ents if len(e) >= 3
    }

    rows = []
    for label in labels:
        g = {x for x in gs if x[3] == label}
        p = {x for x in ps if x[3] == label}
        tp = len(g & p)
        prec = tp / len(p) if p else 0.0
        rec = tp / len(g) if g else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        rows.append({
            "label": label, "gold": len(g), "predicted": len(p),
            "TP": tp, "precision": prec, "recall": rec, "F1": f1
        })

    report = pd.DataFrame(rows).set_index("label")
    return float(report["F1"].mean()), report

# 3. Generate baseline predictions with the existing champion.
required = ["predict_texts", "apply_thr", "champion_model", "champ_thr",
            "counsel_spans", "merge_policy"]
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing required notebook functions/variables: {missing}")

print("\nmerge_policy signature:", inspect.signature(merge_policy))
print("counsel_spans signature:", inspect.signature(counsel_spans))

fold1_base = apply_thr(
    predict_texts(champion_model, fold1_texts),
    champ_thr
)

# 4. Determine how this notebook's merge_policy expects its inputs.
# Do not guess the argument order: stop if it differs from the known call.
sig = inspect.signature(merge_policy)
params = list(sig.parameters)
print("merge_policy parameters:", params)

if len(params) != 3:
    raise RuntimeError(
        "The merge_policy signature is not the expected three-argument form. "
        "No V5 evaluation was run; share the printed signature so the call "
        "can be matched to this notebook."
    )

# The notebook's helper is expected to merge baseline spans with counsel spans.
# Try the common (base_predictions, texts, policy) signature first.
try:
    fold1_v5 = merge_policy(fold1_base, fold1_texts, "drop")
except Exception as exc:
    raise RuntimeError(
        "Baseline predictions were generated, but the V5 merge call failed. "
        "No result is reported. Share the merge_policy signature and error."
    ) from exc

# 5. Score both predictions against fold-1 gold spans.
base_f1, base_report = exact_entity_score(fold1_base, fold1_gold)
v5_f1, v5_report = exact_entity_score(fold1_v5, fold1_gold)

print("\n=== FOLD-1 FULL ENTITY-LEVEL RESULTS ===")
print(f"Champion:       {base_f1:.4f}")
print(f"Champion + V5:  {v5_f1:.4f}")
print(f"Delta:          {v5_f1 - base_f1:+.4f}")

comparison = pd.DataFrame({
    "champion_F1": base_report["F1"],
    "V5_F1": v5_report["F1"],
    "champion_predicted": base_report["predicted"],
    "V5_predicted": v5_report["predicted"],
})
print("\n=== PER-LABEL RESULTS ===")
print(comparison.round(3).to_string())

print("\nNo model weights or prediction files were modified.")


In [ ]:

import inspect

print("=== FOLD-1 EVALUATION METADATA ===")
for name in ["fold1_candidate_eval", "fold1_policy_scores",
             "gold_map", "champion_model_gold_audit"]:
    obj = globals().get(name)
    print(f"\n{name}:")
    print(repr(obj)[:5000])

print("\n=== RELEVANT FUNCTION DEFINITIONS ===")
for name in ["eval_variants", "counsel_f1", "counsel_spans",
             "predict_texts", "merge_policy"]:
    obj = globals().get(name)
    print(f"\n--- {name} ---")
    if callable(obj):
        try:
            print(inspect.getsource(obj)[:6000])
        except Exception as e:
            print("Source unavailable:", repr(e))
    else:
        print("Not defined")

print("\n=== AVAILABLE TEXT/GOLD KEYS ===")
for name in ["train_texts", "train_gold", "val_texts", "val_gold",
             "texts", "gold"]:
    obj = globals().get(name)
    if isinstance(obj, dict):
        print(name, "documents:", len(obj),
              "sample IDs:", list(obj.keys())[:5])


In [ ]:

# CS781 — Verify fold-1 split and evaluate counsel V5
# Does not modify model weights or write prediction files.

import pandas as pd

assert len(train_texts) == len(train_gold) == 35
assert set(train_texts) == set(train_gold)

# Check whether fold1_candidate_eval's original spans correspond to
# annotations in train_gold or val_gold.
candidate_eval = globals().get("fold1_candidate_eval", {})
candidate_gold = candidate_eval.get("gold", set())
candidate_original = candidate_eval.get("original", set())

def span_keys(gold_dict):
    return {
        (str(doc_id), int(ent[0]), int(ent[1]))
        for doc_id, ents in gold_dict.items()
        for ent in ents
        if len(ent) >= 3
    }

train_keys = span_keys(train_gold)
val_keys = span_keys(val_gold)

print("=== SPLIT IDENTITY CHECK ===")
print("train docs:", len(train_texts), "val docs:", len(val_texts))
print("train/val document overlap:", len(set(train_texts) & set(val_texts)))
print("fold1 candidate gold spans:", len(candidate_gold))
print("overlap with train gold:", len(candidate_gold & train_keys))
print("overlap with val gold:", len(candidate_gold & val_keys))
print("fold1 original spans:", len(candidate_original))

# Refuse to call this fold-1 validation unless the metadata supports it.
if len(candidate_gold) and len(candidate_gold & train_keys) > len(candidate_gold & val_keys):
    eval_texts, eval_gold = train_texts, train_gold
    split_name = "train_texts/train_gold (matches fold1 candidate metadata)"
elif len(candidate_gold) and len(candidate_gold & val_keys) > len(candidate_gold & train_keys):
    eval_texts, eval_gold = val_texts, val_gold
    split_name = "val_texts/val_gold (matches fold1 candidate metadata)"
else:
    raise RuntimeError(
        "The fold-1 metadata does not uniquely identify train or validation. "
        "No score was computed to avoid a misleading result."
    )

print("\nEvaluation split:", split_name)

def exact_entity_score(predictions, gold):
    labels = sorted({
        str(e[2])
        for ents in gold.values()
        for e in ents if len(e) >= 3
    })
    gs = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in gold.items()
        for e in ents if len(e) >= 3
    }
    ps = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in predictions.items()
        for e in ents if len(e) >= 3
    }
    rows = []
    for lab in labels:
        g = {x for x in gs if x[3] == lab}
        p = {x for x in ps if x[3] == lab}
        tp = len(g & p)
        pr = tp / len(p) if p else 0.0
        rc = tp / len(g) if g else 0.0
        f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        rows.append({
            "label": lab, "gold": len(g), "predicted": len(p),
            "TP": tp, "precision": pr, "recall": rc, "F1": f1
        })
    report = pd.DataFrame(rows).set_index("label")
    return float(report.F1.mean()), report

# Generate baseline predictions with the existing champion.
base_raw = predict_texts(champion_model, eval_texts)
base_filtered = apply_thr(base_raw, champ_thr)

# Correct argument order, matching the notebook's function definition.
v5_predictions = merge_policy(eval_texts, base_filtered, "drop")

base_f1, base_report = exact_entity_score(base_filtered, eval_gold)
v5_f1, v5_report = exact_entity_score(v5_predictions, eval_gold)

print("\n=== ENTITY-LEVEL RESULTS ===")
print("Split:", split_name)
print(f"Champion:       {base_f1:.4f}")
print(f"Champion + V5:  {v5_f1:.4f}")
print(f"Delta:          {v5_f1 - base_f1:+.4f}")

print("\n=== PER-LABEL COMPARISON ===")
comparison = pd.DataFrame({
    "champion_F1": base_report.F1,
    "V5_F1": v5_report.F1,
    "champion_pred": base_report.predicted,
    "V5_pred": v5_report.predicted,
})
print(comparison.round(3).to_string())

print("\nNo files or model weights were modified.")


In [ ]:

# CS781 — Held-out validation evaluation + counsel error audit
# Read-only with respect to model checkpoints and prediction files.

import pandas as pd

assert len(val_texts) == len(val_gold) == 35
assert set(val_texts).isdisjoint(set(train_texts)), (
    "Train and validation document IDs overlap; verify split provenance."
)

def exact_span_report(predictions, gold):
    labels = sorted({
        str(e[2])
        for ents in gold.values()
        for e in ents if len(e) >= 3
    })

    gs = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in gold.items()
        for e in ents if len(e) >= 3
    }
    ps = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in predictions.items()
        for e in ents if len(e) >= 3
    }

    rows = []
    for label in labels:
        g = {x for x in gs if x[3] == label}
        p = {x for x in ps if x[3] == label}
        tp = len(g & p)
        fp, fn = len(p - g), len(g - p)
        precision = tp / len(p) if p else 0.0
        recall = tp / len(g) if g else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )
        rows.append({
            "label": label, "gold": len(g), "predicted": len(p),
            "TP": tp, "FP": fp, "FN": fn,
            "precision": precision, "recall": recall, "F1": f1
        })

    report = pd.DataFrame(rows).set_index("label")
    return float(report["F1"].mean()), report, gs, ps

# Recompute on validation documents; don't reuse train-side predictions.
val_base = apply_thr(predict_texts(champion_model, val_texts), champ_thr)
val_v5 = merge_policy(val_texts, val_base, "drop")

base_macro, base_report, gold_spans, base_spans = exact_span_report(
    val_base, val_gold
)
v5_macro, v5_report, _, v5_spans = exact_span_report(val_v5, val_gold)

print("=== VALIDATION RESULTS ===")
print(f"Champion:       {base_macro:.4f}")
print(f"Champion + V5:  {v5_macro:.4f}")
print(f"Delta:          {v5_macro - base_macro:+.4f}")

print("\n=== PER-LABEL COMPARISON ===")
display(pd.DataFrame({
    "champion_F1": base_report["F1"],
    "V5_F1": v5_report["F1"],
    "champion_pred": base_report["predicted"],
    "V5_pred": v5_report["predicted"],
}).round(3))

# Counsel-only exact-match audit.
counsel_labels = {"A.COUNSEL", "R.COUNSEL"}
c_gold = {x for x in gold_spans if x[3] in counsel_labels}
c_base = {x for x in base_spans if x[3] in counsel_labels}
c_v5 = {x for x in v5_spans if x[3] in counsel_labels}

def counsel_summary(gold, pred, title):
    tp = gold & pred
    fp = pred - gold
    fn = gold - pred
    p = len(tp) / len(pred) if pred else 0.0
    r = len(tp) / len(gold) if gold else 0.0
    f = 2*p*r/(p+r) if p+r else 0.0
    print(f"\n{title}: TP={len(tp)} FP={len(fp)} FN={len(fn)} "
          f"P={p:.3f} R={r:.3f} F1={f:.3f}")
    return fp, fn

base_fp, base_fn = counsel_summary(c_gold, c_base, "Champion counsel")
v5_fp, v5_fn = counsel_summary(c_gold, c_v5, "Champion + V5 counsel")

# Print text snippets for V5 false positives and missed gold counsel spans.
def show_examples(spans, texts, heading, limit=15):
    print(f"\n=== {heading} (up to {limit}) ===")
    for doc, start, end, label in sorted(spans)[:limit]:
        text = texts.get(doc)
        if text is None:
            print((doc, start, end, label), "[document text unavailable]")
            continue
        lo, hi = max(0, start - 90), min(len(text), end + 90)
        marked = text[lo:start] + " ⟦" + text[start:end] + "⟧ " + text[end:hi]
        print(f"{label} | doc={doc} | span=({start},{end}) | {marked!r}")

show_examples(v5_fp, val_texts, "V5 COUNSEL FALSE POSITIVES")
show_examples(v5_fn, val_texts, "V5 COUNSEL MISSED GOLD SPANS")

print("\nNo model weights or prediction files were modified.")


In [ ]:

# CS781 — Inspect training/split provenance without changing anything

import inspect

print("=== MODEL OBJECT ===")
print("champion_model type:", type(champion_model))
print("model class:", champion_model.__class__.__name__)

print("\n=== POSSIBLE TRAINING METADATA ===")
keywords = (
    "train", "fold", "split", "dataset", "checkpoint",
    "best_epoch", "history", "trainer", "config"
)

for name, obj in list(globals().items()):
    if not any(k in name.lower() for k in keywords):
        continue
    if name.startswith("_"):
        continue
    if isinstance(obj, (dict, list, tuple, str, int, float, bool)):
        try:
            summary = repr(obj)
            if len(summary) > 1000:
                summary = summary[:1000] + "... [truncated]"
            print(f"\n{name} ({type(obj).__name__}): {summary}")
        except Exception:
            pass

print("\n=== AVAILABLE CHECKPOINT PATHS ===")
for name in ["CHAMPION_PATH", "MODEL_PATH", "BEST_MODEL_PATH",
             "champion_path", "checkpoint_path", "best_path"]:
    if name in globals():
        print(name, "=", repr(globals()[name]))

print("\nInspect the original training cell/config to establish whether")
print("the champion was trained on train_texts, val_texts, or another split.")
print("No weights or files were modified.")


In [ ]:

# CS781 — V5 counsel error breakdown by candidate source
# No retraining, checkpoint changes, or file writes.

from collections import defaultdict
import pandas as pd

assert "val_texts" in globals() and "val_gold" in globals()
assert "rule_spans" in globals()
assert "cleaned_counsel_candidates" in globals()
assert "explicit_side" in globals()

LABELS = {"A.COUNSEL", "R.COUNSEL"}

gold_labeled = set()
original_labeled = set()
cleaned_labeled = set()

for doc_id, entities in val_gold.items():
    for ent in entities:
        if len(ent) >= 3 and ent[2] in LABELS:
            gold_labeled.add(
                (str(doc_id), int(ent[0]), int(ent[1]), str(ent[2]))
            )

for doc_id, text in val_texts.items():
    for s, e in rule_spans(text):
        label = explicit_side(text, s, e)
        if label in LABELS:
            original_labeled.add(
                (str(doc_id), int(s), int(e), label)
            )

    for s, e in cleaned_counsel_candidates(text):
        label = explicit_side(text, s, e)
        if label in LABELS:
            cleaned_labeled.add(
                (str(doc_id), int(s), int(e), label)
            )

def metric_row(name, pred):
    tp = len(pred & gold_labeled)
    fp = len(pred - gold_labeled)
    fn = len(gold_labeled - pred)
    p = tp / len(pred) if pred else 0.0
    r = tp / len(gold_labeled) if gold_labeled else 0.0
    f = 2*p*r/(p+r) if p+r else 0.0
    return {
        "candidate_source": name,
        "predicted": len(pred),
        "TP": tp, "FP": fp, "FN": fn,
        "precision": p, "recall": r, "F1": f
    }

combined = original_labeled | cleaned_labeled
results = pd.DataFrame([
    metric_row("Original rules", original_labeled),
    metric_row("Cleaned candidates", cleaned_labeled),
    metric_row("Union", combined),
])
display(results.round(3))

# Display errors from the union for manual review.
false_positives = sorted(combined - gold_labeled)
false_negatives = sorted(gold_labeled - combined)

def print_spans(spans, heading, limit=25):
    print(f"\n=== {heading}: {len(spans)} total ===")
    for doc_id, s, e, label in spans[:limit]:
        text = val_texts.get(doc_id)
        if text is None:
            text = val_texts.get(next(
                (k for k in val_texts if str(k) == doc_id), None
            ))
        if text is None:
            print(doc_id, s, e, label, "[text missing]")
            continue

        print(
            f"{label} | {text[max(0,s-100):s]!r}"
            f" ⟦{text[s:e]}⟧ "
            f"{text[e:min(len(text),e+100)]!r}"
        )

print_spans(false_positives, "V5 RULE FALSE POSITIVES")
print_spans(false_negatives, "V5 RULE MISSED GOLD")

print("\nThis cell measures rule candidates separately from the full model.")
print("Do not replace the existing V5 pipeline based on these metrics alone.")


In [ ]:

# CS781 — Conservative counsel-rule experiment
# Only evaluates alternatives; does not modify the champion or save files.

import re
import pandas as pd

COUNSEL_ROLE = re.compile(
    r"\b(?:"
    r"learned\s+(?:senior\s+)?counsel|"
    r"senior\s+advocate|"
    r"advocate\s+for|"
    r"counsel\s+for|"
    r"appearing\s+for|"
    r"on\s+behalf\s+of|"
    r"for\s+the\s+(?:appellant|appellants|petitioner|petitioners|"
    r"respondent|respondents|state|revenue|prosecution|complainant)"
    r")\b",
    re.I
)

NEGATIVE_ROLE = re.compile(
    r"\b(?:"
    r"appellant(?:\s+no\.?\s*\d+)?|"
    r"petitioner\s+in\s+person|"
    r"appeared\s+in\s+person|"
    r"witness|deponent|informant|"
    r"investigating\s+officer|"
    r"amicus\s+curiae"
    r")\b",
    re.I
)

def has_nearby_counsel_role(text, start, end, window=150):
    left = text[max(0, start-window):start]
    right = text[end:min(len(text), end+window)]

    # Strong legal-role cues immediately near the candidate.
    if COUNSEL_ROLE.search(left) or COUNSEL_ROLE.search(right):
        # Avoid treating a nearby explicit non-counsel role as sufficient.
        nearby = left[-100:] + " " + right[:100]
        if NEGATIVE_ROLE.search(nearby) and not re.search(
            r"\b(?:senior\s+advocate|learned\s+counsel|counsel\s+for|"
            r"advocate\s+for|appearing\s+for)\b",
            nearby, re.I
        ):
            return False
        return True
    return False

def build_rule_predictions(texts, require_cue=False, window=150):
    out = {}
    for doc_id, text in texts.items():
        spans = set()

        # Use the existing candidate sources.
        candidates = list(rule_spans(text))
        candidates += list(cleaned_counsel_candidates(text))

        for start, end in candidates:
            if start < 0 or end <= start or end > len(text):
                continue

            label = explicit_side(text, start, end)
            if label not in {"A.COUNSEL", "R.COUNSEL"}:
                continue

            if require_cue and not has_nearby_counsel_role(
                text, start, end, window=window
            ):
                continue

            spans.add((str(doc_id), int(start), int(end), label))

        out[str(doc_id)] = [
            (s, e, label)
            for d, s, e, label in sorted(spans)
        ]

    return out

# Gold counsel spans in validation.
gold_counsel = {
    (str(doc), int(e[0]), int(e[1]), str(e[2]))
    for doc, ents in val_gold.items()
    for e in ents
    if len(e) >= 3 and e[2] in {"A.COUNSEL", "R.COUNSEL"}
}

def flatten_predictions(pred):
    return {
        (str(doc), int(e[0]), int(e[1]), str(e[2]))
        for doc, ents in pred.items()
        for e in ents if len(e) >= 3
        and e[2] in {"A.COUNSEL", "R.COUNSEL"}
    }

def counsel_metrics(name, pred_set):
    tp = len(pred_set & gold_counsel)
    fp = len(pred_set - gold_counsel)
    fn = len(gold_counsel - pred_set)
    p = tp / len(pred_set) if pred_set else 0.0
    r = tp / len(gold_counsel) if gold_counsel else 0.0
    f = 2*p*r/(p+r) if p+r else 0.0
    return {
        "variant": name, "pred": len(pred_set),
        "TP": tp, "FP": fp, "FN": fn,
        "precision": p, "recall": r, "F1": f
    }

variants = []
for window in [50, 100, 150]:
    pred = build_rule_predictions(
        val_texts, require_cue=True, window=window
    )
    variants.append(
        counsel_metrics(f"role cue ±{window} chars", flatten_predictions(pred))
    )

# Reference: current full V5 counsel predictions.
current_counsel = {
    (str(doc), int(e[0]), int(e[1]), str(e[2]))
    for doc, ents in val_v5.items()
    for e in ents if len(e) >= 3
    and e[2] in {"A.COUNSEL", "R.COUNSEL"}
}
variants.insert(0, counsel_metrics("current V5", current_counsel))

print("=== COUNSEL FILTER COMPARISON ===")
display(pd.DataFrame(variants).set_index("variant").round(3))

print(
    "\nInterpretation: a stricter filter is useful only if it reduces false "
    "positives without collapsing recall. This is a validation experiment; "
    "keep the original V5 output unchanged."
)


In [ ]:

# CS781 — Compare counsel side-assignment policies on validation
# Read-only experiment.

import pandas as pd

assert "val_texts" in globals() and "val_gold" in globals()
assert "val_base" in globals()
assert "merge_policy" in globals()

def entity_report(predictions, gold):
    labels = sorted({
        str(e[2]) for ents in gold.values()
        for e in ents if len(e) >= 3
    })
    gs = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in gold.items()
        for e in ents if len(e) >= 3
    }
    ps = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in predictions.items()
        for e in ents if len(e) >= 3
    }

    rows = []
    for lab in labels:
        g = {x for x in gs if x[3] == lab}
        p = {x for x in ps if x[3] == lab}
        tp = len(g & p)
        pr = tp / len(p) if p else 0.0
        rc = tp / len(g) if g else 0.0
        f1 = 2*pr*rc/(pr+rc) if pr+rc else 0.0
        rows.append({
            "label": lab, "F1": f1,
            "gold": len(g), "predicted": len(p),
            "TP": tp, "FP": len(p-g), "FN": len(g-p)
        })
    return float(pd.DataFrame(rows)["F1"].mean()), pd.DataFrame(rows).set_index("label")

policy_results = []
policy_predictions = {}

for policy in ["R", "A", "drop"]:
    pred = merge_policy(val_texts, val_base, policy)
    macro, report = entity_report(pred, val_gold)
    policy_predictions[policy] = pred

    counsel_gold = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in val_gold.items()
        for e in ents if len(e) >= 3
        and e[2] in {"A.COUNSEL", "R.COUNSEL"}
    }
    counsel_pred = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in pred.items()
        for e in ents if len(e) >= 3
        and e[2] in {"A.COUNSEL", "R.COUNSEL"}
    }
    tp = len(counsel_gold & counsel_pred)
    p = tp / len(counsel_pred) if counsel_pred else 0
    r = tp / len(counsel_gold) if counsel_gold else 0
    f = 2*p*r/(p+r) if p+r else 0

    policy_results.append({
        "policy": policy,
        "macro_F1": macro,
        "counsel_precision": p,
        "counsel_recall": r,
        "counsel_F1": f,
        "counsel_predictions": len(counsel_pred),
    })

policy_table = pd.DataFrame(policy_results).set_index("policy")
display(policy_table.round(4))

print("Best policy by full macro-F1:", policy_table["macro_F1"].idxmax())
print("This is an evaluation only; no model or files were changed.")


In [ ]:

# CS781 — Per-label comparison of counsel policies
# Read-only analysis.

import pandas as pd

def exact_counts(predictions, gold):
    labels = sorted({
        str(e[2])
        for ents in gold.values()
        for e in ents if len(e) >= 3
    })
    gs = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in gold.items()
        for e in ents if len(e) >= 3
    }

    rows = []
    for policy, pred in predictions.items():
        ps = {
            (str(d), int(e[0]), int(e[1]), str(e[2]))
            for d, ents in pred.items()
            for e in ents if len(e) >= 3
        }
        for label in labels:
            g = {x for x in gs if x[3] == label}
            p = {x for x in ps if x[3] == label}
            tp = len(g & p)
            precision = tp / len(p) if p else 0.0
            recall = tp / len(g) if g else 0.0
            f1 = (
                2 * precision * recall / (precision + recall)
                if precision + recall else 0.0
            )
            rows.append({
                "policy": policy,
                "label": label,
                "F1": f1,
                "predicted": len(p),
                "TP": tp,
                "FP": len(p-g),
                "FN": len(g-p),
            })
    return pd.DataFrame(rows)

per_label = exact_counts(policy_predictions, val_gold)

wide = per_label.pivot(index="label", columns="policy", values="F1")
wide["R_minus_drop"] = wide["R"] - wide["drop"]
print("=== PER-LABEL F1 ===")
display(wide.round(4))

counts = per_label[per_label["label"].isin(["A.COUNSEL", "R.COUNSEL"])]
print("\n=== COUNSEL COUNTS ===")
display(counts.sort_values(["label", "policy"]).round(4))

print("\nNo model weights or prediction files were modified.")


In [ ]:

# CS781 — Hybrid counsel policy: A.COUNSEL from policy A,
# R.COUNSEL from policy R. All other predictions stay unchanged.
# Evaluation only: no model/checkpoint/file modifications.

import pandas as pd

assert all(p in policy_predictions for p in ["A", "R", "drop"])

def make_hybrid(base, pred_A, pred_R):
    result = {}
    for doc_id, ents in base.items():
        kept = [
            tuple(e) for e in ents
            if len(e) >= 3 and e[2] not in {"A.COUNSEL", "R.COUNSEL"}
        ]
        a_ents = [
            tuple(e) for e in pred_A.get(doc_id, [])
            if len(e) >= 3 and e[2] == "A.COUNSEL"
        ]
        r_ents = [
            tuple(e) for e in pred_R.get(doc_id, [])
            if len(e) >= 3 and e[2] == "R.COUNSEL"
        ]
        result[doc_id] = kept + a_ents + r_ents
    return result

hybrid_pred = make_hybrid(
    val_base,
    policy_predictions["A"],
    policy_predictions["R"]
)

def exact_macro_and_report(predictions, gold):
    labels = sorted({
        str(e[2]) for ents in gold.values()
        for e in ents if len(e) >= 3
    })
    gs = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in gold.items()
        for e in ents if len(e) >= 3
    }
    ps = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in predictions.items()
        for e in ents if len(e) >= 3
    }

    rows = []
    for lab in labels:
        g = {x for x in gs if x[3] == lab}
        p = {x for x in ps if x[3] == lab}
        tp = len(g & p)
        pr = tp / len(p) if p else 0.0
        rc = tp / len(g) if g else 0.0
        f1 = 2*pr*rc/(pr+rc) if pr+rc else 0.0
        rows.append({
            "label": lab, "F1": f1,
            "predicted": len(p), "TP": tp,
            "FP": len(p-g), "FN": len(g-p)
        })
    report = pd.DataFrame(rows).set_index("label")
    return float(report["F1"].mean()), report

hybrid_macro, hybrid_report = exact_macro_and_report(hybrid_pred, val_gold)

summary = pd.DataFrame({
    "policy_R": exact_macro_and_report(policy_predictions["R"], val_gold)[1]["F1"],
    "policy_A": exact_macro_and_report(policy_predictions["A"], val_gold)[1]["F1"],
    "policy_drop": exact_macro_and_report(policy_predictions["drop"], val_gold)[1]["F1"],
    "hybrid_A_for_A_R_for_R": hybrid_report["F1"],
})
print("=== FULL MACRO-F1 ===")
print(f"R:      {exact_macro_and_report(policy_predictions['R'], val_gold)[0]:.4f}")
print(f"A:      {exact_macro_and_report(policy_predictions['A'], val_gold)[0]:.4f}")
print(f"drop:   {exact_macro_and_report(policy_predictions['drop'], val_gold)[0]:.4f}")
print(f"hybrid: {hybrid_macro:.4f}")

print("\n=== PER-LABEL F1 ===")
display(summary.round(4))

print("\n=== HYBRID COUNSEL COUNTS ===")
display(hybrid_report.loc[["A.COUNSEL", "R.COUNSEL"]].round(4))

print("\nNo model weights or prediction files were modified.")


In [ ]:

# CS781 — Counsel confidence sweep on held-out validation
# Read-only; keeps the champion and current predictions unchanged.

import pandas as pd

assert "raw_fold2" in globals(), "raw_fold2 not found."
assert "val_gold" in globals(), "val_gold not found."

sample_doc = next(iter(raw_fold2))
print("Sample raw prediction:", raw_fold2[sample_doc][:5])

# raw_fold2 entries are expected to be (start, end, label, confidence).
# Stop rather than silently misinterpret a different format.
sample = next(
    (e for ents in raw_fold2.values() for e in ents if len(e) >= 4),
    None
)
if sample is None or not isinstance(sample[3], (int, float)):
    raise RuntimeError(
        "raw_fold2 does not appear to contain numeric confidence scores "
        "in position 4. No threshold experiment was run."
    )

gold_counsel = {
    (str(d), int(e[0]), int(e[1]), str(e[2]))
    for d, ents in val_gold.items()
    for e in ents if len(e) >= 3
    and e[2] in {"A.COUNSEL", "R.COUNSEL"}
}

# Keep only rule-generated counsel spans whose confidence can be matched
# to an exact raw model span. This tests model confidence, not rule quality.
def counsel_confidence_sweep(raw, gold):
    raw_lookup = {}
    for doc, ents in raw.items():
        for e in ents:
            if len(e) >= 4:
                s, end, label, conf = e[:4]
                if label in {"A.COUNSEL", "R.COUNSEL"}:
                    raw_lookup[(str(doc), int(s), int(end), str(label))] = float(conf)

    rows = []
    for threshold in [0.05, 0.10, 0.15, 0.20, 0.30, 0.40]:
        pred = {
            span for span, conf in raw_lookup.items()
            if conf >= threshold
        }
        tp = len(pred & gold)
        fp = len(pred - gold)
        fn = len(gold - pred)
        p = tp / len(pred) if pred else 0.0
        r = tp / len(gold) if gold else 0.0
        f1 = 2*p*r/(p+r) if p+r else 0.0
        rows.append({
            "confidence_threshold": threshold,
            "predicted": len(pred), "TP": tp, "FP": fp, "FN": fn,
            "precision": p, "recall": r, "F1": f1
        })
    return pd.DataFrame(rows)

display(counsel_confidence_sweep(raw_fold2, gold_counsel).round(3))
print(
    "\nNote: this sweep evaluates only model-generated counsel spans with "
    "matching exact boundaries. It does not replace the V5 rule evaluation."
)


In [ ]:

# CS781 — Freeze experimental policy R and run consistency checks.
# No file writes; no model weight changes.

import pandas as pd

assert "policy_predictions" in globals()
assert "R" in policy_predictions
assert "val_texts" in globals() and "val_gold" in globals()

# Preserve a separate in-memory copy; do not overwrite val_v5 or champion outputs.
experimental_policy = "R"
experimental_val_predictions = {
    doc_id: [tuple(e) for e in ents]
    for doc_id, ents in policy_predictions["R"].items()
}

assert set(experimental_val_predictions) == set(val_gold)

# Validate spans and labels.
allowed_labels = {
    str(e[2])
    for ents in val_gold.values()
    for e in ents if len(e) >= 3
}
invalid = []
for doc_id, ents in experimental_val_predictions.items():
    text = val_texts[doc_id]
    for ent in ents:
        if len(ent) < 3:
            invalid.append((doc_id, ent, "short tuple"))
            continue
        s, end, label = int(ent[0]), int(ent[1]), str(ent[2])
        if s < 0 or end <= s or end > len(text):
            invalid.append((doc_id, ent, "invalid boundaries"))
        if label not in allowed_labels:
            invalid.append((doc_id, ent, "unknown label"))

if invalid:
    print("Invalid spans found:", len(invalid))
    print(invalid[:20])
else:
    print("All predicted spans have valid boundaries and known labels.")

counts = {}
for doc_id, ents in experimental_val_predictions.items():
    for ent in ents:
        label = str(ent[2])
        counts[label] = counts.get(label, 0) + 1

print("\nExperimental policy:", experimental_policy)
print("Documents:", len(experimental_val_predictions))
print("Total predicted spans:", sum(counts.values()))
print("\nEntity counts:")
display(
    pd.DataFrame(
        sorted(counts.items()),
        columns=["label", "predictions"]
    ).set_index("label")
)

print("\nExperimental predictions remain in memory only.")
print("Champion model and saved prediction files are untouched.")


In [ ]:

# CS781 — Final validation report: Champion vs experimental policy R
# Read-only. Uses existing predictions and exact entity-span matching.

import pandas as pd

def final_exact_span_report(predictions, gold):
    labels = sorted({
        str(e[2])
        for ents in gold.values()
        for e in ents if len(e) >= 3
    })

    gold_set = {
        (str(doc), int(e[0]), int(e[1]), str(e[2]))
        for doc, ents in gold.items()
        for e in ents if len(e) >= 3
    }
    pred_set = {
        (str(doc), int(e[0]), int(e[1]), str(e[2]))
        for doc, ents in predictions.items()
        for e in ents if len(e) >= 3
    }

    rows = []
    for label in labels:
        g = {x for x in gold_set if x[3] == label}
        p = {x for x in pred_set if x[3] == label}
        tp = len(g & p)
        fp = len(p - g)
        fn = len(g - p)
        precision = tp / len(p) if p else 0.0
        recall = tp / len(g) if g else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )
        rows.append({
            "label": label,
            "gold": len(g),
            "predicted": len(p),
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "precision": precision,
            "recall": recall,
            "F1": f1,
        })

    report = pd.DataFrame(rows).set_index("label")
    return float(report["F1"].mean()), report

# Reuse the baseline that was generated on val_texts.
assert "val_base" in globals()
assert "experimental_val_predictions" in globals()

champion_macro, champion_report = final_exact_span_report(val_base, val_gold)
policy_r_macro, policy_r_report = final_exact_span_report(
    experimental_val_predictions, val_gold
)

print("=== FINAL HELD-OUT VALIDATION COMPARISON ===")
print(f"Champion:  {champion_macro:.4f}")
print(f"Policy R:  {policy_r_macro:.4f}")
print(f"Delta:     {policy_r_macro - champion_macro:+.4f}")

print("\n=== PER-ENTITY F1 ===")
display(pd.DataFrame({
    "champion_F1": champion_report["F1"],
    "policy_R_F1": policy_r_report["F1"],
    "champion_pred": champion_report["predicted"],
    "policy_R_pred": policy_r_report["predicted"],
}).round(4))

print("\n=== MACRO-F1 CHECK ===")
print("Macro-F1 is the unweighted mean across entity labels.")
print("These results are validation metrics, not final test-set metrics.")
print("No model weights or prediction files were modified.")


In [ ]:
# ============================================================
# FIXED TEST INFERENCE — POLICY R
# Both inputs to merge_policy are dictionaries.
# ============================================================

import json
from collections import Counter

# 1. Check test data structure
print("test_texts type:", type(test_texts).__name__)
print("test_texts length:", len(test_texts))

# merge_policy expects texts.items(), so construct a dictionary.
if isinstance(test_texts, dict):
    test_texts_by_id = test_texts
else:
    # If test_texts is a list, use the IDs returned by predict_texts
    # to align the texts with the predictions.
    test_texts_by_id = None

# 2. Run champion inference
test_raw_predictions = predict_texts(champion_model, test_texts)
test_base_predictions = apply_thr(test_raw_predictions, champ_thr)

if not isinstance(test_base_predictions, dict):
    raise TypeError(
        f"apply_thr returned {type(test_base_predictions).__name__}, "
        "but a dictionary is required."
    )

test_ids = list(test_base_predictions.keys())

# Align document text IDs to prediction IDs.
if test_texts_by_id is None:
    if len(test_ids) != len(test_texts):
        raise ValueError(
            f"Cannot align documents: {len(test_ids)} prediction IDs "
            f"but {len(test_texts)} texts."
        )
    test_texts_by_id = dict(zip(test_ids, test_texts))

if set(test_texts_by_id.keys()) != set(test_base_predictions.keys()):
    raise ValueError(
        "Text and prediction document IDs do not match. "
        "Do not generate output until the correct test IDs are aligned."
    )

print("Aligned test documents:", len(test_ids))

# 3. Apply policy R using dictionaries, as required by merge_policy()
test_policy_predictions = merge_policy(
    test_texts_by_id,
    test_base_predictions,
    "R"
)

if not isinstance(test_policy_predictions, dict):
    raise TypeError(
        "Expected merge_policy to return a dictionary, got "
        f"{type(test_policy_predictions).__name__}."
    )

# 4. Validate predictions
allowed_labels = {
    "A.COUNSEL", "R.COUNSEL", "APP", "RESP", "JUDGE",
    "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
}

label_counts = Counter()
total_spans = 0

for doc_id, spans in test_policy_predictions.items():
    text = test_texts_by_id[doc_id]

    for span in spans:
        start, end, label = span[:3]

        if not isinstance(start, int) or not isinstance(end, int):
            raise TypeError(f"Invalid offsets: document={doc_id}, span={span}")

        if not (0 <= start < end <= len(text)):
            raise ValueError(
                f"Invalid span: document={doc_id}, span={span}, "
                f"text length={len(text)}"
            )

        if label not in allowed_labels:
            raise ValueError(f"Unexpected label: {label}")

        label_counts[label] += 1
        total_spans += 1

print("Validated spans:", total_spans)
print("Label counts:", dict(sorted(label_counts.items())))

# 5. Write a separate intermediate JSON
# Format: {document_id: [[start, end, label], ...]}
prediction_dict = {
    str(doc_id): [
        [int(span[0]), int(span[1]), str(span[2])]
        for span in sorted(spans, key=lambda s: (s[0], s[1], s[2]))
    ]
    for doc_id, spans in test_policy_predictions.items()
}

output_path = "/kaggle/working/prediction.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(prediction_dict, f, ensure_ascii=False, indent=2)

# 6. Verify saved file
with open(output_path, "r", encoding="utf-8") as f:
    check = json.load(f)

assert len(check) == len(test_texts_by_id)
assert sum(len(v) for v in check.values()) == total_spans

print("\nSUCCESS")
print("Saved:", output_path)
print("Documents:", len(check))
print("Total spans:", total_spans)
print("Preview:", next(iter(check.items())))

In [ ]:

# ============================================================
# CS781: COUNSEL + JUDGE ABLATION
# Validation only. Does not modify model weights or test output.
# ============================================================

import pandas as pd
from collections import Counter

assert "val_texts" in globals() and "val_gold" in globals()
assert "c_filt" in globals()
assert "judge_patterns2" in globals()
assert "NAMES" in globals()
assert "merge_policy" in globals()
assert "merge_all" in globals()

# Use the exact-span scorer already defined in the notebook.
def evaluate_candidate(name, predictions):
    macro, report = final_exact_span_report(predictions, val_gold)
    print(f"\n{name}: macro-F1 = {macro:.4f}")
    return macro, report

# Baseline: original champion predictions
baseline_macro, baseline_report = evaluate_candidate(
    "1. Champion",
    c_filt
)

# Counsel-only policies, without judge rules
counsel_results = {}
for policy in ["R", "A", "drop"]:
    preds = merge_policy(val_texts, c_filt, policy)
    macro, report = evaluate_candidate(
        f"2. Champion + counsel policy {policy}",
        preds
    )
    counsel_results[policy] = (macro, report, preds)

# Counsel + judge rules selected from fold 1
judge_preds = merge_all(val_texts, c_filt, keep)
judge_macro, judge_report = evaluate_candidate(
    "3. Champion + counsel + selected judge rules",
    judge_preds
)

# Per-label comparison for the two strongest candidates
best_counsel_policy = max(counsel_results, key=lambda p: counsel_results[p][0])
best_counsel_report = counsel_results[best_counsel_policy][1]

comparison = pd.DataFrame({
    "Champion_F1": baseline_report["F1"],
    f"Counsel_{best_counsel_policy}_F1": best_counsel_report["F1"],
    "Counsel_Judge_F1": judge_report["F1"],
    "Champion_predicted": baseline_report["predicted"],
    f"Counsel_{best_counsel_policy}_predicted": best_counsel_report["predicted"],
    "Counsel_Judge_predicted": judge_report["predicted"],
})

print("\n=== PER-ENTITY COMPARISON ===")
display(comparison.round(4))

print("\n=== JUDGE PATTERN COUNTS ON FOLD 2 ===")
for pattern_name in NAMES:
    pred_set = {
        (doc_id, s, e)
        for doc_id, text in val_texts.items()
        for s, e in judge_patterns2(text)[pattern_name]
    }
    gold_set = {
        (doc_id, int(ent[0]), int(ent[1]))
        for doc_id, ents in val_gold.items()
        for ent in ents
        if ent[2] == "JUDGE"
    }
    tp = len(pred_set & gold_set)
    precision = tp / max(len(pred_set), 1)
    recall = tp / max(len(gold_set), 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    print(
        f"{pattern_name:24s} predictions={len(pred_set):4d} "
        f"TP={tp:4d} P={precision:.3f} R={recall:.3f} F1={f1:.3f}"
    )

print("\n=== DECISION ===")
print("Best counsel-only policy:", best_counsel_policy)
print(f"Champion macro-F1: {baseline_macro:.4f}")
print(f"Best counsel-only macro-F1: {counsel_results[best_counsel_policy][0]:.4f}")
print(f"Counsel + judge macro-F1: {judge_macro:.4f}")
print(
    "Judge-rule change over counsel-only:",
    f"{judge_macro - counsel_results[best_counsel_policy][0]:+.4f}"
)
print("\nDo not change the final test file until these results are reviewed.")


In [ ]:
# ================================================================
# CS781 — SELF-CONTAINED JUDGE + COUNSEL EXPERIMENT
# Recreates judge helpers; selects rules on fold 1; evaluates fold 2.
# Does not train or modify model weights.
# ================================================================

import re
import pandas as pd
from collections import Counter

# ------------------------------------------------
# 0. Check only foundational data/model dependencies
# ------------------------------------------------
_required = [
    "train_texts", "train_gold",
    "val_texts", "val_gold",
    "champion_model", "champ_thr",
    "diag_doc_probs", "bio_viterbi_decode", "decode_spans",
    "rule_spans", "assign_side", "score"
]
_missing = [x for x in _required if x not in globals()]

if _missing:
    raise RuntimeError(
        "Missing foundational variables/functions: "
        + ", ".join(_missing)
        + "\nThis experiment reconstructs the judge and counsel "
          "helpers, but needs the loaded datasets and champion inference "
          "functions. Do not retrain; restore/load those foundational "
          "objects first."
    )

# ------------------------------------------------
# 1. Rebuild judge-pattern helpers in this cell
# ------------------------------------------------
CAP = r"[A-ZÀ-Ö][A-Za-zÀ-ÖØ-öø-ÿ'’\-]*"
INIT = r"[A-Z]\.(?:[ \t]*[A-Z]\.)*"
WORD = rf"(?:{INIT}|{CAP})"
NAME2 = rf"{WORD}(?:[ \t]+{WORD}){{0,3}}"

TOK = r"(?:[A-Z]\.(?:\s?[A-Z]\.)*|[A-Z][a-z']+|[A-Z]{2,})"
TOK_RE = re.compile(TOK)

STOP = {
    "JUDGMENT", "JUDGEMENT", "ORDER", "BENCH", "CORAM", "BEFORE",
    "PRESENT", "HON'BLE", "MR", "MRS", "MS", "JUSTICE", "THE",
    "AND", "OF", "DATE", "NEW", "DELHI", "BY", "DELIVERED"
}

def _judge_trim(text, s, e):
    while s < e and (text[s].isspace() or text[s] in ".,;:"):
        s += 1
    while e > s and (text[e - 1].isspace() or text[e - 1] in ",;:"):
        e -= 1
    return (s, e) if e > s else None

def _judge_back_name(text, end, max_toks=4):
    ws = max(0, end - 80)
    toks = [
        (m.start() + ws, m.end() + ws)
        for m in TOK_RE.finditer(text[ws:end])
    ]
    run = []

    for a, b in reversed(toks):
        nxt = run[0][0] if run else end
        if text[b:nxt].strip(" \t") != "":
            break
        if text[a:b].upper().strip(".") in STOP:
            break
        run.insert(0, (a, b))
        if len(run) == max_toks:
            break

    return (run[0][0], run[-1][1]) if run else None

AJ_RE = re.compile(rf"(?i:\bjustice)[ \t]+({NAME2})")
J_B_RE = re.compile(r",[ \t]*J{1,2}\b\.?")
J_B2_RE = re.compile(
    r"[ \t]J{1,2}\.?(?=[ \t]*(?:in\b|speaking|writing|observed|"
    r"held\b|was\b|said\b|while\b|who\b|delivered|dissent|"
    r"agreed|also\b|noted|stated))"
)
J_C_RE = re.compile(r"\bJJ?\.?[ \t]*\(([^()\n]{3,45})\)")
J_D_RE = re.compile(
    r"(?:BENCH\s*:|\(J\)|\(J\.\))\s*"
    r"([A-Z][A-Za-z']+,[ \t]*[A-Z](?:[ \t]*\.?[ \t]*[A-Z])*\.?)"
    r"[ \t]*\(J\)"
)
J_E_RE = re.compile(
    r"\bJJ?\.?[ \t]*\[[ \t]*([A-Za-zÀ-ÿ\.'’ \t\-]{3,45}?)[ \t]*\]"
)
J_F_RE = re.compile(
    r"BENCH[ \t]*:[ \t]*([^\n]{3,120}?)"
    r"(?=[ \t]*(?:JUDG(?:E)?MENT|CITATION|ACT\b|\n|$))"
)

JUDGE_PATTERN_NAMES = [
    "A Justice NAME",
    "B NAME, J.",
    "B2 NAME J",
    "C J.(NAME)",
    "D BENCH X,Y",
    "E J [NAME]",
    "F BENCH list"
]

def judge_patterns_fixed(text):
    out = {n: set() for n in JUDGE_PATTERN_NAMES}

    def add(name, s, e):
        sp = _judge_trim(text, s, e)
        if sp:
            out[name].add(sp)

    for m in AJ_RE.finditer(text):
        add(JUDGE_PATTERN_NAMES[0], m.start(1), m.end(1))

    for rx, name in [
        (J_B_RE, JUDGE_PATTERN_NAMES[1]),
        (J_B2_RE, JUDGE_PATTERN_NAMES[2])
    ]:
        for m in rx.finditer(text):
            bn = _judge_back_name(text, m.start())
            if bn:
                add(name, *bn)

    for rx, name in [
        (J_C_RE, JUDGE_PATTERN_NAMES[3]),
        (J_E_RE, JUDGE_PATTERN_NAMES[5])
    ]:
        for m in rx.finditer(text):
            inner = m.group(1).strip()
            if (
                inner
                and inner[0].isupper()
                and inner.upper() not in {"J", "JJ"}
                and re.fullmatch(r"[A-Za-zÀ-ÿ\.'’, \-]+", inner)
            ):
                add(name, m.start(1), m.end(1))

    for m in J_D_RE.finditer(text):
        add(JUDGE_PATTERN_NAMES[4], m.start(1), m.end(1))

    for m in J_F_RE.finditer(text):
        segment = m.group(1)
        base = m.start(1)
        pos = 0

        for part in re.split(r"[ \t]*(?:&|\bAND\b)[ \t]*", segment):
            k = segment.find(part, pos)
            if k < 0:
                continue
            pos = k + len(part)

            part2 = re.sub(
                r"[ \t]*\((?:C?J|J\.?)\)\s*$", "", part
            )
            sp = _judge_trim(text, base + k, base + k + len(part2))
            if sp and re.fullmatch(NAME2, text[sp[0]:sp[1]]):
                out[JUDGE_PATTERN_NAMES[6]].add(sp)

    return out

def _resolve_judge_spans(spans):
    out = []
    for s, e in sorted(set(spans), key=lambda x: (x[0], -(x[1] - x[0]))):
        if out and s < out[-1][1]:
            continue
        out.append((s, e))
    return out

# ------------------------------------------------
# 2. Select judge patterns on fold 1 ONLY
# ------------------------------------------------
train_judge_gold = {
    (str(d), int(e[0]), int(e[1]))
    for d, ents in train_gold.items()
    for e in ents
    if len(e) >= 3 and str(e[2]) == "JUDGE"
}

pattern_predictions = {n: set() for n in JUDGE_PATTERN_NAMES}

for doc_id, text in train_texts.items():
    for name, spans in judge_patterns_fixed(text).items():
        pattern_predictions[name].update(
            (str(doc_id), s, e) for s, e in spans
        )

print("=== JUDGE PATTERN SELECTION — FOLD 1 ===")
print(f"Gold JUDGE spans: {len(train_judge_gold)}")
print(f"{'Pattern':22s} {'Pred':>6s} {'TP':>6s} {'P':>7s} {'F1':>7s}")

keep_judge_patterns = []

for name in JUDGE_PATTERN_NAMES:
    pred = pattern_predictions[name]
    tp = len(pred & train_judge_gold)
    precision = tp / len(pred) if pred else 0.0
    recall = tp / len(train_judge_gold) if train_judge_gold else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    print(f"{name:22s} {len(pred):6d} {tp:6d} {precision:7.3f} {f1:7.3f}")

    # Predeclared conservative rule: precision >= .40 and at least 5 TPs.
    if precision >= 0.40 and tp >= 5:
        keep_judge_patterns.append(name)

print("\nSelected judge patterns:", keep_judge_patterns)

def judge_spans_fixed(text):
    candidates = set()
    for name, spans in judge_patterns_fixed(text).items():
        if name in keep_judge_patterns:
            candidates.update(spans)
    return _resolve_judge_spans(candidates)

# ------------------------------------------------
# 3. Rebuild counsel policy helpers locally
# ------------------------------------------------
PARTY_A = (
    r"(?:petitioners?|appellants?|applicants?|plaintiffs?|"
    r"complainants?|prosecution|revisionists?)"
)
PARTY_R = (
    r"(?:respondents?|defendants?|accused|non-\s*applicants?|"
    r"opposite\s+part(?:y|ies))"
)
FOR = r"(?:for|on\s+behalf\s+of|appearing\s+for|representing)"
MODS = r"(?:(?:learned|first|second|third|\d+\w*|all|other)\s+)*"

CUE_A = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_A}", re.I)
CUE_R = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_R}", re.I)
HDR_A = re.compile(rf"{PARTY_A}s?[\s\.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HDR_R = re.compile(rf"{PARTY_R}s?[\s\.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HON_TAIL = re.compile(r"(?:Mr|Ms|Mrs|Shri|Sri|Smt)\.?[ \t]*$")

A_LABEL, R_LABEL = "A.COUNSEL", "R.COUNSEL"
COUNSEL_ORDER = ["hdr", "right", "left", "inherit"]

def _cues_fixed(text, lo, hi):
    out = [(m.start(), m.end(), A_LABEL) for m in CUE_A.finditer(text, lo, hi)]
    out += [(m.start(), m.end(), R_LABEL) for m in CUE_R.finditer(text, lo, hi)]
    return out

def _assign_side_fixed(text, s, e, prev, W_L=300, W_R=200):
    for step in COUNSEL_ORDER:
        if step == "right":
            cues = _cues_fixed(text, e, min(len(text), e + W_R))
            if cues:
                return min(cues)[2], "right"
        elif step == "left":
            cues = _cues_fixed(text, max(0, s - W_L), s)
            cues = [x for x in cues if x[1] <= s]
            if cues:
                return max(cues, key=lambda x: x[1])[2], "left"
        elif step == "hdr":
            tail = HON_TAIL.sub("", text[max(0, s - 80):s])
            if HDR_A.search(tail):
                return A_LABEL, "hdr"
            if HDR_R.search(tail):
                return R_LABEL, "hdr"
        elif step == "inherit":
            if prev and s - prev[0] <= 200 and prev[2] != "default":
                return prev[1], "inherit"
    return R_LABEL, "default"

def counsel_spans_fixed(text, policy="R"):
    out, prev = [], None
    for s, e in sorted(rule_spans(text)):
        label, source = _assign_side_fixed(text, s, e, prev)
        prev = (e, label, source)

        if source == "default":
            if policy == "drop":
                continue
            if policy == "A":
                label = A_LABEL

        out.append((int(s), int(e), label))
    return out

def merge_counsel_fixed(texts, champion_filtered, policy="R"):
    out = {}
    for doc_id, text in texts.items():
        rules = counsel_spans_fixed(text, policy)
        kept = [
            ent for ent in champion_filtered.get(doc_id, [])
            if ent[2] not in (A_LABEL, R_LABEL)
        ]
        kept = [
            ent for ent in kept
            if not any(
                ent[0] < re_ and ent[1] > rs
                for rs, re_, _ in rules
            )
        ]
        out[doc_id] = sorted(set(kept + rules))
    return out

# ------------------------------------------------
# 4. Rebuild inference and thresholding helpers
# ------------------------------------------------
import torch
from bisect import bisect_left, bisect_right

@torch.inference_mode()
def predict_fixed(model, texts):
    result = {}
    model.eval()

    for doc_id, text in texts.items():
        offsets, probs = diag_doc_probs(model, text)

        if len(offsets) == 0:
            result[doc_id] = []
            continue

        pred_ids = torch.as_tensor(
            bio_viterbi_decode(torch.log(probs.clamp_min(1e-12))),
            dtype=torch.long
        )
        starts = [a for a, b in offsets]
        ends = [b for a, b in offsets]
        spans = []

        for s, e, label in decode_spans(text, offsets, pred_ids.tolist()):
            ids = [
                i for i in range(bisect_right(ends, s), bisect_left(starts, e))
                if ends[i] > s
            ]
            confidence = (
                float(probs[ids, pred_ids[ids]].mean()) if ids else 0.0
            )
            spans.append((int(s), int(e), label, confidence))

        result[doc_id] = spans

    return result

def threshold_fixed(raw, thresholds):
    return {
        doc_id: [
            (s, e, label)
            for s, e, label, confidence in spans
            if confidence >= thresholds.get(label, 0.0)
        ]
        for doc_id, spans in raw.items()
    }

# ------------------------------------------------
# 5. Exact-span evaluation helpers
# ------------------------------------------------
def exact_report_fixed(predictions, gold):
    labels = sorted({
        str(e[2]) for ents in gold.values()
        for e in ents if len(e) >= 3
    })

    gold_set = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in gold.items()
        for e in ents if len(e) >= 3
    }
    pred_set = {
        (str(d), int(e[0]), int(e[1]), str(e[2]))
        for d, ents in predictions.items()
        for e in ents if len(e) >= 3
    }

    rows = []
    for label in labels:
        g = {x for x in gold_set if x[3] == label}
        p = {x for x in pred_set if x[3] == label}
        tp = len(g & p)
        fp = len(p - g)
        fn = len(g - p)
        precision = tp / len(p) if p else 0.0
        recall = tp / len(g) if g else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )
        rows.append({
            "label": label, "gold": len(g), "predicted": len(p),
            "TP": tp, "FP": fp, "FN": fn,
            "precision": precision, "recall": recall, "F1": f1
        })

    report = pd.DataFrame(rows).set_index("label")
    return float(report["F1"].mean()), report

# ------------------------------------------------
# 6. Run validation comparisons
# ------------------------------------------------
# Recompute champion predictions to avoid depending on c_filt or keep.
print("\nRunning champion inference on validation...")
val_raw_fixed = predict_fixed(champion_model, val_texts)
val_champion_fixed = threshold_fixed(val_raw_fixed, champ_thr)

# Compare policies, but do not choose the policy using validation results.
candidates = {}
for policy in ("R", "A", "drop"):
    candidates[policy] = merge_counsel_fixed(
        val_texts, val_champion_fixed, policy
    )

# Use R as the preselected policy from prior experiments.
val_counsel_R_fixed = candidates["R"]

# Add judges: remove champion JUDGE spans and replace them with rules.
val_combined_fixed = {}
for doc_id, text in val_texts.items():
    counsel_rules = counsel_spans_fixed(text, "R")
    judge_rules = [
        (s, e, "JUDGE") for s, e in judge_spans_fixed(text)
    ]
    all_rules = counsel_rules + judge_rules

    kept = [
        ent for ent in val_champion_fixed.get(doc_id, [])
        if ent[2] not in (A_LABEL, R_LABEL, "JUDGE")
    ]
    kept = [
        ent for ent in kept
        if not any(
            ent[0] < re_ and ent[1] > rs
            for rs, re_, _ in all_rules
        )
    ]
    val_combined_fixed[doc_id] = sorted(set(kept + all_rules))

# Score each candidate.
summary = []
reports = {}

for name, pred in [
    ("Champion", val_champion_fixed),
    ("Counsel R", val_counsel_R_fixed),
    ("Counsel A", candidates["A"]),
    ("Counsel drop", candidates["drop"]),
    ("Counsel R + Judge rules", val_combined_fixed),
]:
    macro, report = exact_report_fixed(pred, val_gold)
    reports[name] = report
    summary.append({
        "candidate": name,
        "macro_F1": macro,
        "total_spans": sum(len(v) for v in pred.values())
    })

summary_df = pd.DataFrame(summary).sort_values("macro_F1", ascending=False)
print("\n=== VALIDATION RESULTS ===")
display(summary_df.round(4))

print("\n=== PER-LABEL F1: CHAMPION vs COUNSEL R vs COUNSEL R + JUDGE ===")
per_label_df = pd.DataFrame({
    "champion_F1": reports["Champion"]["F1"],
    "counsel_R_F1": reports["Counsel R"]["F1"],
    "counsel_R_plus_judge_F1": reports["Counsel R + Judge rules"]["F1"],
    "champion_pred": reports["Champion"]["predicted"],
    "counsel_R_pred": reports["Counsel R"]["predicted"],
    "counsel_R_plus_judge_pred": reports["Counsel R + Judge rules"]["predicted"],
})
display(per_label_df.round(4))

print("\n=== JUDGE IMPACT ===")
judge_base = reports["Counsel R"].loc["JUDGE", "F1"] if "JUDGE" in reports["Counsel R"].index else 0.0
judge_new = reports["Counsel R + Judge rules"].loc["JUDGE", "F1"] if "JUDGE" in reports["Counsel R + Judge rules"].index else 0.0
print(f"JUDGE F1 before rules: {judge_base:.4f}")
print(f"JUDGE F1 after rules:  {judge_new:.4f}")
print(
    "Macro-F1 change from adding judge rules: "
    f"{summary_df.loc[summary_df.candidate == 'Counsel R + Judge rules', 'macro_F1'].iloc[0] - summary_df.loc[summary_df.candidate == 'Counsel R', 'macro_F1'].iloc[0]:+.4f}"
)

print("\nDone. No training, checkpoint edits, or output-file writes occurred.")

In [ ]:

# ============================================================
# CELL 1: AUDIT JUDGE ERRORS AND COUNSEL OVERLAPS
# Validation only. No checkpoint or file changes.
# ============================================================

import pandas as pd
from collections import defaultdict

# Check dependencies created by the standalone experiment cell.
_required = [
    "val_texts", "val_gold", "val_champion_fixed",
    "val_combined_fixed", "judge_spans_fixed",
    "counsel_spans_fixed", "exact_report_fixed"
]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError(
        "Run the standalone judge+counsel experiment cell first. "
        "Missing: " + ", ".join(_missing)
    )

def _gold_spans(doc_id, label):
    return [
        (int(e[0]), int(e[1]), str(e[2]))
        for e in val_gold.get(doc_id, [])
        if len(e) >= 3 and str(e[2]) == label
    ]

def _pred_spans(predictions, doc_id, label):
    return [
        (int(e[0]), int(e[1]), str(e[2]))
        for e in predictions.get(doc_id, [])
        if len(e) >= 3 and str(e[2]) == label
    ]

def _span_text(text, span):
    s, e, _ = span
    return text[max(0, s-70):min(len(text), e+70)].replace("\n", " ")

# 1. Judge true positives, false positives and false negatives.
judge_rows = []
for doc_id, text in val_texts.items():
    gold = set(_gold_spans(doc_id, "JUDGE"))
    pred = set(_pred_spans(val_combined_fixed, doc_id, "JUDGE"))

    for span in sorted(pred & gold):
        judge_rows.append({
            "type": "TP",
            "doc_id": doc_id,
            "span": text[span[0]:span[1]],
            "start": span[0], "end": span[1],
            "context": _span_text(text, span)
        })

    for span in sorted(pred - gold):
        judge_rows.append({
            "type": "FP",
            "doc_id": doc_id,
            "span": text[span[0]:span[1]],
            "start": span[0], "end": span[1],
            "context": _span_text(text, span)
        })

    for span in sorted(gold - pred):
        judge_rows.append({
            "type": "FN",
            "doc_id": doc_id,
            "span": text[span[0]:span[1]],
            "start": span[0], "end": span[1],
            "context": _span_text(text, span)
        })

judge_audit_df = pd.DataFrame(judge_rows)
print("=== JUDGE AUDIT COUNTS ===")
if not judge_audit_df.empty:
    display(judge_audit_df.groupby("type").size().rename("count").to_frame())
    print("\n=== JUDGE FALSE POSITIVES ===")
    display(judge_audit_df[judge_audit_df.type == "FP"].head(40))
    print("\n=== JUDGE FALSE NEGATIVES ===")
    display(judge_audit_df[judge_audit_df.type == "FN"].head(40))
else:
    print("No judge rows found.")

# 2. Counsel errors by side.
counsel_rows = []
for doc_id, text in val_texts.items():
    for label in ("A.COUNSEL", "R.COUNSEL"):
        gold = set(_gold_spans(doc_id, label))
        pred = set(_pred_spans(val_combined_fixed, doc_id, label))

        for span in sorted(pred - gold):
            counsel_rows.append({
                "type": "FP",
                "label": label,
                "doc_id": doc_id,
                "span": text[span[0]:span[1]],
                "start": span[0], "end": span[1],
                "context": _span_text(text, span)
            })

        for span in sorted(gold - pred):
            counsel_rows.append({
                "type": "FN",
                "label": label,
                "doc_id": doc_id,
                "span": text[span[0]:span[1]],
                "start": span[0], "end": span[1],
                "context": _span_text(text, span)
            })

counsel_audit_df = pd.DataFrame(counsel_rows)
print("\n=== COUNSEL AUDIT COUNTS ===")
if not counsel_audit_df.empty:
    display(
        counsel_audit_df.groupby(["label", "type"])
        .size().rename("count").to_frame()
    )
    print("\n=== COUNSEL FALSE POSITIVES ===")
    display(counsel_audit_df[counsel_audit_df.type == "FP"].head(50))
    print("\n=== COUNSEL FALSE NEGATIVES ===")
    display(counsel_audit_df[counsel_audit_df.type == "FN"].head(50))

# 3. Overlaps between counsel candidates and gold non-counsel entities.
overlap_rows = []
for doc_id, text in val_texts.items():
    counsel = counsel_spans_fixed(text, "R")
    other_gold = [
        (int(e[0]), int(e[1]), str(e[2]))
        for e in val_gold.get(doc_id, [])
        if len(e) >= 3 and str(e[2]) not in ("A.COUNSEL", "R.COUNSEL")
    ]

    for cs, ce, clabel in counsel:
        for gs, ge, glabel in other_gold:
            if cs < ge and ce > gs:
                overlap_rows.append({
                    "doc_id": doc_id,
                    "counsel_label": clabel,
                    "other_gold_label": glabel,
                    "counsel_text": text[cs:ce],
                    "gold_text": text[gs:ge],
                    "counsel_start": cs,
                    "counsel_end": ce,
                    "gold_start": gs,
                    "gold_end": ge,
                    "context": text[max(0, cs-80):min(len(text), ce+80)]
                              .replace("\n", " ")
                })

overlap_df = pd.DataFrame(overlap_rows)
print("\n=== COUNSEL OVERLAPS WITH OTHER GOLD LABELS ===")
if not overlap_df.empty:
    display(
        overlap_df.groupby(["counsel_label", "other_gold_label"])
        .size().rename("overlap_count").to_frame()
    )
    display(overlap_df.head(60))
else:
    print("No overlaps found.")

print("\nAudit complete. No files or model weights were modified.")


In [ ]:

# ============================================================
# CELL 2: COUNSEL FILTER ABLATION
# Uses validation only; never uses gold labels to filter predictions.
# Gold labels are used ONLY for scoring.
# ============================================================

import pandas as pd

_required = [
    "val_texts", "val_gold", "val_combined_fixed",
    "counsel_spans_fixed", "exact_report_fixed",
    "val_champion_fixed", "judge_spans_fixed"
]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError("Missing dependencies: " + ", ".join(_missing))

def _overlap(a, b):
    return a[0] < b[1] and a[1] > b[0]

def make_counsel_variant(mode):
    result = {}

    for doc_id, text in val_texts.items():
        counsel = counsel_spans_fixed(text, "R")
        judges = [
            (s, e, "JUDGE") for s, e in judge_spans_fixed(text)
        ]

        # Preserve all champion entities except the labels being replaced.
        kept = [
            tuple(e[:3])
            for e in val_champion_fixed.get(doc_id, [])
            if e[2] not in ("A.COUNSEL", "R.COUNSEL", "JUDGE")
        ]

        accepted = []
        for cs, ce, clabel in counsel:
            overlaps_judge = any(_overlap((cs, ce), j[:2]) for j in judges)

            if mode == "block_judge_overlap" and overlaps_judge:
                continue

            # "block_header_only" is a no-op fallback if no source metadata
            # exists; do not invent a cue/source label.
            accepted.append((cs, ce, clabel))

        # Resolve exact/overlapping counsel predictions conservatively.
        if mode == "longest_nonoverlap":
            chosen = []
            for span in sorted(
                accepted, key=lambda x: (-(x[1]-x[0]), x[0], x[1])
            ):
                if not any(_overlap(span, old) for old in chosen):
                    chosen.append(span)
            accepted = chosen

        # Judges have priority over counsel if they overlap.
        if mode == "judge_priority":
            accepted = [
                c for c in accepted
                if not any(_overlap(c, j[:2]) for j in judges)
            ]

        result[doc_id] = sorted(set(kept + accepted + judges))

    return result

variants = {
    "Current Counsel R + Judge": val_combined_fixed,
    "Block counsel overlapping judge": make_counsel_variant("block_judge_overlap"),
    "Longest non-overlapping counsel": make_counsel_variant("longest_nonoverlap"),
    "Judge priority over counsel": make_counsel_variant("judge_priority"),
}

rows = []
reports = {}

for name, pred in variants.items():
    macro, report = exact_report_fixed(pred, val_gold)
    reports[name] = report
    rows.append({
        "candidate": name,
        "macro_F1": macro,
        "JUDGE_F1": report.loc["JUDGE", "F1"] if "JUDGE" in report.index else 0.0,
        "A_COUNSEL_F1": report.loc["A.COUNSEL", "F1"] if "A.COUNSEL" in report.index else 0.0,
        "R_COUNSEL_F1": report.loc["R.COUNSEL", "F1"] if "R.COUNSEL" in report.index else 0.0,
        "total_spans": sum(len(v) for v in pred.values())
    })

ablation_df = pd.DataFrame(rows).sort_values("macro_F1", ascending=False)
display(ablation_df.round(4))

print(
    "\nReminder: these are validation experiments, not test scores. "
    "Choose conservatively and avoid repeated tuning against fold 2."
)


In [ ]:

# ============================================================
# CELL 1 — COUNSEL CONTEXT WINDOW ABLATION
# Keeps the current judge rules and champion checkpoint unchanged.
# ============================================================

import re
import pandas as pd

_required = [
    "val_texts", "val_gold", "val_champion_fixed",
    "val_combined_fixed", "counsel_spans_fixed",
    "judge_spans_fixed", "exact_report_fixed",
    "CUE_A", "CUE_R"
]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError(
        "Run the self-contained judge+counsel experiment cell first. "
        "Missing: " + ", ".join(_missing)
    )

def _counsel_cue_positions(text):
    cues = []
    for label, rx in [
        ("A.COUNSEL", CUE_A),
        ("R.COUNSEL", CUE_R)
    ]:
        for m in rx.finditer(text):
            cues.append((m.start(), m.end(), label))
    return cues

def _build_context_candidate(window, default_side="R", require_cue=False):
    """
    Keep the current counsel candidate only if a matching-side cue is
    within the selected distance. A candidate without a cue can be
    retained or dropped based on require_cue.
    """
    result = {}

    for doc_id, text in val_texts.items():
        counsel = counsel_spans_fixed(text, "R")
        judges = [(s, e, "JUDGE") for s, e in judge_spans_fixed(text)]
        cues = _counsel_cue_positions(text)

        accepted = []
        for s, e, label in counsel:
            matching = [
                c for c in cues
                if c[2] == label
                and min(abs(s - c[1]), abs(c[0] - e)) <= window
            ]

            if matching:
                accepted.append((s, e, label))
            elif not require_cue:
                accepted.append((s, e, label))

        # Preserve champion entities other than counsel and judge.
        kept = [
            tuple(ent[:3])
            for ent in val_champion_fixed.get(doc_id, [])
            if ent[2] not in ("A.COUNSEL", "R.COUNSEL", "JUDGE")
        ]

        # Judges retain priority over overlapping counsel.
        accepted = [
            c for c in accepted
            if not any(c[0] < j[1] and c[1] > j[0] for j in judges)
        ]

        result[doc_id] = sorted(set(kept + accepted + judges))

    return result

context_candidates = {
    "Current R + Judge": val_combined_fixed,
    "Window 100, allow no cue": _build_context_candidate(100, require_cue=False),
    "Window 250, allow no cue": _build_context_candidate(250, require_cue=False),
    "Window 500, allow no cue": _build_context_candidate(500, require_cue=False),
    "Window 100, require cue": _build_context_candidate(100, require_cue=True),
    "Window 250, require cue": _build_context_candidate(250, require_cue=True),
    "Window 500, require cue": _build_context_candidate(500, require_cue=True),
}

print("Built", len(context_candidates), "variants.")
print("No model weights or output files were changed.")


In [ ]:

# ============================================================
# CELL 2 — SCORE CONTEXT WINDOW VARIANTS
# ============================================================

rows = []
context_reports = {}

for name, predictions in context_candidates.items():
    macro, report = exact_report_fixed(predictions, val_gold)
    context_reports[name] = report

    def _f1(label):
        return float(report.loc[label, "F1"]) if label in report.index else 0.0

    def _pred(label):
        return int(report.loc[label, "predicted"]) if label in report.index else 0

    rows.append({
        "candidate": name,
        "macro_F1": macro,
        "A_COUNSEL_F1": _f1("A.COUNSEL"),
        "R_COUNSEL_F1": _f1("R.COUNSEL"),
        "JUDGE_F1": _f1("JUDGE"),
        "A_COUNSEL_pred": _pred("A.COUNSEL"),
        "R_COUNSEL_pred": _pred("R.COUNSEL"),
        "total_spans": sum(len(v) for v in predictions.values())
    })

context_results_df = (
    pd.DataFrame(rows)
    .sort_values("macro_F1", ascending=False)
    .reset_index(drop=True)
)

display(context_results_df.round(4))

print("\n=== COUNSEL PRECISION / RECALL FOR TOP VARIANTS ===")
for name in context_results_df.head(4)["candidate"]:
    print(f"\n{name}")
    display(
        context_reports[name].loc[
            context_reports[name].index.isin(["A.COUNSEL", "R.COUNSEL", "JUDGE"]),
            ["gold", "predicted", "TP", "FP", "FN", "precision", "recall", "F1"]
        ].round(4)
    )

print(
    "\nKeep the existing 0.3059 candidate unless a new variant improves "
    "validation macro-F1 and has sensible per-label precision/recall."
)


In [ ]:
# ============================================================
# NEXT EXPERIMENT: R.COUNSEL FALSE-POSITIVE FREQUENCY AUDIT
# Read-only. Does not change model weights or predictions.
# ============================================================

from collections import Counter, defaultdict
import pandas as pd

assert "val_texts" in globals()
assert "val_gold" in globals()
assert "val_combined_fixed" in globals()

fp_rows = []
name_counts = Counter()
name_contexts = defaultdict(list)

for doc_id, text in val_texts.items():
    gold = {
        (int(e[0]), int(e[1]), str(e[2]))
        for e in val_gold.get(doc_id, [])
        if len(e) >= 3
    }

    for span in val_combined_fixed.get(doc_id, []):
        if len(span) < 3 or span[2] != "R.COUNSEL":
            continue

        s, e, label = int(span[0]), int(span[1]), str(span[2])
        key = (s, e, label)

        if key not in gold:
            surface = text[s:e].strip()
            context = text[max(0, s-100):min(len(text), e+100)]
            context = context.replace("\n", " ")

            name_counts[surface] += 1
            name_contexts[surface].append({
                "doc_id": doc_id,
                "start": s,
                "end": e,
                "context": context
            })

            fp_rows.append({
                "doc_id": doc_id,
                "surface": surface,
                "start": s,
                "end": e,
                "context": context
            })

print("R.COUNSEL false-positive spans:", len(fp_rows))
print("Unique surface strings:", len(name_counts))

print("\n=== MOST FREQUENT FALSE-POSITIVE SURFACES ===")
freq_df = pd.DataFrame([
    {"surface": surface, "count": count}
    for surface, count in name_counts.most_common(40)
])
display(freq_df)

print("\n=== CONTEXT FOR THE MOST FREQUENT FALSE POSITIVES ===")
context_rows = []
for surface, count in name_counts.most_common(20):
    for item in name_contexts[surface][:3]:
        context_rows.append({
            "surface": surface,
            "count": count,
            **item
        })

display(pd.DataFrame(context_rows))

print(
    "\nUse these examples to identify repeated failure patterns. "
    "Do not automatically blacklist a name: a name can be counsel in "
    "one location and a non-counsel mention elsewhere."
)

In [ ]:
# Audit repeated R.COUNSEL false-positive names in context.
# Read-only: does not change model weights or prediction artifacts.

from collections import defaultdict
import pandas as pd

assert "val_texts" in globals()
assert "val_gold" in globals()
assert "val_combined_fixed" in globals()

targets = {"Ghosh", "Lingayat", "Chari", "Tayeng"}
rows = []

for doc_id, text in val_texts.items():
    gold = {
        (int(e[0]), int(e[1]), str(e[2]))
        for e in val_gold.get(doc_id, [])
        if len(e) >= 3
    }

    # Predicted R.COUNSEL spans
    for span in val_combined_fixed.get(doc_id, []):
        if len(span) < 3 or str(span[2]) != "R.COUNSEL":
            continue

        s, e = int(span[0]), int(span[1])
        surface = text[s:e].strip()

        if surface not in targets:
            continue

        is_exact_gold = (s, e, "R.COUNSEL") in gold
        context = text[max(0, s - 180):min(len(text), e + 180)]
        context = context.replace("\n", " ")

        rows.append({
            "surface": surface,
            "doc_id": doc_id,
            "start": s,
            "end": e,
            "exact_gold": is_exact_gold,
            "context": context
        })

audit_df = pd.DataFrame(rows)

if audit_df.empty:
    print("No matching predicted spans found.")
else:
    print("Total matching predictions:", len(audit_df))
    print("Exact gold matches:", int(audit_df["exact_gold"].sum()))
    print("\nOccurrences by name:")
    display(
        audit_df.groupby("surface")
        .agg(
            occurrences=("surface", "size"),
            exact_gold_matches=("exact_gold", "sum"),
            documents=("doc_id", "nunique")
        )
        .sort_values("occurrences", ascending=False)
    )

    # Show every occurrence, including true positives, to avoid
    # learning a misleading global blacklist.
    display(
        audit_df.sort_values(["surface", "doc_id", "start"])
        [["surface", "doc_id", "exact_gold", "context"]]
        .reset_index(drop=True)
    )

In [ ]:
import re
import pandas as pd

assert "val_texts" in globals()
assert "val_gold" in globals()
assert "val_combined_fixed" in globals()

targets = {
    "1404484": ["Chari"],
    "184861797": ["Ghosh"],
    "50547742": ["Lingayat"],
    "1680408": ["Tayeng"],
}

rows = []

for doc_id, names in targets.items():
    text = val_texts.get(doc_id)

    if text is None:
        print(f"Document {doc_id} not found in val_texts.")
        continue

    gold_spans = [
        (int(e[0]), int(e[1]), str(e[2]))
        for e in val_gold.get(doc_id, [])
        if len(e) >= 3
    ]

    pred_spans = [
        (int(e[0]), int(e[1]), str(e[2]))
        for e in val_combined_fixed.get(doc_id, [])
        if len(e) >= 3
    ]

    for name in names:
        pattern = re.compile(
            rf"(?<!\w){re.escape(name)}(?!\w)",
            flags=re.IGNORECASE
        )

        for match in pattern.finditer(text):
            s, e = match.span()

            nearby_gold = [
                (gs, ge, label, text[gs:ge])
                for gs, ge, label in gold_spans
                if gs < e + 100 and ge > s - 100
            ]

            nearby_preds = [
                (ps, pe, label, text[ps:pe])
                for ps, pe, label in pred_spans
                if ps < e + 100 and pe > s - 100
            ]

            rows.append({
                "doc_id": doc_id,
                "name": name,
                "position": s,
                "exact_predicted_surface": text[s:e],
                "context": text[max(0, s-180):min(len(text), e+180)]
                    .replace("\n", " "),
                "nearby_gold": nearby_gold,
                "nearby_predictions": nearby_preds,
            })

audit_all = pd.DataFrame(rows)

print("Total text occurrences examined:", len(audit_all))
display(audit_all[["doc_id", "name", "position", "context"]])

print("\nNearby gold annotations and predictions:")
display(audit_all[["doc_id", "name", "position",
                   "nearby_gold", "nearby_predictions"]])

In [ ]:
import copy

assert "val_texts" in globals()
assert "val_gold" in globals()
assert "val_combined_fixed" in globals()
assert "exact_report_fixed" in globals()

oracle_corrected = copy.deepcopy(val_combined_fixed)
corrected_count = 0

for doc_id in val_texts:
    gold_a = {
        (int(e[0]), int(e[1]))
        for e in val_gold.get(doc_id, [])
        if len(e) >= 3 and str(e[2]) == "A.COUNSEL"
    }

    updated = []

    for span in oracle_corrected.get(doc_id, []):
        s, e, label = int(span[0]), int(span[1]), str(span[2])

        if label == "R.COUNSEL" and (s, e) in gold_a:
            updated.append((s, e, "A.COUNSEL"))
            corrected_count += 1
        else:
            updated.append(tuple(span))

    oracle_corrected[doc_id] = updated

base_f1, base_report = exact_report_fixed(val_gold, val_combined_fixed)
oracle_f1, oracle_report = exact_report_fixed(val_gold, oracle_corrected)

print("Exact-span R->A corrections:", corrected_count)
print(f"Baseline validation macro-F1: {base_f1:.4f}")
print(f"Oracle diagnostic macro-F1:   {oracle_f1:.4f}")
print(f"Diagnostic gain:              {oracle_f1-base_f1:+.4f}")

print("\nCounsel metrics before:")
display(base_report.loc[["A.COUNSEL", "R.COUNSEL"]])

print("\nCounsel metrics after oracle correction:")
display(oracle_report.loc[["A.COUNSEL", "R.COUNSEL"]])

In [ ]:
import pandas as pd

assert "val_texts" in globals()
assert "val_gold" in globals()
assert "val_combined_fixed" in globals()

rows = []

for doc_id, text in val_texts.items():
    gold_a = {
        (int(e[0]), int(e[1]))
        for e in val_gold.get(doc_id, [])
        if len(e) >= 3 and str(e[2]) == "A.COUNSEL"
    }

    for span in val_combined_fixed.get(doc_id, []):
        if len(span) < 3 or str(span[2]) != "R.COUNSEL":
            continue

        s, e = int(span[0]), int(span[1])

        if (s, e) not in gold_a:
            continue

        left = text[max(0, s - 220):s]
        right = text[e:min(len(text), e + 220)]

        rows.append({
            "doc_id": doc_id,
            "surface": text[s:e],
            "left_context": left.replace("\n", " "),
            "right_context": right.replace("\n", " "),
        })

correction_audit = pd.DataFrame(rows)

print("Exact-span R->A correction cases:", len(correction_audit))
display(correction_audit)

In [ ]:
import re
import copy
import pandas as pd

assert "val_texts" in globals()
assert "val_gold" in globals()
assert "val_combined_fixed" in globals()
assert "exact_report_fixed" in globals()

# Candidate contextual cues. These are hypotheses to test, not
# assumed universal rules for the dataset's label convention.
SIDE_CUES = {
    "A.COUNSEL": [
        r"\bfor the appellant\b",
        r"\bon behalf of the appellant\b",
        r"\bappellant[s]?\s*,?\s*(?:appearing|represented)\b",
    ],
    "R.COUNSEL": [
        r"\bfor the respondent\b",
        r"\bon behalf of the respondent\b",
        r"\bfor the state\b",
        r"\bon behalf of the state\b",
        r"\brespondent[s]?\s*,?\s*(?:appearing|represented)\b",
    ],
}

compiled_cues = {
    label: [re.compile(p, re.I) for p in patterns]
    for label, patterns in SIDE_CUES.items()
}

def infer_side_from_context(text, start, end, window=220):
    context = text[max(0, start-window):min(len(text), end+window)]
    hits = {
        label: any(p.search(context) for p in patterns)
        for label, patterns in compiled_cues.items()
    }

    if hits["A.COUNSEL"] and not hits["R.COUNSEL"]:
        return "A.COUNSEL"
    if hits["R.COUNSEL"] and not hits["A.COUNSEL"]:
        return "R.COUNSEL"
    return None  # ambiguous or no explicit cue

diagnostic_preds = copy.deepcopy(val_combined_fixed)
eligible = 0
changed = 0
audit = []

for doc_id, text in val_texts.items():
    gold_counsel = {
        (int(e[0]), int(e[1]), str(e[2]))
        for e in val_gold.get(doc_id, [])
        if len(e) >= 3 and str(e[2]) in {"A.COUNSEL", "R.COUNSEL"}
    }

    updated = []

    for span in diagnostic_preds.get(doc_id, []):
        s, e, label = int(span[0]), int(span[1]), str(span[2])

        if label not in {"A.COUNSEL", "R.COUNSEL"}:
            updated.append(tuple(span))
            continue

        # Oracle eligibility: exact boundaries match a gold counsel span.
        gold_match = next(
            (glabel for gs, ge, glabel in gold_counsel if gs == s and ge == e),
            None
        )

        cue_side = infer_side_from_context(text, s, e)

        if gold_match is not None and cue_side is not None:
            eligible += 1
            if cue_side != label:
                changed += 1
                audit.append({
                    "doc_id": doc_id,
                    "surface": text[s:e],
                    "predicted_before": label,
                    "cue_side": cue_side,
                    "gold_label": gold_match,
                    "corrected": cue_side == gold_match,
                    "context": text[max(0, s-150):min(len(text), e+150)]
                        .replace("\n", " ")
                })
                updated.append((s, e, cue_side))
            else:
                updated.append(tuple(span))
        else:
            updated.append(tuple(span))

    diagnostic_preds[doc_id] = updated

base_f1, base_report = exact_report_fixed(val_gold, val_combined_fixed)
diag_f1, diag_report = exact_report_fixed(val_gold, diagnostic_preds)

print("Exact-span counsel predictions with unambiguous cues:", eligible)
print("Predictions relabelled:", changed)
print(f"Baseline macro-F1: {base_f1:.4f}")
print(f"Oracle cue diagnostic macro-F1: {diag_f1:.4f}")
print(f"Diagnostic gain: {diag_f1-base_f1:+.4f}")

print("\nCorrection audit:")
display(pd.DataFrame(audit))

print("\nCounsel metrics after diagnostic:")
display(diag_report.loc[["A.COUNSEL", "R.COUNSEL"]])

In [ ]:
# Inspect raw counsel prediction structure without modifying anything.

for name in ["val_raw_fixed", "val_champion_fixed", "champ_thr"]:
    if name in globals():
        obj = globals()[name]
        print(f"\n{name}: {type(obj).__name__}")
        if isinstance(obj, dict):
            print("Example keys:", list(obj.keys())[:3])
            if obj:
                first_key = next(iter(obj))
                print("First value type:", type(obj[first_key]).__name__)
                print("First value preview:", repr(obj[first_key])[:1200])
        else:
            print(repr(obj)[:1200])

In [ ]:

import pandas as pd
import numpy as np

assert "val_raw_fixed" in globals()
assert "val_texts" in globals()
assert "val_gold" in globals()
assert "champ_thr" in globals()
assert "threshold_fixed" in globals()
assert "exact_report_fixed" in globals()

# Preserve all existing thresholds except the two counsel labels.
base_thresholds = dict(champ_thr)

# Keep the sweep small and interpretable.
a_grid = [0.15, 0.25, 0.35, 0.45, 0.55, 0.65]
r_grid = [0.15, 0.25, 0.35, 0.45, 0.55, 0.65]

results = []

for a_thr in a_grid:
    for r_thr in r_grid:
        trial_thr = dict(base_thresholds)
        trial_thr["A.COUNSEL"] = a_thr
        trial_thr["R.COUNSEL"] = r_thr

        trial_pred = threshold_fixed(val_raw_fixed, trial_thr)
        score, report = exact_report_fixed(val_gold, trial_pred)

        results.append({
            "macro_f1": score,
            "A_threshold": a_thr,
            "R_threshold": r_thr,
            "A_F1": (
                float(report.loc["A.COUNSEL", "F1"])
                if "A.COUNSEL" in report.index else np.nan
            ),
            "R_F1": (
                float(report.loc["R.COUNSEL", "F1"])
                if "R.COUNSEL" in report.index else np.nan
            ),
        })

sweep_df = (
    pd.DataFrame(results)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

print("Current counsel thresholds:",
      base_thresholds.get("A.COUNSEL"),
      base_thresholds.get("R.COUNSEL"))
print("Current validation baseline: compare against 0.3670")
display(sweep_df.head(15))

print("\nMacro-F1 by threshold pair:")
display(
    sweep_df.pivot(
        index="A_threshold",
        columns="R_threshold",
        values="macro_f1"
    ).round(4)
)


In [ ]:
from collections import Counter

print("=== GOLD LABEL COUNTS ===")
gold_counts = Counter(
    str(e[2])
    for doc_id in val_gold
    for e in val_gold[doc_id]
    if len(e) >= 3
)
print(gold_counts)

print("\n=== EXISTING FILTERED PREDICTIONS ===")
pred_counts = Counter(
    str(e[2])
    for doc_id in val_champion_fixed
    for e in val_champion_fixed[doc_id]
    if len(e) >= 3
)
print(pred_counts)

print("\n=== THRESHOLD FUNCTION OUTPUT ===")
probe_thr = dict(champ_thr)
probe_pred = threshold_fixed(val_raw_fixed, probe_thr)

probe_counts = Counter(
    str(e[2])
    for doc_id in probe_pred
    for e in probe_pred[doc_id]
    if len(e) >= 3
)
print(probe_counts)

print("\n=== COUNSEL THRESHOLDS ===")
print({
    "A.COUNSEL": probe_thr.get("A.COUNSEL"),
    "R.COUNSEL": probe_thr.get("R.COUNSEL"),
})

print("\n=== REPORT INDEX ===")
probe_f1, probe_report = exact_report_fixed(val_gold, probe_pred)
print("Macro-F1:", probe_f1)
print("Report labels:", list(probe_report.index))
display(probe_report)

In [ ]:

from collections import Counter
import pandas as pd

assert "val_raw_fixed" in globals()
assert "val_gold" in globals()
assert "champ_thr" in globals()
assert "exact_report_fixed" in globals()

def threshold_fixed_v2(raw_predictions, thresholds):
    output = {}

    for doc_id, spans in raw_predictions.items():
        kept = []

        for span in spans:
            if len(span) < 4:
                continue

            start, end, label, confidence = span[:4]
            label = str(label)
            threshold = thresholds.get(label)

            if threshold is not None and float(confidence) >= float(threshold):
                kept.append((int(start), int(end), label))

        output[doc_id] = kept

    return output

# Apply the current thresholds without changing them.
val_threshold_v2 = threshold_fixed_v2(val_raw_fixed, champ_thr)

gold_counts = Counter(
    str(e[2])
    for entities in val_gold.values()
    for e in entities
    if len(e) >= 3
)

pred_counts = Counter(
    str(e[2])
    for entities in val_threshold_v2.values()
    for e in entities
    if len(e) >= 3
)

print("=== GOLD COUNTS ===")
print(gold_counts)

print("\n=== THRESHOLD OUTPUT COUNTS ===")
print(pred_counts)

score_v2, report_v2 = exact_report_fixed(val_gold, val_threshold_v2)
print(f"\nMacro-F1: {score_v2:.4f}")
display(report_v2)

expected = {"A.COUNSEL", "R.COUNSEL", "RESP", "WIT"}
missing = expected - set(pred_counts)

if missing:
    print("\nWARNING: Still missing prediction labels:", sorted(missing))
    print("Check raw model outputs and threshold configuration before tuning.")
else:
    print("\nAll four previously missing labels have predictions.")


In [ ]:

# Diagnose why four entity labels are absent from raw predictions.
# Read-only inspection.

from collections import Counter

assert "val_raw_fixed" in globals()

raw_counts = Counter(
    str(span[2])
    for spans in val_raw_fixed.values()
    for span in spans
    if len(span) >= 4
)

print("=== LABELS PRESENT IN RAW SPANS ===")
print(raw_counts)

print("\n=== MODEL / LABEL CONFIG VARIABLES ===")
for name in sorted(globals()):
    name_lower = name.lower()
    if any(k in name_lower for k in [
        "id2label", "label2id", "label_list", "labels",
        "num_labels", "model_config", "ner_model"
    ]):
        obj = globals()[name]
        if isinstance(obj, (dict, list, tuple, str, int)):
            print(f"{name}: {repr(obj)[:2000]}")
        else:
            print(f"{name}: {type(obj).__name__}")

print("\n=== RAW SPAN EXAMPLES FOR COUNSEL / RESP / WIT ===")
for doc_id, spans in val_raw_fixed.items():
    matches = [
        span for span in spans
        if len(span) >= 4 and any(
            token in str(span[2]).upper()
            for token in ["COUNSEL", "RESP", "WIT"]
        )
    ]
    if matches:
        print(doc_id, matches[:20])


In [ ]:
import inspect

names_to_inspect = [
    "predict_fixed",
    "threshold_fixed",
    "threshold_fixed_v2",
    "predict_texts",
    "apply_thr",
    "merge_policy",
    "covering_labels",
    "rule_spans",
]

for name in names_to_inspect:
    obj = globals().get(name)

    if obj is None:
        continue

    print("\n" + "=" * 90)
    print("FUNCTION:", name)
    try:
        print(inspect.getsource(obj)[:10000])
    except (OSError, TypeError):
        print("Source unavailable:", repr(obj))

print("\n" + "=" * 90)
print("MODEL TYPE:", type(ner_model).__name__)

for attr in ["allowed_labels", "weak_labels", "counsel_labels",
             "id2label", "label2id", "config"]:
    if hasattr(ner_model, attr):
        value = getattr(ner_model, attr)
        print(f"\nner_model.{attr} =", repr(value)[:3000])

print("\n_original_ner_model type:",
      type(_original_ner_model).__name__ if "_original_ner_model" in globals()
      else "not present")

In [ ]:

import torch
import pandas as pd
from collections import Counter

assert "diag_doc_probs" in globals()
assert "bio_viterbi_decode" in globals()
assert "BIO_LABELS" in globals()
assert "val_texts" in globals()

model = ner_model
model.eval()

weak_bio = {
    i: lab for i, lab in enumerate(BIO_LABELS)
    if lab != "O" and any(
        entity in lab for entity in
        ["A.COUNSEL", "R.COUNSEL", "RESP", "WIT"]
    )
}

prob_rows = []
decoded_counts = Counter()
documents_checked = 0

with torch.inference_mode():
    for doc_id, text in val_texts.items():
        offsets, probs = diag_doc_probs(model, text)
        if len(offsets) == 0:
            continue

        probs = torch.as_tensor(probs)
        pred_ids = bio_viterbi_decode(
            torch.log(probs.clamp_min(1e-12))
        )
        documents_checked += 1

        for idx, label in enumerate(BIO_LABELS):
            if label in weak_bio.values():
                prob_rows.append({
                    "label": label,
                    "doc_id": doc_id,
                    "mean_probability": float(probs[:, idx].mean()),
                    "max_probability": float(probs[:, idx].max()),
                    "tokens_above_0.10": int((probs[:, idx] >= 0.10).sum()),
                    "tokens_above_0.25": int((probs[:, idx] >= 0.25).sum()),
                })

        decoded_counts.update(BIO_LABELS[int(i)] for i in pred_ids)

print("Documents checked:", documents_checked)

print("\n=== BIO TAGS SELECTED BY VITERBI ===")
display(pd.DataFrame([
    {"BIO_label": label, "decoded_tokens": decoded_counts[label]}
    for label in weak_bio.values()
]))

print("\n=== PROBABILITY SUMMARY BY BIO LABEL ===")
prob_df = pd.DataFrame(prob_rows)
if not prob_df.empty:
    display(
        prob_df.groupby("label")[
            ["mean_probability", "max_probability",
             "tokens_above_0.10", "tokens_above_0.25"]
        ].mean().round(4)
    )


In [ ]:

import json
from pathlib import Path
from collections import Counter, defaultdict

# Confirm the existing pipeline state.
_required = [
    "champion_model", "predict_texts", "apply_thr", "champ_thr",
    "merge_policy", "merge_all", "keep", "chosen",
    "val_texts", "val_gold", "ENT"
]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError("Missing definitions: " + ", ".join(_missing))

def _normalise_entities(obj):
    """Convert document -> entity-list mappings into exact span triples."""
    out = {}
    for doc_id, entities in obj.items():
        rows = []
        for ent in entities:
            if len(ent) < 3:
                raise ValueError(f"Malformed entity for document {doc_id}: {ent}")
            rows.append((int(ent[0]), int(ent[1]), str(ent[2])))
        out[str(doc_id)] = set(rows)
    return out

def _exact_macro_f1(pred_obj, gold_obj):
    pred = _normalise_entities(pred_obj)
    gold = _normalise_entities(gold_obj)
    labels = sorted(
        {x[2] for ents in gold.values() for x in ents}
        | {x[2] for ents in pred.values() for x in ents}
    )

    per_label = {}
    for label in labels:
        tp = fp = fn = 0
        for doc_id in set(pred) | set(gold):
            p = {x for x in pred.get(doc_id, set()) if x[2] == label}
            g = {x for x in gold.get(doc_id, set()) if x[2] == label}
            tp += len(p & g)
            fp += len(p - g)
            fn += len(g - p)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )
        per_label[label] = (tp, fp, fn, precision, recall, f1)

    macro = sum(x[-1] for x in per_label.values()) / max(len(per_label), 1)
    return macro, per_label

# Recompute validation predictions from the existing champion model.
champion_model.eval()
val_raw_now = predict_texts(champion_model, val_texts)
val_filtered_now = apply_thr(val_raw_now, champ_thr)

val_counsel_now = merge_policy(val_texts, val_filtered_now, chosen)
val_judge_now = merge_all(val_texts, val_filtered_now, keep)

counsel_f1, counsel_by_label = _exact_macro_f1(val_counsel_now, val_gold)
judge_f1, judge_by_label = _exact_macro_f1(val_judge_now, val_gold)

print(f"Fold-2 champion + counsel ({chosen}): {counsel_f1:.4f}")
print(f"Fold-2 champion + counsel + JUDGE:     {judge_f1:.4f}")
print(f"Difference:                            {judge_f1 - counsel_f1:+.4f}")

print("\nPer-label F1 comparison:")
for label in sorted(set(counsel_by_label) | set(judge_by_label)):
    a = counsel_by_label.get(label, (0, 0, 0, 0, 0, 0))[-1]
    b = judge_by_label.get(label, (0, 0, 0, 0, 0, 0))[-1]
    print(f"{label:12s} counsel={a:.3f}  +JUDGE={b:.3f}")

# Do not silently select a worse policy.
if judge_f1 + 1e-12 < counsel_f1:
    raise RuntimeError(
        "Validation gate failed: JUDGE rules reduce exact-span macro-F1. "
        "No test predictions were written. Use the counsel-only pipeline "
        "only after explicitly deciding that is the desired final policy."
    )

# Load the same fold-3 file identified in your previous output.
fold3_path = Path(
    "/kaggle/input/datasets/oipranesh/"
    "cs781-working-backup-20261010-215243/"
    "phase2_lner/Phase_2_LNER/Test/fold3-text.json"
)
if not fold3_path.is_file():
    raise FileNotFoundError(f"Fold-3 file not found: {fold3_path}")

with open(fold3_path, encoding="utf-8") as f:
    test_texts_final = json.load(f)

test_raw_now = predict_texts(champion_model, test_texts_final)
test_filtered_now = apply_thr(test_raw_now, champ_thr)
test_final_now = merge_all(test_texts_final, test_filtered_now, keep)

prediction_obj = {
    str(doc_id): [
        [int(s), int(e), str(label)]
        for s, e, label in sorted(set(spans))
    ]
    for doc_id, spans in test_final_now.items()
}

# Validate offsets, labels, and overlaps before writing.
issues = Counter()
text_by_id = {str(k): v for k, v in test_texts_final.items()}
for doc_id, spans in prediction_obj.items():
    text = text_by_id.get(doc_id)
    if text is None:
        issues["unknown_document"] += 1
        continue

    checked = []
    for span in spans:
        if len(span) != 3:
            issues["malformed_span"] += 1
            continue
        s, e, label = span
        if label not in ENT:
            issues["unknown_label"] += 1
        if not (0 <= s < e <= len(text)):
            issues["invalid_offsets"] += 1
            continue
        if text[s:e] != text[s:e].strip():
            issues["whitespace_boundaries"] += 1
        checked.append((s, e))

    checked.sort()
    issues["overlaps"] += sum(
        1 for a, b in zip(checked, checked[1:]) if b[0] < a[1]
    )

if len(prediction_obj) != len(text_by_id):
    issues["document_count_mismatch"] += 1

issues = Counter({k: v for k, v in issues.items() if v})
print("\nFold-3 documents:", len(prediction_obj))
print("Fold-3 spans:", sum(len(v) for v in prediction_obj.values()))
print("Format checks:", dict(issues) if issues else "PASS")

if issues:
    raise RuntimeError("Format checks failed; no prediction file was written.")

output_path = Path("/kaggle/working/prediction_champion_plus_rules.json")
if output_path.exists():
    raise FileExistsError(
        f"Refusing to overwrite {output_path}. "
        "Rename the output path explicitly if you want another version."
    )

with open(output_path, "x", encoding="utf-8") as f:
    json.dump(prediction_obj, f, ensure_ascii=False)

with open(output_path, encoding="utf-8") as f:
    assert json.load(f) == prediction_obj

print("FINAL PREDICTIONS SAVED:", output_path)


In [ ]:

import re
from collections import Counter

# Confirm that the existing notebook state contains the required inputs.
_required = [
    "train_texts", "train_gold", "counsel_spans",
    "chosen", "A_", "R_", "champion_model",
    "predict_texts", "apply_thr", "champ_thr",
    "val_texts", "c_filt", "score", "as_scored", "ENT"
]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError(
        "Missing existing notebook definitions: " + ", ".join(_missing)
        + ". Stop here; do not generate predictions yet."
    )

# Recreate the exact pattern definitions from original cell 117.
CAP = r"[A-ZÀ-Ö][A-Za-zÀ-ÖØ-öø-ÿ'’\-]*"
INIT = r"[A-Z]\.(?:[ \t]*[A-Z]\.)*"
WORD = rf"(?:{INIT}|{CAP})"
NAME2 = rf"{WORD}(?:[ \t]+{WORD}){{0,3}}"
TOK_RE = re.compile(WORD)

AJ_RE = re.compile(rf"(?i:\bjustice)[ \t]+({NAME2})")
B_RE = re.compile(r",[ \t]*J{1,2}\b\.?")
B2_RE = re.compile(
    r"[ \t]J{1,2}\.?(?=[ \t]*(?:in\b|speaking|writing|observed|"
    r"held\b|was\b|said\b|while\b|who\b|delivered|dissent|"
    r"agreed|also\b|noted|stated))"
)
C_RE = re.compile(r"\bJJ?\.?[ \t]*\(([^()\n]{3,45})\)")
D_RE = re.compile(
    r"(?:BENCH\s*:|\(J\)|\(J\.\))\s*"
    r"([A-Z][A-Za-z']+,[ \t]*[A-Z](?:[ \t]*\.?[ \t]*[A-Z])*\.?)[ \t]*\(J\)"
)
E_RE = re.compile(r"\bJJ?\.?[ \t]*\[[ \t]*([A-Za-zÀ-ÿ\.'’ \t\-]{3,45}?)[ \t]*\]")
F_RE = re.compile(
    r"BENCH[ \t]*:[ \t]*([^\n]{3,120}?)(?=[ \t]*(?:JUDG(?:E)?MENT|"
    r"CITATION|ACT\b|\n|$))"
)
NAMES = [
    "A Justice NAME", "B NAME, J.", "B2 NAME J (no comma)",
    "C J.(NAME)", "D BENCH: X, Y (J)", "E J [NAME]", "F BENCH: A & B"
]

def _trim_judge(text, s, e):
    while s < e and (text[s].isspace() or text[s] in ".,;:"):
        s += 1
    while e > s and (text[e - 1].isspace() or text[e - 1] in ",;:"):
        e -= 1
    return (s, e) if e > s else None

def _back_name(text, end, max_toks=4):
    ws = max(0, end - 80)
    toks = [(m.start() + ws, m.end() + ws)
            for m in TOK_RE.finditer(text[ws:end])]
    run = []
    for a, b in reversed(toks):
        nxt = run[0][0] if run else end
        if text[b:nxt].strip(" \t") != "":
            break
        if text[a:b].upper().strip(".") in {
            "JUDGMENT", "JUDGEMENT", "ORDER", "BENCH", "CORAM",
            "BEFORE", "PRESENT", "HON'BLE", "MR", "MRS", "MS",
            "JUSTICE", "THE", "AND", "OF", "DATE", "NEW",
            "DELHI", "BY", "DELIVERED"
        }:
            break
        run.insert(0, (a, b))
        if len(run) == max_toks:
            break
    return (run[0][0], run[-1][1]) if run else None

def judge_patterns2(text):
    out = {n: set() for n in NAMES}

    def add(name, s, e):
        span = _trim_judge(text, s, e)
        if span:
            out[name].add(span)

    for m in AJ_RE.finditer(text):
        add(NAMES[0], m.start(1), m.end(1))

    for rx, name in [(B_RE, NAMES[1]), (B2_RE, NAMES[2])]:
        for m in rx.finditer(text):
            bn = _back_name(text, m.start())
            if bn:
                add(name, *bn)

    for rx, name in [(C_RE, NAMES[3]), (E_RE, NAMES[5])]:
        for m in rx.finditer(text):
            inner = m.group(1).strip()
            if (
                inner and inner[0].isupper()
                and inner.upper() not in {"J", "JJ"}
                and re.fullmatch(r"[A-Za-zÀ-ÿ\.'’, \-]+", inner)
            ):
                add(name, m.start(1), m.end(1))

    for m in D_RE.finditer(text):
        add(NAMES[4], m.start(1), m.end(1))

    for m in F_RE.finditer(text):
        seg, base, pos = m.group(1), m.start(1), 0
        for part in re.split(r"[ \t]*(?:&|\bAND\b)[ \t]*", seg):
            k = seg.find(part, pos)
            if k < 0:
                continue
            pos = k + len(part)
            part2 = re.sub(r"[ \t]*\((?:C?J|J\.?)\)\s*$", "", part)
            span = _trim_judge(text, base + k, base + k + len(part2))
            if span and re.fullmatch(NAME2, text[span[0]:span[1]]):
                out[NAMES[6]].add(span)

    return out

def resolve(spans):
    out = []
    for s, e in sorted(set(spans), key=lambda x: (x[0], -(x[1] - x[0]))):
        if out and s < out[-1][1]:
            continue
        out.append((s, e))
    return out

# Select patterns using FOLD 1 only, matching the original notebook rule.
G = {
    (d, int(e[0]), int(e[1]))
    for d, entities in train_gold.items()
    for e in entities
    if e[2] == "JUDGE"
}
if not G:
    raise RuntimeError("No fold-1 JUDGE gold spans found; refusing to select patterns.")

Pn = {name: set() for name in NAMES}
for d, text in train_texts.items():
    for name, spans in judge_patterns2(text).items():
        Pn[name] |= {(d, s, e) for s, e in spans}

keep = []
print("Fold-1 JUDGE pattern selection:")
for name in NAMES:
    tp = len(G & Pn[name])
    precision = tp / max(len(Pn[name]), 1)
    print(f"{name:24s} predictions={len(Pn[name]):4d} "
          f"TP={tp:3d} precision={precision:.3f}")
    if precision >= 0.4 and tp >= 5:
        keep.append(name)

print("Selected patterns:", keep)

def judge_spans_doc(text, keep):
    spans = set()
    for name, matches in judge_patterns2(text).items():
        if name in keep:
            spans |= matches
    return resolve(spans)

# Restore the original merge logic from cell 117.
def merge_all(texts, champ_filtered, keep):
    out = {}
    for d, text in texts.items():
        rules = list(counsel_spans(text, chosen)) + [
            (s, e, "JUDGE") for s, e in judge_spans_doc(text, keep)
        ]
        kept = [
            span for span in champ_filtered[d]
            if span[2] not in (A_, R_, "JUDGE")
        ]
        kept = [
            span for span in kept
            if not any(span[0] < re_ and span[1] > rs
                       for rs, re_, _ in rules)
        ]
        out[d] = sorted(set(kept + rules))
    return out

print("\nRecovery complete.")
print("keep and merge_all are now defined.")
print("Champion checkpoint was not modified.")


In [ ]:
required = [
    "judge_patterns2",
    "NAMES",
    "merge_all",
    "merge_policy",
    "final_exact_span_report",
    "val_texts",
    "val_gold",
    "c_filt",
    "keep",
]

missing = [name for name in required if name not in globals()]

if missing:
    print("Still missing:", missing)
    print("Run the notebook cell that defines these helpers before evaluation.")
else:
    print("All required helpers are available. Ready to evaluate.")

In [ ]:
required = [
    "judge_patterns2", "NAMES", "merge_all", "keep",
    "merge_policy", "final_exact_span_report",
    "val_texts", "val_gold", "c_filt"
]

missing = [x for x in required if x not in globals()]
print("Missing:", missing)
print("Ready:", not missing)

In [ ]:
import random
from collections import defaultdict
random.seed(1)

HON_BEFORE = re.compile(r"(?:Mr|Ms|Mrs|Shri|Sri|Smt)\.?[ \t]*$")

def side_state(text, s):
    ctx = text[max(0, s - 600):s]
    a = [m.start() for m in SIDE_A.finditer(ctx)]; r = [m.start() for m in SIDE_R.finditer(ctx)]
    if not a and not r: return "none"
    if a and not r: return "A-cue only"
    if r and not a: return "R-cue only"
    return "both (nearer wins)"

def side_error_report(texts, gold, name, show=6):
    acc = defaultdict(lambda: [0, 0]); wrong, missed = [], []
    for d, text in texts.items():
        gl = {(int(e[0]), int(e[1])): e[2] for e in gold[d] if e[2] in ("A.COUNSEL", "R.COUNSEL")}
        rule = set(rule_spans(text))
        for (s, e) in rule:
            if (s, e) in gl:
                st, pred = side_state(text, s), side(text, s)
                ok = pred == gl[(s, e)]
                acc[st][0] += ok; acc[st][1] += 1
                if not ok:
                    wrong.append((text[max(0, s - 160):s].replace("\n", " "), text[s:e], gl[(s, e)], pred, st))
        for (s, e), lab in gl.items():
            if (s, e) not in rule:
                left = text[max(0, s - 40):s]
                missed.append((left.replace("\n", " "), text[s:e].replace("\n", " "), lab, bool(HON_BEFORE.search(left))))
    print(f"\n=== {name}: side accuracy on boundary-correct spans, by cue state ===")
    for st, (ok, n) in sorted(acc.items()):
        print(f"  {st:20s} {ok:3d}/{n:3d} = {ok/max(n,1):.2f}")
    print(f"\n--- {min(show, len(wrong))} wrong-side examples (160 chars of left context) ---")
    for ctx, nm, gl_lab, pr, st in random.sample(wrong, min(show, len(wrong))):
        print(f"[{st}] gold={gl_lab} pred={pr}\n   ctx: ...{ctx!r}\n   name: {nm!r}")
    n_hon = sum(1 for m in missed if m[3])
    print(f"\n--- missed gold counsel spans: {len(missed)} total; {n_hon} follow an honorific (boundary issue), "
          f"{len(missed) - n_hon} do not (no honorific) ---")
    for left, nm, lab, hon in random.sample(missed, min(show, len(missed))):
        print(f"gold={lab} honorific_before={hon}\n   left: ...{left!r}\n   gold: {nm!r}")

side_error_report(train_texts, train_gold, "FOLD 1 (dev)")

In [ ]:
PARTY_A = r"(?:petitioners?|appellants?|applicants?|plaintiffs?|complainants?|prosecution|revisionists?)"
PARTY_R = r"(?:respondents?|defendants?|accused|non-\s*applicants?|opposite\s+part(?:y|ies))"
FOR = r"(?:for|on\s+behalf\s+of|appearing\s+for|representing)"
MODS = r"(?:(?:learned|first|second|third|\d+\w*|all|other)\s+)*"
CUE_A = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_A}", re.I)
CUE_R = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_R}", re.I)
HDR_A = re.compile(rf"{PARTY_A}s?[\s\.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HDR_R = re.compile(rf"{PARTY_R}s?[\s\.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HON_TAIL = re.compile(r"(?:Mr|Ms|Mrs|Shri|Sri|Smt)\.?[ \t]*$")
A_, R_ = "A.COUNSEL", "R.COUNSEL"

def _cues(text, lo, hi):
    out = [(m.start(), m.end(), A_) for m in CUE_A.finditer(text, lo, hi)]
    out += [(m.start(), m.end(), R_) for m in CUE_R.finditer(text, lo, hi)]
    return out

def assign_side(text, s, e, order, prev, W_L=300, W_R=200):
    for step in order:
        if step == "right":
            c = _cues(text, e, min(len(text), e + W_R))
            if c: return min(c)[2], "right"
        elif step == "left":
            c = _cues(text, max(0, s - W_L), s)
            c = [x for x in c if x[1] <= s]
            if c: return max(c, key=lambda x: x[1])[2], "left"
        elif step == "hdr":
            tail = HON_TAIL.sub("", text[max(0, s - 80):s])
            if HDR_A.search(tail): return A_, "hdr"
            if HDR_R.search(tail): return R_, "hdr"
        elif step == "inherit":
            if prev and s - prev[0] <= 200 and prev[2] != "default":
                return prev[1], "inherit"
    return R_, "default"

VARIANTS = {"V1 left-for": ["left"], "V2 right-for": ["right"], "V3 right>left": ["right", "left"],
            "V4 left>right": ["left", "right"], "V5 hdr>right>left>inh": ["hdr", "right", "left", "inherit"],
            "V6 right>hdr>left>inh": ["right", "hdr", "left", "inherit"]}

def eval_variants(texts, gold):
    G = {A_: set(), R_: set()}
    for d, ents in gold.items():
        for x in ents:
            if x[2] in G: G[x[2]].add((d, int(x[0]), int(x[1])))
    gl = {}
    for lab, st in G.items():
        for k in st: gl[k] = lab
    rows, src_stats = [], None
    runs = {"V0 current (600-char any-word)": None, **VARIANTS}
    for name, order in runs.items():
        P = {A_: set(), R_: set()}; ok = n = cue = tot = 0; stats = {}
        for d, text in texts.items():
            prev = None
            for s, e in sorted(rule_spans(text)):
                if order is None:
                    lab, src = side(text, s), ("cue" if side_state(text, s) != "none" else "default")
                else:
                    lab, src = assign_side(text, s, e, order, prev)
                    prev = (e, lab, src)
                P[lab].add((d, s, e)); tot += 1; cue += (src != "default")
                if (d, s, e) in gl:
                    n += 1; hit = (lab == gl[(d, s, e)]); ok += hit
                    t = stats.setdefault(src, [0, 0]); t[0] += hit; t[1] += 1
        f = {}
        for lab in (A_, R_):
            tp = len(G[lab] & P[lab]); p = tp / max(len(P[lab]), 1); r = tp / max(len(G[lab]), 1)
            f[lab] = 2 * p * r / (p + r) if p + r else 0.0
        rows.append({"variant": name, "side_acc": ok / n, "n_bnd_correct": n, "cue_coverage": cue / tot,
                     "A.COUNSEL_F1": f[A_], "R.COUNSEL_F1": f[R_], "A+R_mean": (f[A_] + f[R_]) / 2})
        if name.startswith("V5"): src_stats = stats
    print(pd.DataFrame(rows).round(3).to_string(index=False))
    print("\nV5 accuracy by cue source (boundary-correct spans):")
    for src, (h, k) in src_stats.items(): print(f"  {src:8s} {h:3d}/{k:3d} = {h/k:.2f}")

eval_variants(train_texts, train_gold)   # FOLD 1 ONLY

In [ ]:
ORDER = ["hdr", "right", "left", "inherit"]   # V5

def counsel_spans(text, policy):
    out, prev = [], None
    for s, e in sorted(rule_spans(text)):
        lab, src = assign_side(text, s, e, ORDER, prev)
        prev = (e, lab, src)
        if src == "default":
            if policy == "drop": continue
            if policy == "A": lab = A_
        out.append((s, e, lab))
    return out

def counsel_f1(texts, gold, policy):
    G = {A_: set(), R_: set()}; P = {A_: set(), R_: set()}
    for d, ents in gold.items():
        for x in ents:
            if x[2] in G: G[x[2]].add((d, int(x[0]), int(x[1])))
    for d, text in texts.items():
        for s, e, lab in counsel_spans(text, policy): P[lab].add((d, s, e))
    f = {}
    for lab in (A_, R_):
        tp = len(G[lab] & P[lab]); p = tp / max(len(P[lab]), 1); r = tp / max(len(G[lab]), 1)
        f[lab] = 2 * p * r / (p + r) if p + r else 0.0
    return f

# ---- 1. choose default policy on FOLD 1 only ----
fold1 = {pol: counsel_f1(train_texts, train_gold, pol) for pol in ["R", "A", "drop"]}
for pol, f in fold1.items():
    print(f"fold 1, default={pol:5s}: A={f[A_]:.3f} R={f[R_]:.3f} mean={(f[A_]+f[R_])/2:.3f}")
chosen = max(fold1, key=lambda p: (fold1[p][A_] + fold1[p][R_], p == "R"))
print("chosen default policy (fold 1):", chosen)

# ---- 2. fold 2, once ----
def predict_texts(model, texts):
    preds = {}
    for doc_id, text in texts.items():
        offsets, probs = diag_doc_probs(model, text)
        if len(offsets) == 0: preds[doc_id] = []; continue
        pt = torch.tensor(bio_viterbi_decode(torch.log(probs.clamp_min(1e-12))))
        starts = [a for a, b in offsets]; ends = [b for a, b in offsets]
        res = []
        for s, e, lab in decode_spans(text, offsets, pt.tolist()):
            tk = [i for i in range(bisect_right(ends, s), bisect_left(starts, e)) if ends[i] > starts[i]]
            res.append((int(s), int(e), lab, float(probs[tk, pt[tk]].mean()) if tk else 0.0))
        preds[doc_id] = res
    return preds

def apply_thr(raw, thr):
    return {d: [(s, e, l) for s, e, l, c in sp if c >= thr.get(l, 0.0)] for d, sp in raw.items()}

def as_scored(p): return {d: [(s, e, l, 1.0) for s, e, l in sp] for d, sp in p.items()}

def merge_policy(texts, champ_filtered, policy):
    out = {}
    for d, text in texts.items():
        rules = counsel_spans(text, policy)
        keep = [k for k in champ_filtered[d] if k[2] not in (A_, R_)]
        keep = [k for k in keep if not any(k[0] < re_ and k[1] > rs for rs, re_, _ in rules)]
        out[d] = sorted(set(keep + rules))
    return out

champion_model.eval()
c_filt = apply_thr(predict_texts(champion_model, val_texts), champ_thr)
c_macro, c_f1 = score(as_scored(c_filt), {})
assert abs(c_macro - 0.1586) < 2e-3, f"champion fold-2 macro {c_macro:.4f} != 0.1586; stop"
m_macro, m_f1 = score(as_scored(merge_policy(val_texts, c_filt, chosen)), {})
print(f"\nFOLD 2: champion {c_macro:.4f} -> champion + counsel rules (default={chosen}) {m_macro:.4f}  (delta {m_macro - c_macro:+.4f})")
print(pd.DataFrame({"champion": c_f1, "champ+rules": m_f1}).round(3).to_string())

In [ ]:
import re, random, numpy as np
from collections import Counter
random.seed(2)

LOOK = ["APP", "RESP", "WIT", "JUDGE"]
def norm(s): return re.sub(r"\s+", " ", s).strip().lower()

stats = {l: {"n": 0, "len": [], "pos": [], "prev": Counter(), "next": Counter(), "ex": []} for l in LOOK}
for d, text in train_texts.items():          # FOLD 1 ONLY
    L = len(text)
    for e in train_gold[d]:
        l = e[2]
        if l not in stats: continue
        s, t = int(e[0]), int(e[1]); st = stats[l]
        st["n"] += 1; st["len"].append(t - s); st["pos"].append(s / max(L, 1))
        left = norm(text[max(0, s - 40):s]).split(); right = norm(text[t:t + 40]).split()
        st["prev"][" ".join(left[-2:]) or "<bos>"] += 1
        st["next"][" ".join(right[:2]) or "<eos>"] += 1
        st["ex"].append((text[max(0, s - 60):s].replace("\n", " "),
                         text[s:t].replace("\n", " "),
                         text[t:t + 40].replace("\n", " ")))

for l in LOOK:
    st = stats[l]; ln = np.array(st["len"]); ps = np.array(st["pos"])
    print(f"\n===== {l}: n={st['n']}  chars median={np.median(ln):.0f} p90={np.percentile(ln, 90):.0f}  "
          f"doc-position median={np.median(ps):.2f}  share in first 5% of doc={np.mean(ps < 0.05):.2f}")
    print("  prev2:", st["prev"].most_common(8))
    print("  next2:", st["next"].most_common(8))
    for a, b, c in random.sample(st["ex"], min(5, len(st["ex"]))):
        print(f"   ...{a!r} [[{b[:80]!r}]] {c!r}")

In [ ]:
import re, random
random.seed(3)

TOK = r"(?:[A-Z]\.(?:\s?[A-Z]\.)*|[A-Z][a-z']+|[A-Z]{2,})"
TOK_RE = re.compile(TOK)
NAME = rf"{TOK}(?:[ \t]+{TOK}){{0,3}}"
STOP = {"JUDGMENT","JUDGEMENT","ORDER","BENCH","CORAM","BEFORE","PRESENT","HON'BLE","MR","MRS","MS",
        "JUSTICE","THE","AND","OF","DATE","NEW","DELHI","BY","DELIVERED"}

def back_name(text, end, max_toks=4):
    ws = max(0, end - 80)
    toks = [(m.start() + ws, m.end() + ws) for m in TOK_RE.finditer(text[ws:end])]
    run = []
    for a, b in reversed(toks):
        nxt = run[0][0] if run else end
        if text[b:nxt].strip(" \t") != "": break
        if text[a:b].upper().strip(".") in STOP: break
        run.insert(0, (a, b))
        if len(run) == max_toks: break
    return (run[0][0], run[-1][1]) if run else None

def trim(text, s, e):
    while s < e and (text[s].isspace() or text[s] in ".,;:"): s += 1
    while e > s and (text[e-1].isspace() or text[e-1] in ",;:"): e -= 1
    return (s, e) if e > s else None

A_RE = re.compile(rf"\bJustice[ \t]+({NAME})")
B_RE = re.compile(r",[ \t]*J{1,2}\b\.?")
C_RE = re.compile(r"\bJJ?\.?[ \t]*\(([^()\n]{3,45})\)")
D_RE = re.compile(r"(?:BENCH\s*:|\(J\)|\(J\.\))\s*([A-Z][A-Za-z']+,[ \t]*[A-Z](?:[ \t]*\.?[ \t]*[A-Z])*\.?)[ \t]*\(J\)")

def judge_patterns(text):
    out = {"A Justice NAME": set(), "B NAME, J.": set(), "C J.(NAME)": set(), "D BENCH: X, Y (J)": set()}
    for m in A_RE.finditer(text):
        sp = trim(text, m.start(1), m.end(1)); sp and out["A Justice NAME"].add(sp)
    for m in B_RE.finditer(text):
        bn = back_name(text, m.start())
        if bn:
            sp = trim(text, *bn); sp and out["B NAME, J."].add(sp)
    for m in C_RE.finditer(text):
        inner = m.group(1)
        if re.fullmatch(r"[A-Za-z\.' ,\-]+", inner) and inner.strip().upper() not in {"J", "JJ"}:
            sp = trim(text, m.start(1), m.end(1)); sp and out["C J.(NAME)"].add(sp)
    for m in D_RE.finditer(text):
        sp = trim(text, m.start(1), m.end(1)); sp and out["D BENCH: X, Y (J)"].add(sp)
    return out

def eval_judge(texts, gold, show=4):
    names = list(judge_patterns("x").keys())
    G = {(d, int(e[0]), int(e[1])) for d, ents in gold.items() for e in ents if e[2] == "JUDGE"}
    P = {n: set() for n in names}
    for d, text in texts.items():
        for n, sp in judge_patterns(text).items():
            P[n] |= {(d, s, e) for s, e in sp}
    P["UNION"] = set().union(*[P[n] for n in names])
    print(f"gold JUDGE spans: {len(G)}\n{'pattern':22s}{'pred':>6}{'TP':>5}{'P':>7}{'R':>7}{'F1':>7}")
    for n in names + ["UNION"]:
        tp = len(G & P[n]); p = tp / max(len(P[n]), 1); r = tp / max(len(G), 1)
        print(f"{n:22s}{len(P[n]):>6}{tp:>5}{p:>7.3f}{r:>7.3f}{(2*p*r/(p+r) if p+r else 0):>7.3f}")
    cov = sum(1 for (d, s, e) in G if any(pd_ == d and ps < e and pe > s for pd_, ps, pe in P["UNION"]))
    print(f"\nboundary-agnostic coverage of gold by union: {cov}/{len(G)} = {cov/len(G):.2f}  "
          f"(exact: {len(G & P['UNION'])/len(G):.2f})")
    for n in names:
        fps = sorted(P[n] - G)
        print(f"\n--- {n}: false positives (showing {min(show, len(fps))} of {len(fps)}) ---")
        for d, s, e in random.sample(fps, min(show, len(fps))):
            t = texts[d]; print(f"   ...{t[max(0,s-50):s]!r} [[{t[s:e]!r}]] {t[e:e+25]!r}")
    miss = sorted(G - P["UNION"])
    print(f"\n--- missed gold JUDGE (showing {min(8, len(miss))} of {len(miss)}) ---")
    for d, s, e in random.sample(miss, min(8, len(miss))):
        t = texts[d]; print(f"   ...{t[max(0,s-50):s]!r} [[{t[s:e]!r}]] {t[e:e+25]!r}")

eval_judge(train_texts, train_gold)   # FOLD 1 ONLY

In [ ]:
import re, random
random.seed(4)

CAP   = r"[A-ZÀ-Ö][A-Za-zÀ-ÖØ-öø-ÿ'’\-]*"
INIT  = r"[A-Z]\.(?:[ \t]*[A-Z]\.)*"
WORD  = rf"(?:{INIT}|{CAP})"
NAME2 = rf"{WORD}(?:[ \t]+{WORD}){{0,3}}"
TOK_RE = re.compile(WORD)                      # back_name() reads this global

AJ_RE = re.compile(rf"(?i:\bjustice)[ \t]+({NAME2})")
B_RE  = re.compile(r",[ \t]*J{1,2}\b\.?")
B2_RE = re.compile(r"[ \t]J{1,2}\.?(?=[ \t]*(?:in\b|speaking|writing|observed|held\b|was\b|said\b|while\b|who\b|delivered|dissent|agreed|also\b|noted|stated))")
C_RE  = re.compile(r"\bJJ?\.?[ \t]*\(([^()\n]{3,45})\)")
D_RE  = re.compile(r"(?:BENCH\s*:|\(J\)|\(J\.\))\s*([A-Z][A-Za-z']+,[ \t]*[A-Z](?:[ \t]*\.?[ \t]*[A-Z])*\.?)[ \t]*\(J\)")
E_RE  = re.compile(r"\bJJ?\.?[ \t]*\[[ \t]*([A-Za-zÀ-ÿ\.'’ \t\-]{3,45}?)[ \t]*\]")
F_RE  = re.compile(r"BENCH[ \t]*:[ \t]*([^\n]{3,120}?)(?=[ \t]*(?:JUDG(?:E)?MENT|CITATION|ACT\b|\n|$))")
NAMES = ["A Justice NAME", "B NAME, J.", "B2 NAME J (no comma)", "C J.(NAME)", "D BENCH: X, Y (J)", "E J [NAME]", "F BENCH: A & B"]

def judge_patterns2(text):
    out = {n: set() for n in NAMES}
    def add(n, s, e):
        sp = trim(text, s, e)
        if sp: out[n].add(sp)
    for m in AJ_RE.finditer(text): add(NAMES[0], m.start(1), m.end(1))
    for rx, n in [(B_RE, NAMES[1]), (B2_RE, NAMES[2])]:
        for m in rx.finditer(text):
            bn = back_name(text, m.start())
            if bn: add(n, *bn)
    for rx, n in [(C_RE, NAMES[3]), (E_RE, NAMES[5])]:
        for m in rx.finditer(text):
            inner = m.group(1).strip()
            if inner and inner[0].isupper() and inner.upper() not in {"J", "JJ"} and re.fullmatch(r"[A-Za-zÀ-ÿ\.'’, \-]+", inner):
                add(n, m.start(1), m.end(1))
    for m in D_RE.finditer(text): add(NAMES[4], m.start(1), m.end(1))
    for m in F_RE.finditer(text):
        seg, base, pos = m.group(1), m.start(1), 0
        for part in re.split(r"[ \t]*(?:&|\bAND\b)[ \t]*", seg):
            k = seg.find(part, pos)
            if k < 0: continue
            pos = k + len(part)
            part2 = re.sub(r"[ \t]*\((?:C?J|J\.?)\)\s*$", "", part)
            sp = trim(text, base + k, base + k + len(part2))
            if sp and re.fullmatch(NAME2, text[sp[0]:sp[1]]): out[NAMES[6]].add(sp)
    return out

def resolve(spans):
    out = []
    for s, e in sorted(set(spans), key=lambda x: (x[0], -(x[1] - x[0]))):
        if out and s < out[-1][1]: continue
        out.append((s, e))
    return out

# ---- 1. FOLD 1: per-pattern stats and pre-declared selection (P >= 0.4 and TP >= 5) ----
G = {(d, int(e[0]), int(e[1])) for d, ents in train_gold.items() for e in ents if e[2] == "JUDGE"}
Pn = {n: set() for n in NAMES}
for d, text in train_texts.items():
    for n, sp in judge_patterns2(text).items(): Pn[n] |= {(d, s, e) for s, e in sp}
print(f"gold JUDGE (fold 1): {len(G)}\n{'pattern':24s}{'pred':>6}{'TP':>5}{'P':>7}{'R':>7}")
keep = []
for n in NAMES:
    tp = len(G & Pn[n]); p = tp / max(len(Pn[n]), 1)
    print(f"{n:24s}{len(Pn[n]):>6}{tp:>5}{p:>7.3f}{tp / len(G):>7.3f}")
    if p >= 0.4 and tp >= 5: keep.append(n)
print("kept patterns:", keep)

def judge_spans_doc(text, keep):
    allsp = set()
    for n, sp in judge_patterns2(text).items():
        if n in keep: allsp |= sp
    return resolve(allsp)

U1 = {(d, s, e) for d, text in train_texts.items() for s, e in judge_spans_doc(text, keep)}
tp = len(G & U1); p = tp / max(len(U1), 1); r = tp / len(G)
print(f"fold 1 union of kept (overlaps resolved): pred={len(U1)} TP={tp} P={p:.3f} R={r:.3f} F1={2*p*r/(p+r):.3f}")

# ---- 2. FOLD 2, once: champion + counsel rules + JUDGE rules (champion JUDGE spans replaced) ----
assert "c_filt" in globals(), "re-run the earlier fold-2 cell to define c_filt"
def merge_all(texts, champ_filtered, keep):
    out = {}
    for d, text in texts.items():
        rules = list(counsel_spans(text, chosen)) + [(s, e, "JUDGE") for s, e in judge_spans_doc(text, keep)]
        kept = [k for k in champ_filtered[d] if k[2] not in (A_, R_, "JUDGE")]
        kept = [k for k in kept if not any(k[0] < re_ and k[1] > rs for rs, re_, _ in rules)]
        out[d] = sorted(set(kept + rules))
    return out

base_macro, base_f1 = score(as_scored(merge_policy(val_texts, c_filt, chosen)), {})
new_macro, new_f1 = score(as_scored(merge_all(val_texts, c_filt, keep)), {})
print(f"\nFOLD 2: champion {c_macro:.4f} | +counsel {base_macro:.4f} | +counsel+JUDGE {new_macro:.4f}  (JUDGE step: {new_macro - base_macro:+.4f})")
print(pd.DataFrame({"champ+counsel": base_f1, "champ+counsel+judge": new_f1}).round(3).to_string())

In [ ]:
import json, os
from collections import Counter

FOLD3 = "/kaggle/working/cs781_backup_download/phase2_lner/Phase_2_LNER/Test/fold3-text.json"
NEW_PATH = "/kaggle/working/prediction_champion_plus_rules.json"
assert not os.path.exists(NEW_PATH), "refusing to overwrite an existing file"

with open(FOLD3, encoding="utf-8") as f: test_texts = json.load(f)
assert isinstance(test_texts, dict), f"fold3 file is {type(test_texts)}"
print("fold 3 docs:", len(test_texts))

champion_model.eval()
test_filt = apply_thr(predict_texts(champion_model, test_texts), champ_thr)
test_merged = merge_all(test_texts, test_filt, keep)
new_obj = {str(d): [[int(s), int(e), l] for s, e, l in sp] for d, sp in test_merged.items()}

def validate(obj, texts, name):
    T = {str(k): v for k, v in texts.items()}
    issues, labs = Counter(), Counter()
    for d, sp in obj.items():
        text = T.get(str(d))
        if text is None: issues["doc not in fold3"] += 1; continue
        for x in sp:
            if len(x) != 3: issues["span not length 3"] += 1; continue
            s, e, l = x
            if l not in ENT: issues["unknown label"] += 1
            if not (isinstance(s, int) and isinstance(e, int) and 0 <= s < e <= len(text)):
                issues["bad offsets"] += 1; continue
            if text[s:e] != text[s:e].strip(): issues["whitespace edge"] += 1
            labs[l] += 1
        ss = sorted((x[0], x[1]) for x in sp if len(x) == 3)
        issues["overlapping pairs"] += sum(1 for a, b in zip(ss, ss[1:]) if b[0] < a[1])
    issues["missing docs"] = len(set(T) - set(map(str, obj)))
    issues = {k: v for k, v in issues.items() if v}
    print(f"\n[{name}] docs={len(obj)} spans={sum(labs.values())} issues={issues or 'none'}")
    print("  per-label:", dict(sorted(labs.items())))
    return issues

old_path = "/kaggle/working/prediction.json"
if os.path.exists(old_path):
    with open(old_path, encoding="utf-8") as f: old = json.load(f)
    print("existing prediction.json: top-level", type(old).__name__, "len", len(old))
    if isinstance(old, dict) and old:
        k = next(iter(old)); print("  sample key:", repr(k), "| sample value:", old[k][:3])
        validate(old, test_texts, "existing prediction.json (read-only)")

issues = validate(new_obj, test_texts, "NEW candidate")
fold2_merged = merge_all(val_texts, c_filt, keep)
print(f"\nspans/doc: fold 2 merged = {sum(map(len, fold2_merged.values()))/len(val_texts):.1f}, "
      f"fold 3 merged = {sum(map(len, new_obj.values()))/len(test_texts):.1f}")

hard = {"bad offsets", "unknown label", "span not length 3", "missing docs", "doc not in fold3"}
if hard & set(issues):
    print("\nNOT WRITTEN: hard format errors:", hard & set(issues))
else:
    with open(NEW_PATH, "x", encoding="utf-8") as f: json.dump(new_obj, f)
    print("\nWrote:", NEW_PATH)

In [ ]:

# CS781 — Priority 1 + 2 diagnostic
# Read-only: does not change thresholds, model weights, or prediction files.

import pandas as pd
from collections import Counter, defaultdict

_required = [
    "champion_model", "predict_texts", "apply_thr", "champ_thr",
    "val_texts", "val_gold", "ENT"
]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError("Missing notebook definitions: " + ", ".join(_missing))

TARGETS = ["RESP", "WIT", "PREC", "AUTH", "CASENO"]

def _span_set(obj):
    return {
        (str(d), int(x[0]), int(x[1]), str(x[2]))
        for d, ents in obj.items()
        for x in ents
        if len(x) >= 3
    }

def _label_report(pred, gold):
    ps, gs = _span_set(pred), _span_set(gold)
    rows = []
    for lab in ENT:
        g = {x for x in gs if x[3] == lab}
        p = {x for x in ps if x[3] == lab}
        tp = len(g & p)
        fp = len(p - g)
        fn = len(g - p)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows.append({
            "label": lab, "gold": len(g), "predicted": len(p),
            "TP": tp, "FP": fp, "FN": fn, "precision": precision,
            "recall": recall, "F1": f1
        })
    return pd.DataFrame(rows).set_index("label")

def _show_span(doc_id, s, e, label, texts, heading, limit_state):
    if limit_state[0] >= limit_state[1]:
        return
    text = texts.get(doc_id, texts.get(str(doc_id)))
    if text is None:
        return
    lo, hi = max(0, s - 90), min(len(text), e + 90)
    snippet = text[lo:s] + " ⟦" + text[s:e] + "⟧ " + text[e:hi]
    print(f"{heading} | doc={doc_id} | {label} | [{s}:{e}] {snippet!r}")
    limit_state[0] += 1

champion_model.eval()

# 1. Decode the champion's spans BEFORE thresholding.
raw_val = predict_texts(champion_model, val_texts)
filtered_val = apply_thr(raw_val, champ_thr)

raw_counts = Counter(
    str(x[2]) for spans in raw_val.values() for x in spans
    if len(x) >= 4
)
filtered_counts = Counter(
    str(x[2]) for spans in filtered_val.values() for x in spans
    if len(x) >= 3
)
gold_counts = Counter(
    str(x[2]) for ents in val_gold.values() for x in ents
    if len(x) >= 3
)

print("=" * 85)
print("PRIORITY 1 — RESP / WIT RAW-DECODE AND THRESHOLD CHECK")
print("=" * 85)
print("Raw decoded counts:", {l: raw_counts[l] for l in ["RESP", "WIT"]})
print("After thresholds:", {l: filtered_counts[l] for l in ["RESP", "WIT"]})
print("Fold-2 gold counts:", {l: gold_counts[l] for l in ["RESP", "WIT"]})
print("Current thresholds:", {l: champ_thr.get(l) for l in ["RESP", "WIT"]})

print("\nRaw decoded confidence summary:")
for lab in ["RESP", "WIT"]:
    confs = [
        float(x[3])
        for spans in raw_val.values()
        for x in spans
        if len(x) >= 4 and str(x[2]) == lab
    ]
    print(
        lab,
        "count =", len(confs),
        "max =", round(max(confs), 4) if confs else None,
        "median =", round(float(pd.Series(confs).median()), 4) if confs else None
    )

if not raw_counts["RESP"] and not raw_counts["WIT"]:
    print(
        "\nInterpretation: neither label appears in decoded spans before "
        "thresholding. Lowering their thresholds alone cannot recover spans "
        "that the decoder never emitted. Next we must inspect the model's "
        "token-level label probabilities and BIO decoding."
    )
elif not filtered_counts["RESP"] or not filtered_counts["WIT"]:
    print(
        "\nInterpretation: at least one label is decoded before thresholding "
        "but absent afterwards. A controlled threshold sweep may be useful."
    )

# 2. Per-label exact-span errors from the established champion pipeline.
report = _label_report(filtered_val, val_gold)
print("\n" + "=" * 85)
print("PRIORITY 2 — CHAMPION LABEL-WISE ERROR REPORT")
print("=" * 85)
display(report.loc[TARGETS].round(4))

# 3. For PREC/AUTH/CASENO, show false positives and false negatives.
gold_set = _span_set(val_gold)
pred_set = _span_set(filtered_val)

print("\n" + "=" * 85)
print("PREC / AUTH / CASENO — REPRESENTATIVE ERRORS")
print("=" * 85)

for lab in ["PREC", "AUTH", "CASENO"]:
    g = {x for x in gold_set if x[3] == lab}
    p = {x for x in pred_set if x[3] == lab}
    fp = sorted(p - g)
    fn = sorted(g - p)

    print(f"\n--- {lab}: {len(fp)} false positives, {len(fn)} false negatives ---")
    limit = [0, 8]
    for d, s, e, l in fp:
        _show_span(d, s, e, l, val_texts, "FP", limit)
    limit = [0, 8]
    for d, s, e, l in fn:
        _show_span(d, s, e, l, val_texts, "FN", limit)

# 4. Check whether missed target spans were predicted at the same offsets
# under a different label (useful for label confusion analysis).
print("\n" + "=" * 85)
print("WRONG-LABEL OVERLAP CHECK")
print("=" * 85)

for lab in TARGETS:
    missed = [x for x in gold_set if x[3] == lab and x not in pred_set]
    overlaps = Counter()
    examples = []
    for d, s, e, gold_lab in missed:
        alternatives = [
            x for x in pred_set
            if x[0] == d and x[3] != gold_lab
            and x[1] < e and x[2] > s
        ]
        if alternatives:
            for x in alternatives:
                overlaps[x[3]] += 1
            if len(examples) < 4:
                examples.append(((d, s, e, gold_lab), alternatives[:3]))

    print(f"\n{lab}: missed gold spans={len(missed)}")
    print("Overlapping predictions under other labels:", dict(overlaps))
    for gold_span, alternatives in examples:
        print(" gold:", gold_span, "| overlapping predictions:", alternatives)

print("\nDiagnostic complete. No model weights, thresholds, or prediction files changed.")


In [ ]:

import numpy as np
import pandas as pd
import torch

_required = [
    "champion_model", "predict_texts", "apply_thr", "champ_thr",
    "val_texts", "val_gold", "diag_doc_probs", "BIO_LABELS",
    "bio_viterbi_decode"
]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError("Missing definitions: " + ", ".join(_missing))

def _exact_label_stats(predictions, gold, label):
    pred_set = {
        (str(d), int(s), int(e))
        for d, ents in predictions.items()
        for s, e, lab in ents
        if lab == label
    }
    gold_set = {
        (str(d), int(x[0]), int(x[1]))
        for d, ents in gold.items()
        for x in ents
        if len(x) >= 3 and x[2] == label
    }
    tp = len(pred_set & gold_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return {
        "label": label, "gold": len(gold_set), "pred": len(pred_set),
        "TP": tp, "FP": fp, "FN": fn,
        "precision": p, "recall": r, "F1": f
    }

# ------------------------------------------------------------
# A. RESP: test thresholds against the existing raw predictions.
# ------------------------------------------------------------
champion_model.eval()
raw_val_sweep = predict_texts(champion_model, val_texts)

sweep_rows = []
for threshold in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.45]:
    test_thresholds = dict(champ_thr)
    test_thresholds["RESP"] = threshold

    filtered = apply_thr(raw_val_sweep, test_thresholds)
    stats = _exact_label_stats(filtered, val_gold, "RESP")
    stats["threshold"] = threshold
    sweep_rows.append(stats)

resp_sweep = pd.DataFrame(sweep_rows)[
    ["threshold", "gold", "pred", "TP", "FP", "FN",
     "precision", "recall", "F1"]
]
print("RESP threshold sweep — fold 2")
display(resp_sweep.round(4))

# ------------------------------------------------------------
# B. RESP/WIT: inspect model probabilities on gold entity tokens.
#    This distinguishes weak model evidence from decoder failure.
# ------------------------------------------------------------
bio_to_id = {lab: i for i, lab in enumerate(BIO_LABELS)}
target_labels = ["RESP", "WIT"]
token_rows = []

with torch.inference_mode():
    for doc_id, text in val_texts.items():
        offsets, probs = diag_doc_probs(champion_model, text)
        probs = np.asarray(
            probs.detach().cpu() if torch.is_tensor(probs) else probs
        )

        if probs.ndim != 2 or len(offsets) != len(probs):
            raise RuntimeError(
                f"Unexpected probability/offset shape in document {doc_id}: "
                f"{probs.shape}, {len(offsets)}"
            )

        decoded = bio_viterbi_decode(
            torch.log(torch.as_tensor(probs).clamp_min(1e-12))
        )
        if torch.is_tensor(decoded):
            decoded = decoded.detach().cpu().tolist()

        for ent in val_gold[doc_id]:
            if len(ent) < 3 or ent[2] not in target_labels:
                continue

            gs, ge, lab = int(ent[0]), int(ent[1]), ent[2]
            tok_idx = [
                i for i, (s, e) in enumerate(offsets)
                if e > s and s < ge and e > gs
            ]
            if not tok_idx:
                token_rows.append({
                    "doc_id": str(doc_id), "label": lab,
                    "gold_text": text[gs:ge],
                    "token_count": 0, "max_B_prob": np.nan,
                    "mean_B_prob": np.nan, "max_I_prob": np.nan,
                    "decoded_target_tokens": 0
                })
                continue

            b_id = bio_to_id.get("B-" + lab)
            i_id = bio_to_id.get("I-" + lab)

            b_probs = probs[tok_idx, b_id] if b_id is not None else np.zeros(len(tok_idx))
            i_probs = probs[tok_idx, i_id] if i_id is not None else np.zeros(len(tok_idx))

            decoded_target = sum(
                1 for i in tok_idx
                if str(BIO_LABELS[int(decoded[i])]).endswith(lab)
            )

            token_rows.append({
                "doc_id": str(doc_id),
                "label": lab,
                "gold_text": text[gs:ge],
                "token_count": len(tok_idx),
                "max_B_prob": float(np.max(b_probs)),
                "mean_B_prob": float(np.mean(b_probs)),
                "max_I_prob": float(np.max(i_probs)) if i_id is not None else 0.0,
                "decoded_target_tokens": decoded_target
            })

token_diag = pd.DataFrame(token_rows)

print("\nGold-token probability summary")
if not token_diag.empty:
    display(
        token_diag.groupby("label").agg(
            gold_spans=("gold_text", "size"),
            spans_with_tokens=("token_count", lambda x: int((x > 0).sum())),
            mean_max_B=("max_B_prob", "mean"),
            median_max_B=("max_B_prob", "median"),
            mean_max_I=("max_I_prob", "mean"),
            spans_with_decoded_target=("decoded_target_tokens",
                                       lambda x: int((x > 0).sum()))
        ).round(4)
    )

    print("\nRESP/WIT examples with strongest B-label probability")
    for lab in target_labels:
        sub = token_diag[token_diag["label"] == lab].sort_values(
            "max_B_prob", ascending=False
        )
        print(f"\n{lab}")
        display(sub.head(10))

print("\nDiagnostic complete. No thresholds, weights, or files were changed.")


In [ ]:

import torch
from collections import Counter

_required = [
    "train_gold", "train_texts", "BIO_LABELS",
    "champion_model", "ENT"
]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError("Missing definitions: " + ", ".join(_missing))

print("=" * 75)
print("1. TRAINING GOLD COUNTS")
print("=" * 75)

train_counts = Counter(
    ent[2]
    for ents in train_gold.values()
    for ent in ents
    if len(ent) >= 3
)
for label in ENT:
    print(f"{label:12s} {train_counts[label]:6d}")

print("\n" + "=" * 75)
print("2. BIO LABEL VOCABULARY")
print("=" * 75)

for i, label in enumerate(BIO_LABELS):
    print(f"{i:3d}  {label}")

for label in ["RESP", "WIT"]:
    print(
        f"\n{label}:",
        {
            "gold_train": train_counts[label],
            "B_label_present": "B-" + label in BIO_LABELS,
            "I_label_present": "I-" + label in BIO_LABELS,
            "B_id": BIO_LABELS.index("B-" + label)
                    if "B-" + label in BIO_LABELS else None,
            "I_id": BIO_LABELS.index("I-" + label)
                    if "I-" + label in BIO_LABELS else None,
        }
    )

print("\n" + "=" * 75)
print("3. MODEL OUTPUT DIMENSIONS")
print("=" * 75)

for name, module in champion_model.named_modules():
    if isinstance(module, torch.nn.Linear):
        print(f"{name or '<root>'}: in={module.in_features}, out={module.out_features}")

print("\n" + "=" * 75)
print("4. TRAINING DOCUMENTS WITH TARGET LABELS")
print("=" * 75)

for label in ["RESP", "WIT"]:
    docs = []
    for doc_id, entities in train_gold.items():
        matches = [
            ent for ent in entities
            if len(ent) >= 3 and ent[2] == label
        ]
        if matches:
            docs.append((str(doc_id), len(matches), matches[:3]))

    print(f"\n{label}: {len(docs)} training documents contain this label")
    for row in docs[:10]:
        print(row)

print("\nRead-only diagnostic complete.")


In [ ]:

import pandas as pd

_required = ["train_texts", "train_gold", "val_texts", "val_gold", "tokenizer"]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError("Missing definitions: " + ", ".join(_missing))

def _get_offsets(text):
    # Transformers PreTrainedTokenizerFast
    if callable(tokenizer):
        enc = tokenizer(
            text,
            return_offsets_mapping=True,
            truncation=False,
            add_special_tokens=True
        )
        offsets = enc["offset_mapping"]
        if hasattr(offsets, "tolist"):
            offsets = offsets.tolist()
        if offsets and isinstance(offsets[0], list) and len(offsets[0]) == 2:
            return [(int(s), int(e)) for s, e in offsets]

    # Hugging Face tokenizers.Tokenizer
    if hasattr(tokenizer, "encode"):
        enc = tokenizer.encode(text, add_special_tokens=True)
        return [(int(s), int(e)) for s, e in enc.offsets]

    raise TypeError(
        f"Unsupported tokenizer type: {type(tokenizer)}"
    )

def _audit_alignment(texts, gold, split_name):
    rows = []
    for doc_id, entities in gold.items():
        text = texts[doc_id]
        offsets = _get_offsets(text)
        usable = [(s, e) for s, e in offsets if e > s]

        for ent in entities:
            if len(ent) < 3 or ent[2] not in ("RESP", "WIT"):
                continue

            gs, ge, label = int(ent[0]), int(ent[1]), ent[2]
            covered = [(s, e) for s, e in usable if s < ge and e > gs]
            start_aligned = any(s == gs for s, e in covered)
            end_aligned = any(e == ge for s, e in covered)

            rows.append({
                "split": split_name,
                "doc_id": str(doc_id),
                "label": label,
                "gold_text": text[gs:ge],
                "token_overlap_count": len(covered),
                "start_boundary_aligned": start_aligned,
                "end_boundary_aligned": end_aligned,
                "fully_boundary_aligned": start_aligned and end_aligned,
                "no_token_overlap": len(covered) == 0
            })
    return rows

alignment_df = pd.DataFrame(
    _audit_alignment(train_texts, train_gold, "train")
    + _audit_alignment(val_texts, val_gold, "validation")
)

print("Tokenizer type:", type(tokenizer))
print("\nGold-span token alignment summary:")
display(
    alignment_df.groupby(["split", "label"]).agg(
        spans=("gold_text", "size"),
        no_token_overlap=("no_token_overlap", "sum"),
        start_aligned=("start_boundary_aligned", "sum"),
        end_aligned=("end_boundary_aligned", "sum"),
        fully_aligned=("fully_boundary_aligned", "sum")
    )
)

print("\nExamples with incomplete boundaries:")
display(
    alignment_df[
        ~alignment_df["fully_boundary_aligned"]
    ].groupby(["split", "label"], group_keys=False).head(8)
)

print("\nRead-only audit complete.")


In [ ]:

from collections import Counter
import pandas as pd

# Uses the existing tokenizer and annotations; changes nothing.
_required = ["tokenizer", "train_texts", "train_gold"]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError("Missing definitions: " + ", ".join(_missing))

rows = []
conflicts = Counter()

for doc_id, text in train_texts.items():
    entities = [
        (int(ent[0]), int(ent[1]), str(ent[2]))
        for ent in train_gold[doc_id]
        if len(ent) >= 3 and ent[2] in ("RESP", "WIT")
    ]

    # Same low-level BPE tokenizer API as the notebook's training code.
    enc = tokenizer.encode(text, add_special_tokens=False)
    offsets = [(int(s), int(e)) for s, e in enc.offsets]

    for i, (ts, te) in enumerate(offsets):
        if te <= ts:
            continue

        overlapping = [
            (s, e, lab) for s, e, lab in entities
            if s < te and e > ts
        ]

        labels = {lab for _, _, lab in overlapping}
        if len(overlapping) > 1:
            conflicts["tokens_overlapping_multiple_target_entities"] += 1
        if len(labels) > 1:
            conflicts["tokens_overlapping_both_RESP_and_WIT"] += 1

    for gs, ge, lab in entities:
        overlaps = [
            (i, ts, te) for i, (ts, te) in enumerate(offsets)
            if te > ts and ts < ge and te > gs
        ]
        rows.append({
            "doc_id": str(doc_id),
            "label": lab,
            "gold_text": text[gs:ge],
            "gold_start": gs,
            "gold_end": ge,
            "overlapping_tokens": len(overlaps),
            "first_token_start": overlaps[0][1] if overlaps else None,
            "first_token_end": overlaps[0][2] if overlaps else None,
            "last_token_start": overlaps[-1][1] if overlaps else None,
            "last_token_end": overlaps[-1][2] if overlaps else None,
            "first_token_starts_before_gold": (
                overlaps[0][1] < gs if overlaps else None
            ),
        })

audit = pd.DataFrame(rows)

print("Training RESP/WIT alignment summary:")
display(
    audit.groupby("label").agg(
        gold_spans=("gold_text", "size"),
        no_overlapping_tokens=("overlapping_tokens", lambda x: int((x == 0).sum())),
        mean_tokens_per_span=("overlapping_tokens", "mean"),
        first_token_starts_before_gold=(
            "first_token_starts_before_gold",
            lambda x: int((x == True).sum())
        )
    ).round(3)
)

print("\nOverlap-conflict diagnostics:", dict(conflicts))

print("\nExamples where the first BPE token begins before the gold span:")
display(
    audit[audit["first_token_starts_before_gold"] == True]
    [["doc_id", "label", "gold_text", "gold_start", "first_token_start",
      "first_token_end", "gold_end"]]
    .head(15)
)

print("\nRead-only audit complete.")


In [ ]:

from collections import Counter, defaultdict

_required = ["pilot_windows", "BIO_LABELS", "label2id", "train_gold"]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError(
        "Missing pilot-window definitions: " + ", ".join(_missing)
        + ". Run the notebook's existing pilot window-building cell first."
    )

target_labels = ["RESP", "WIT"]
stored = Counter()
ignored = Counter()
examples = defaultdict(list)

for window in pilot_windows:
    labels = window["labels"]
    for pos, raw_id in enumerate(labels):
        idx = int(raw_id)
        if idx == -100:
            ignored["ignored_positions"] += 1
            continue
        if not 0 <= idx < len(BIO_LABELS):
            raise ValueError(f"Invalid label ID {idx}")

        tag = BIO_LABELS[idx]
        for lab in target_labels:
            if tag in (f"B-{lab}", f"I-{lab}"):
                stored[(lab, tag)] += 1
                if len(examples[lab]) < 5:
                    examples[lab].append({
                        "doc_id": window.get("doc_id"),
                        "window_start": window.get("start"),
                        "position": pos,
                        "tag": tag,
                        "token_id": window["input_ids"][pos],
                    })

print("Pilot-window target counts (these are NOT champion-training counts):")
for lab in target_labels:
    print(
        f"{lab}: B={stored[(lab, 'B-' + lab)]}, "
        f"I={stored[(lab, 'I-' + lab)]}, "
        f"total={stored[(lab, 'B-' + lab)] + stored[(lab, 'I-' + lab)]}"
    )
    for ex in examples[lab]:
        print(" ", ex)

print("\nIgnored label positions:", ignored["ignored_positions"])

print("\nGold training span counts:")
for lab in target_labels:
    print(lab, sum(
        1 for entities in train_gold.values()
        for ent in entities if len(ent) >= 3 and ent[2] == lab
    ))

print("\nNo training, model updates, or file writes performed.")


In [ ]:

import torch
from collections import Counter

_required = ["pilot_windows", "BIO_LABELS", "label2id"]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError("Missing definitions: " + ", ".join(_missing))

target_tags = {"B-RESP", "I-RESP", "B-WIT", "I-WIT"}
tag_counts = Counter()
ignored_by_tag = Counter()
window_counts = Counter()

for w in pilot_windows:
    labels = w["labels"]
    window_counts["windows"] += 1

    for value in labels:
        idx = int(value)
        if idx == -100:
            window_counts["ignored_positions"] += 1
            continue

        tag = BIO_LABELS[idx]
        tag_counts[tag] += 1

        if tag in target_tags:
            window_counts["target_supervised_positions"] += 1

print("Supervised target-label counts:")
for tag in sorted(target_tags):
    print(f"{tag:10s}: {tag_counts[tag]}")

print("\nWindow statistics:")
for key, value in window_counts.items():
    print(f"{key}: {value}")

print("\nLabel ID consistency:")
for tag in sorted(target_tags):
    idx = label2id.get(tag)
    print(
        f"{tag}: label2id={idx}, "
        f"BIO_LABELS index={BIO_LABELS.index(tag)}"
    )
    assert idx == BIO_LABELS.index(tag), f"ID mismatch for {tag}"

print("\nRead-only check complete. No model updates performed.")


In [ ]:

import os
import torch

print("=== Training configuration ===")
print("Pilot windows:", len(pilot_windows))
print("Optimizer:", type(targeted_optimizer).__name__)
print("Learning rate:", targeted_optimizer.param_groups[0]["lr"])
print("Loss:", type(targeted_criterion).__name__)
print("Ignore index:", targeted_criterion.ignore_index)
print("Class weights:")
for tag in ["O", "B-RESP", "I-RESP", "B-WIT", "I-WIT"]:
    idx = label2id[tag]
    print(f"  {tag:10s} {targeted_criterion.weight[idx].item():.2f}")

print("\n=== Existing pilot checkpoint ===")
pilot_path = "/kaggle/working/phase2_ner_targeted_pilot/targeted_pilot_8_steps.pt"
print("Exists:", os.path.isfile(pilot_path))
if os.path.isfile(pilot_path):
    ckpt = torch.load(pilot_path, map_location="cpu", weights_only=False)
    print("Optimizer steps:", ckpt.get("optimizer_steps"))
    print("Loss history:", ckpt.get("loss_history"))
    print("Label mapping matches:", ckpt.get("bio_labels") == BIO_LABELS)

print("\nChampion remains separate:", champion_model is not targeted_model)


In [ ]:

import inspect

print("evaluate_model signature:")
print(inspect.signature(evaluate_model))

print("\nFunction source:")
try:
    print(inspect.getsource(evaluate_model))
except (OSError, TypeError) as e:
    print("Source unavailable:", e)
    print("Return annotation:", getattr(evaluate_model, "__annotations__", {}))

print("\nExisting evaluation result variables:")
for name in [
    "champ_metrics", "champ_predictions",
    "pilot_metrics", "pilot_predictions"
]:
    if name in globals():
        obj = globals()[name]
        print(name, "type =", type(obj).__name__)
        if isinstance(obj, (tuple, list, dict)):
            print("  length/keys =", len(obj) if not isinstance(obj, dict)
                  else list(obj.keys())[:20])


In [ ]:

# Corrected evaluation: matches the actual evaluate_model() return format.
# Read-only; does not modify weights, thresholds, or prediction files.

_required = [
    "evaluate_model", "champion_model", "targeted_model",
    "val_texts", "val_gold", "champ_thr", "ENTITY_LABELS",
    "pilot_metrics"
]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError("Missing definitions: " + ", ".join(_missing))

champion_model.eval()
targeted_model.eval()

print("Evaluating champion on fold 2...")
champ_metrics, champ_raw_predictions, champ_filtered_predictions = evaluate_model(
    champion_model, val_texts, val_gold, champ_thr
)

print("\nEvaluating pilot on fold 2...")
pilot_metrics_new, pilot_raw_predictions, pilot_filtered_predictions = evaluate_model(
    targeted_model, val_texts, val_gold, champ_thr
)

# Use the metrics returned by these two evaluations for a consistent comparison.
rows = []
for label in ENTITY_LABELS:
    c = champ_metrics[label]
    p = pilot_metrics_new[label]
    rows.append({
        "label": label,
        "champion_F1": c["f1"],
        "pilot_F1": p["f1"],
        "F1_delta": p["f1"] - c["f1"],
        "champion_precision": c["precision"],
        "pilot_precision": p["precision"],
        "champion_recall": c["recall"],
        "pilot_recall": p["recall"],
        "champion_predicted": c["pred"],
        "pilot_predicted": p["pred"],
        "predicted_delta": p["pred"] - c["pred"],
    })

comparison = pd.DataFrame(rows).set_index("label")

print("\nPer-label fold-2 comparison:")
display(comparison.round(4))

champ_macro = float(champ_metrics["MACRO_F1"])
pilot_macro = float(pilot_metrics_new["MACRO_F1"])

print(f"Champion macro-F1: {champ_macro:.6f}")
print(f"Pilot macro-F1:    {pilot_macro:.6f}")
print(f"Macro-F1 delta:    {pilot_macro - champ_macro:+.6f}")

print("\nPriority labels:")
display(comparison.loc[["RESP", "WIT", "PREC", "AUTH", "CASENO"]].round(4))


In [ ]:

resp_delta = float(comparison.loc["RESP", "F1_delta"])
wit_delta = float(comparison.loc["WIT", "F1_delta"])
macro_delta = float(pilot_metrics_new["MACRO_F1"] - champ_metrics["MACRO_F1"])

print(f"RESP F1 delta:  {resp_delta:+.6f}")
print(f"WIT F1 delta:   {wit_delta:+.6f}")
print(f"Macro-F1 delta: {macro_delta:+.6f}")

pilot_passes = (
    resp_delta > 0
    and wit_delta > 0
    and macro_delta >= 0
)

if pilot_passes:
    print(
        "\nPILOT PASSES the initial validation gate. "
        "Keep it as a candidate; preserve the current best submission."
    )
else:
    print(
        "\nPILOT DOES NOT PASS the initial validation gate. "
        "Keep the champion as the current best. "
        "Do not generate a replacement submission from this pilot."
    )


In [ ]:

# CS781 — RESP/WIT token-level diagnosis (champion only)
# Read-only: no training, threshold changes, or file writes.

import numpy as np
import pandas as pd
import torch

_required = [
    "champion_model", "val_texts", "val_gold",
    "diag_doc_probs", "bio_viterbi_decode", "BIO_LABELS"
]
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(
        "Missing notebook definitions: " + ", ".join(_missing)
    )

model = champion_model
model.eval()

target_labels = ["RESP", "WIT"]
bio_to_id = {label: i for i, label in enumerate(BIO_LABELS)}
rows = []

with torch.inference_mode():
    for doc_id, text in val_texts.items():
        offsets, probs = diag_doc_probs(model, text)
        probs = np.asarray(
            probs.detach().cpu() if torch.is_tensor(probs) else probs
        )

        if probs.ndim != 2 or len(offsets) != len(probs):
            raise RuntimeError(
                f"Unexpected output for document {doc_id}: "
                f"probabilities={probs.shape}, offsets={len(offsets)}"
            )

        decoded = bio_viterbi_decode(
            torch.log(torch.as_tensor(probs).clamp_min(1e-12))
        )
        if torch.is_tensor(decoded):
            decoded = decoded.detach().cpu().tolist()

        offsets = [(int(s), int(e)) for s, e in offsets]

        for ent in val_gold[doc_id]:
            if len(ent) < 3 or ent[2] not in target_labels:
                continue

            gs, ge, lab = int(ent[0]), int(ent[1]), str(ent[2])
            token_ids = [
                i for i, (s, e) in enumerate(offsets)
                if e > s and s < ge and e > gs
            ]

            if not token_ids:
                rows.append({
                    "doc_id": str(doc_id),
                    "label": lab,
                    "gold_text": text[gs:ge],
                    "token_count": 0,
                    "max_B_prob": np.nan,
                    "max_I_prob": np.nan,
                    "max_O_prob": np.nan,
                    "decoded_tags": "<no overlapping tokens>",
                    "token_evidence": ""
                })
                continue

            b_id = bio_to_id["B-" + lab]
            i_id = bio_to_id["I-" + lab]
            o_id = bio_to_id["O"]

            token_evidence = []
            decoded_tags = []

            for i in token_ids:
                s, e = offsets[i]
                tag = BIO_LABELS[int(decoded[i])]
                decoded_tags.append(tag)
                token_evidence.append(
                    f"{text[s:e]!r}: "
                    f"B={probs[i, b_id]:.3f}, "
                    f"I={probs[i, i_id]:.3f}, "
                    f"O={probs[i, o_id]:.3f}, "
                    f"decoded={tag}"
                )

            rows.append({
                "doc_id": str(doc_id),
                "label": lab,
                "gold_text": text[gs:ge],
                "token_count": len(token_ids),
                "max_B_prob": float(probs[token_ids, b_id].max()),
                "max_I_prob": float(probs[token_ids, i_id].max()),
                "max_O_prob": float(probs[token_ids, o_id].max()),
                "decoded_tags": " ".join(decoded_tags),
                "token_evidence": " | ".join(token_evidence)
            })

diag_df = pd.DataFrame(rows)

print("=== Gold-span probability summary ===")
display(
    diag_df.groupby("label").agg(
        gold_spans=("gold_text", "size"),
        spans_with_tokens=("token_count", lambda x: int((x > 0).sum())),
        mean_max_B=("max_B_prob", "mean"),
        median_max_B=("max_B_prob", "median"),
        mean_max_I=("max_I_prob", "mean"),
        mean_max_O=("max_O_prob", "mean")
    ).round(4)
)

for lab in target_labels:
    subset = diag_df[diag_df["label"] == lab]
    print(f"\n=== {lab}: strongest B-tag evidence ===")
    display(
        subset.sort_values("max_B_prob", ascending=False)
        [["doc_id", "gold_text", "max_B_prob", "max_I_prob",
          "max_O_prob", "decoded_tags", "token_evidence"]]
        .head(8)
    )

    print(f"\n=== {lab}: weakest B-tag evidence ===")
    display(
        subset.sort_values("max_B_prob", ascending=True)
        [["doc_id", "gold_text", "max_B_prob", "max_I_prob",
          "max_O_prob", "decoded_tags", "token_evidence"]]
        .head(8)
    )

print("\nCompleted. No model weights, thresholds, or files changed.")


In [ ]:

# CS781 — RESP/WIT training-data inspection
# Read-only: uses fold 1 training annotations only.

from collections import Counter, defaultdict
import pandas as pd

_required = ["train_texts", "train_gold"]
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(
        "Missing notebook definitions: " + ", ".join(_missing)
    )

target_labels = ["RESP", "WIT"]
surface_stats = {lab: Counter() for lab in target_labels}
doc_stats = {lab: Counter() for lab in target_labels}
lengths = {lab: [] for lab in target_labels}
examples = {lab: [] for lab in target_labels}

for doc_id, text in train_texts.items():
    for ent in train_gold[doc_id]:
        if len(ent) < 3 or ent[2] not in target_labels:
            continue

        s, e, lab = int(ent[0]), int(ent[1]), str(ent[2])
        if not (0 <= s < e <= len(text)):
            print(
                "WARNING: invalid span:",
                doc_id, ent, "text_length=", len(text)
            )
            continue

        surface = text[s:e]
        surface_stats[lab][surface.strip().lower()] += 1
        doc_stats[lab][str(doc_id)] += 1
        lengths[lab].append(e - s)

        examples[lab].append({
            "doc_id": str(doc_id),
            "entity": surface,
            "length_chars": e - s,
            "left_context": text[max(0, s - 100):s].replace("\n", " "),
            "right_context": text[e:min(len(text), e + 100)].replace("\n", " ")
        })

print("=== Training distribution ===")
summary = []
for lab in target_labels:
    summary.append({
        "label": lab,
        "gold_spans": len(examples[lab]),
        "documents_with_label": len(doc_stats[lab]),
        "unique_normalized_surfaces": len(surface_stats[lab]),
        "mean_span_chars": (
            sum(lengths[lab]) / len(lengths[lab])
            if lengths[lab] else 0
        ),
        "median_span_chars": (
            pd.Series(lengths[lab]).median()
            if lengths[lab] else 0
        ),
        "largest_document_share": (
            max(doc_stats[lab].values()) / len(examples[lab])
            if examples[lab] else 0
        )
    })

display(pd.DataFrame(summary).round(3))

for lab in target_labels:
    print(f"\n=== {lab}: most frequent normalized surface forms ===")
    display(pd.DataFrame(
        surface_stats[lab].most_common(20),
        columns=["normalized_surface", "count"]
    ))

    print(f"\n=== {lab}: documents with the most examples ===")
    display(pd.DataFrame(
        doc_stats[lab].most_common(12),
        columns=["doc_id", "gold_span_count"]
    ))

    print(f"\n=== {lab}: representative training contexts ===")
    # Deterministic sample: first 12 annotated examples in notebook order.
    display(pd.DataFrame(examples[lab]).head(12))

print("\nCompleted. Training annotations and model are unchanged.")


In [ ]:

# CS781: Read-only token-type confusion analysis
# Does not train, modify model weights, or write prediction files.

import torch

assert "diag_doc_probs" in globals(), "Run the cell defining diag_doc_probs first."
assert "champion_model" in globals()
assert "train_texts" in globals() and "train_gold" in globals()
assert "val_texts" in globals() and "val_gold" in globals()
assert "BIO_LABELS" in globals() and "label2id" in globals()

ENTITY_TYPES = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
]
ALL_TYPES = ["O"] + ENTITY_TYPES
TARGET_TYPES = ["RESP", "WIT"]

# Aggregate B/I probability into the corresponding entity type.
bio_to_type = torch.tensor([
    0 if tag == "O" else ALL_TYPES.index(tag[2:])
    for tag in BIO_LABELS
], dtype=torch.long)

aggregation = torch.zeros(len(BIO_LABELS), len(ALL_TYPES))
aggregation[
    torch.arange(len(BIO_LABELS)), bio_to_type
] = 1.0

def run_type_confusion(texts, gold, fold_name):
    probability_mass = {lab: [] for lab in TARGET_TYPES}
    argmax_types = {lab: [] for lab in TARGET_TYPES}

    champion_model.eval()

    with torch.inference_mode():
        for doc_id, text in texts.items():
            offsets, probs = diag_doc_probs(champion_model, text)
            probs = torch.as_tensor(probs).float().cpu()

            if probs.ndim != 2 or probs.shape[1] != len(BIO_LABELS):
                raise ValueError(
                    f"Unexpected probability shape for {doc_id}: "
                    f"{tuple(probs.shape)}"
                )

            type_probs = probs @ aggregation
            predicted_types = bio_to_type[probs.argmax(dim=1)]

            for ann in gold.get(doc_id, []):
                s, e, label = int(ann[0]), int(ann[1]), str(ann[2])
                if label not in TARGET_TYPES:
                    continue

                token_ids = [
                    i for i, (a, b) in enumerate(offsets)
                    if b > a and a < e and b > s
                ]
                if token_ids:
                    probability_mass[label].append(type_probs[token_ids])
                    argmax_types[label].append(predicted_types[token_ids])

    print(f"\n{'=' * 18} {fold_name} {'=' * 18}")

    for label in TARGET_TYPES:
        if not probability_mass[label]:
            print(f"\n{label}: no overlapping gold tokens found.")
            continue

        mass = torch.cat(probability_mass[label], dim=0)
        predicted = torch.cat(argmax_types[label], dim=0)

        mean_mass = mass.mean(dim=0)
        type_counts = torch.bincount(
            predicted, minlength=len(ALL_TYPES)
        ).float()
        type_frequency = type_counts / type_counts.sum().clamp_min(1)

        print(f"\nGold label: {label}; token count: {len(predicted)}")
        print("Mean probability mass by type:")
        for idx in torch.argsort(mean_mass, descending=True)[:8].tolist():
            print(f"  {ALL_TYPES[idx]:<12} {mean_mass[idx].item():.4f}")

        print("Predicted type frequency:")
        for idx in torch.argsort(type_frequency, descending=True)[:8].tolist():
            print(f"  {ALL_TYPES[idx]:<12} {type_frequency[idx].item():.2%}")

run_type_confusion(train_texts, train_gold, "FOLD 1 — TRAIN")
run_type_confusion(val_texts, val_gold, "FOLD 2 — VALIDATION")

print("\nRead-only analysis complete; champion weights remain unchanged.")


In [ ]:

# CS781: Compact train-vs-validation comparison for RESP and WIT
# Reuses the exact same inference helper and probability aggregation.

import torch
import pandas as pd

def target_learning_summary(texts, gold, fold_name):
    rows = []
    champion_model.eval()

    with torch.inference_mode():
        for doc_id, text in texts.items():
            offsets, probs = diag_doc_probs(champion_model, text)
            probs = torch.as_tensor(probs).float().cpu()

            for ann in gold.get(doc_id, []):
                s, e, label = int(ann[0]), int(ann[1]), str(ann[2])
                if label not in ("RESP", "WIT"):
                    continue

                ids = [
                    i for i, (a, b) in enumerate(offsets)
                    if b > a and a < e and b > s
                ]
                if not ids:
                    continue

                # Match the notebook's annotation convention:
                # first overlapping token = B-label, remaining = I-label.
                for j, token_id in enumerate(ids):
                    prefix = "B" if j == 0 else "I"
                    target_tag = f"{prefix}-{label}"
                    target_id = label2id[target_tag]

                    rows.append({
                        "fold": fold_name,
                        "label": label,
                        "position": prefix,
                        "gold_tag_probability": probs[token_id, target_id].item(),
                        "gold_tag_is_top1": (
                            int(probs[token_id].argmax().item() == target_id)
                        ),
                        "top1_tag": BIO_LABELS[
                            probs[token_id].argmax().item()
                        ],
                        "document": str(doc_id),
                        "gold_span": text[s:e]
                    })

    return pd.DataFrame(rows)

train_summary_df = target_learning_summary(
    train_texts, train_gold, "train"
)
val_summary_df = target_learning_summary(
    val_texts, val_gold, "validation"
)

combined_summary = pd.concat(
    [train_summary_df, val_summary_df], ignore_index=True
)

if combined_summary.empty:
    print("No RESP/WIT tokens were available for comparison.")
else:
    summary = (
        combined_summary
        .groupby(["fold", "label", "position"])
        .agg(
            tokens=("gold_tag_probability", "size"),
            mean_gold_probability=("gold_tag_probability", "mean"),
            median_gold_probability=("gold_tag_probability", "median"),
            gold_tag_top1_rate=("gold_tag_is_top1", "mean")
        )
        .reset_index()
    )

    summary["gold_tag_top1_rate"] *= 100
    display(summary.round(4))

    print("\nMost common incorrect top-1 tags on validation:")
    incorrect = val_summary_df[
        val_summary_df["gold_tag_is_top1"] == 0
    ]
    for label in ("RESP", "WIT"):
        subset = incorrect[incorrect["label"] == label]
        print(f"\n{label}:")
        if subset.empty:
            print("No incorrect top-1 tokens.")
        else:
            display(
                subset["top1_tag"]
                .value_counts()
                .head(10)
                .rename_axis("predicted_tag")
                .reset_index(name="token_count")
            )

print("\nRead-only analysis complete; no files or model weights changed.")


In [ ]:

# CS781 — Verify classifier dimensions and BIO label-ID consistency.
# Read-only; does not load, train, or overwrite any checkpoint.

import torch

print("=== BIO LABEL MAPPING ===")
print("Number of BIO labels:", len(BIO_LABELS))
print("Number of label2id entries:", len(label2id))

mapping_errors = []

for idx, label in enumerate(BIO_LABELS):
    if label2id.get(label) != idx:
        mapping_errors.append(
            f"BIO_LABELS[{idx}]={label!r}, "
            f"but label2id gives {label2id.get(label)!r}"
        )

if "id2label" in globals():
    for label, idx in label2id.items():
        if id2label.get(idx) != label:
            mapping_errors.append(
                f"id2label[{idx}]={id2label.get(idx)!r}, expected {label!r}"
            )

print("Mapping errors:", len(mapping_errors))
for error in mapping_errors:
    print("  ", error)

print("\n=== CHAMPION CLASSIFIER ===")
classifier = getattr(champion_model, "classifier", None)

if classifier is None:
    print("No attribute named 'classifier' on champion_model.")
    print("Top-level model modules:")
    print(champion_model)
else:
    print("Classifier type:", type(classifier).__name__)
    if hasattr(classifier, "weight"):
        print("Classifier weight shape:", tuple(classifier.weight.shape))
    if hasattr(classifier, "bias") and classifier.bias is not None:
        print("Classifier bias shape:", tuple(classifier.bias.shape))

print("\nExpected output classes:", len(BIO_LABELS))

if classifier is not None and hasattr(classifier, "weight"):
    out_features = classifier.weight.shape[0]
    print("Classifier output classes:", out_features)
    print("Output dimension matches mapping:", out_features == len(BIO_LABELS))

print("\n=== TARGET LABEL IDS ===")
for label in ("RESP", "WIT", "PREC", "APP", "STAT"):
    print(
        f"{label:8} B={label2id.get('B-' + label)}, "
        f"I={label2id.get('I-' + label)}"
    )

if mapping_errors:
    print("\nWARNING: label mapping inconsistency detected.")
else:
    print("\nBIO_LABELS and label2id are consistent.")


In [ ]:

# CS781 — Read-only checkpoint and training-configuration audit
import os
import inspect
import torch

print("=== CHECKPOINT FILES ===")
checkpoint_candidates = [
    "/kaggle/working/phase2_ner_corrected/best_ner_model.pt",
]

for path in checkpoint_candidates:
    print(f"{path}: exists={os.path.isfile(path)}")
    if os.path.isfile(path):
        print(f"  size={os.path.getsize(path) / (1024**2):.2f} MiB")

print("\n=== EXISTING TRAINING VARIABLES ===")
interesting = [
    "criterion", "loss_fn", "loss_function", "class_weights",
    "label_weights", "loss_weights", "optimizer", "scheduler",
    "training_args", "train_args", "config", "num_epochs",
    "epochs", "learning_rate", "lr", "ignore_index",
    "train_dataset", "train_loader", "pilot_windows",
]

for name in interesting:
    if name not in globals():
        continue

    value = globals()[name]
    print(f"\n{name}: type={type(value).__name__}")

    if name in ("criterion", "loss_fn", "loss_function"):
        print(value)
        try:
            print("forward signature:", inspect.signature(value.forward))
        except Exception:
            pass

    elif name in ("class_weights", "label_weights", "loss_weights"):
        try:
            print(value)
        except Exception as exc:
            print("Could not display:", exc)

    elif name in ("config", "training_args", "train_args"):
        try:
            print(value)
        except Exception as exc:
            print("Could not display:", exc)

    elif name in ("train_dataset", "train_loader"):
        try:
            print("length:", len(value))
        except Exception:
            pass

    elif name == "pilot_windows":
        print("windows:", len(value))
        try:
            print("first window keys:", list(value[0].keys()))
        except Exception:
            pass

print("\n=== CHECKPOINT METADATA ===")
path = checkpoint_candidates[0]

if os.path.isfile(path):
    # weights_only=True avoids unrestricted pickle loading where supported.
    try:
        checkpoint = torch.load(
            path, map_location="cpu", weights_only=True
        )
    except TypeError:
        checkpoint = torch.load(path, map_location="cpu")

    if isinstance(checkpoint, dict):
        print("Top-level keys:", list(checkpoint.keys()))

        for key, value in checkpoint.items():
            if key in (
                "epoch", "best_f1", "best_score", "val_f1",
                "learning_rate", "lr", "loss", "label2id",
                "id2label", "training_args", "config"
            ):
                if torch.is_tensor(value):
                    print(key, "tensor shape:", tuple(value.shape))
                else:
                    print(key, "=", value)

        for key in ("model_state_dict", "state_dict", "model"):
            if key in checkpoint and isinstance(checkpoint[key], dict):
                print(f"{key}: {len(checkpoint[key])} entries")
    else:
        print("Checkpoint object type:", type(checkpoint).__name__)

    del checkpoint

print("\nAudit complete. No model weights or files were changed.")


In [ ]:

# CS781 — Find the original training code in notebook input history.
# Read-only; does not execute any historical cells.

if "In" not in globals():
    print(
        "Notebook input history `In` is unavailable. "
        "Please share the original training cell instead."
    )
else:
    keywords = (
        "CrossEntropyLoss", "class_weights", "ignore_index",
        "loss.backward", "optimizer.step", "best_ner_model.pt",
        "load_state_dict", "label2id", "pilot_windows",
    )

    matches = []
    for i, source in enumerate(In):
        if not isinstance(source, str):
            continue
        found = [kw for kw in keywords if kw in source]
        if found:
            matches.append((i, found, source))

    print("Matching cells:", len(matches))
    for i, found, source in matches[-12:]:
        print(f"\n{'=' * 24} INPUT CELL {i} {'=' * 24}")
        print("Matched:", ", ".join(found))
        print(source[:12000])
        if len(source) > 12000:
            print("... output truncated ...")


In [ ]:

from pathlib import Path

roots = [Path("/kaggle/input"), Path("/kaggle/working")]
wanted = {
    "champion_recovery.pt",
    "best_ner_model.pt",
    "checkpoint_latest.pt",
    "targeted_pilot_8_steps.pt",
}

found = []
for root in roots:
    if root.exists():
        for path in root.rglob("*"):
            if path.is_file() and path.name in wanted:
                found.append(path)

print("=== CHECKPOINT SEARCH ===")
if not found:
    print("No matching checkpoints found in the searched directories.")
else:
    for path in sorted(set(found)):
        print(f"{path}  ({path.stat().st_size / (1024**2):.2f} MiB)")

print("\n=== RECOVERY PATH HELPER ===")
if "find_artifact" in globals():
    for filename in (
        "champion_recovery.pt",
        "checkpoint_latest.pt",
        "best_ner_model.pt",
    ):
        try:
            print(filename, "->", find_artifact(filename))
        except FileNotFoundError:
            print(filename, "-> NOT FOUND")
else:
    print("find_artifact is not defined in this runtime.")

print("\nRead-only search complete.")


In [ ]:

import torch

print("=== MODEL OBJECTS ===")
for name in ("champion_model", "targeted_model", "ner_model"):
    obj = globals().get(name)
    if obj is not None:
        print(f"{name}: {type(obj).__name__}, id={id(obj)}")

print("\n=== LOSS CONFIGURATIONS ===")
for name in ("criterion", "loss_fn", "loss_function",
             "targeted_criterion"):
    obj = globals().get(name)
    if obj is not None:
        print(f"\n{name}: {obj}")
        weight = getattr(obj, "weight", None)
        if weight is not None:
            weight = weight.detach().cpu()
            for tag in ("B-RESP", "I-RESP", "B-WIT", "I-WIT",
                        "B-PREC", "I-PREC"):
                if tag in label2id:
                    print(f"  {tag}: {weight[label2id[tag]].item():.3f}")
        print("ignore_index:", getattr(obj, "ignore_index", "not exposed"))

print("\n=== EXISTING TRAINING HISTORY ===")
for name in ("history", "loss_history", "train_losses", "val_losses"):
    if name in globals():
        value = globals()[name]
        print(f"{name}: {repr(value)[:3000]}")

print("\nNo training or checkpoint changes performed.")


In [ ]:

# Read-only audit of the pipeline used for prediction_champion_plus_rules.json

import inspect

for name in (
    "champion_model", "champ_thr", "predict_texts",
    "apply_thr", "merge_all", "keep", "chosen",
    "val_texts", "val_gold", "score", "as_scored"
):
    print(f"{name}: {'AVAILABLE' if name in globals() else 'MISSING'}")

print("\n=== GENERATION FUNCTIONS ===")
for name in ("predict_texts", "apply_thr", "merge_all"):
    fn = globals().get(name)
    if callable(fn):
        print(f"\n--- {name} ---")
        try:
            print(inspect.getsource(fn))
        except (OSError, TypeError):
            print("Source unavailable; signature:", inspect.signature(fn))

print("\n=== THRESHOLDS ===")
if "champ_thr" in globals():
    print(champ_thr)

print("\n=== SELECTED JUDGE PATTERNS ===")
if "keep" in globals():
    print(keep)

print("\n=== COUNSEL POLICY ===")
if "chosen" in globals():
    print(chosen)

print("\nNo predictions or checkpoint files changed.")


In [ ]:

# CS781 — Inspect prediction structure before evaluating
# Read-only: does not train or overwrite submission files.

raw_val = predict_texts(champion_model, val_texts)

print("raw_val type:", type(raw_val).__name__)

if isinstance(raw_val, dict):
    print("Document count:", len(raw_val))
    first_key = next(iter(raw_val), None)
    print("First document key:", repr(first_key))
    print(
        "First value type:",
        type(raw_val[first_key]).__name__ if first_key is not None else "empty"
    )
    print(
        "First value sample:",
        repr(raw_val[first_key])[:1000] if first_key is not None else "empty"
    )

elif isinstance(raw_val, list):
    print("List length:", len(raw_val))
    print("First item type:", type(raw_val[0]).__name__ if raw_val else "empty")
    print("First item sample:", repr(raw_val[0])[:1000] if raw_val else "empty")

else:
    print("Unexpected prediction structure:", repr(raw_val)[:1000])

print("\napply_thr signature:")
import inspect
print(inspect.signature(apply_thr))
print(inspect.getsource(apply_thr))


In [ ]:

# CS781 — Evaluate champion + rules without overwriting notebook variables.
# Read-only: no training, checkpoint changes, or submission writes.

assert callable(calculate_exact_span_metrics), (
    "calculate_exact_span_metrics is not available as a function."
)

# Generate fresh validation predictions and apply the established pipeline.
raw_val_check = predict_texts(champion_model, val_texts)
filtered_val_check = apply_thr(raw_val_check, champ_thr)
merged_val_check = merge_all(val_texts, filtered_val_check, keep)

# Calculate exact-span metrics using the notebook's evaluator.
metrics_check = calculate_exact_span_metrics(
    val_gold, merged_val_check, ENTITY_LABELS
)

print("=== CHAMPION + RULES — FOLD 2 VALIDATION ===")
print("Macro-F1:", metrics_check.get("MACRO_F1"))

for label in ENTITY_LABELS:
    if label not in metrics_check:
        continue
    m = metrics_check[label]
    print(
        f"{label:12} "
        f"F1={m.get('f1', 0):.4f} "
        f"P={m.get('precision', 0):.4f} "
        f"R={m.get('recall', 0):.4f} "
        f"TP={m.get('TP', 0)} "
        f"FP={m.get('FP', 0)} "
        f"FN={m.get('FN', 0)}"
    )

print("\nExisting submission file remains untouched.")


In [ ]:

# Read-only comparison of the uploaded/saved prediction file against
# freshly generated champion-plus-rules predictions.
# No output files are written.

import json
from pathlib import Path

saved_candidates = [
    Path("/kaggle/working/prediction_champion_plus_rules.json"),
    Path("/kaggle/working/prediction.json"),
]

saved_path = next((p for p in saved_candidates if p.is_file()), None)

print("=== SAVED PREDICTION FILE ===")
if saved_path is None:
    print("Neither expected file exists in /kaggle/working.")
    print("No file was created or changed.")
else:
    print("Found:", saved_path)
    with saved_path.open("r", encoding="utf-8") as f:
        saved = json.load(f)

    print("Type:", type(saved).__name__)
    if isinstance(saved, dict):
        print("Document count:", len(saved))
        first_id = next(iter(saved), None)
        print("First document ID:", repr(first_id))
        print("First value sample:", repr(saved.get(first_id))[:1000])

    print("\n=== FRESH GENERATION ===")
    print("Document count:", len(merged_val))
    print("Note: merged_val contains validation documents only.")
    print("Saved submission and validation predictions are not directly comparable")
    print("unless they contain the same document IDs and equivalent formats.")

print("\nRead-only comparison completed.")


In [ ]:

import json
from pathlib import Path
from collections import Counter

path = Path("/kaggle/working/prediction_champion_plus_rules.json")
with path.open("r", encoding="utf-8") as f:
    saved_predictions = json.load(f)

print("=== SAVED SUBMISSION AUDIT ===")
print("Documents:", len(saved_predictions))
print("Matches test document IDs:", set(saved_predictions) == set(test_texts))
print("Missing test IDs:", len(set(test_texts) - set(saved_predictions)))
print("Unexpected IDs:", len(set(saved_predictions) - set(test_texts)))

label_counts = Counter()
doc_counts = Counter()
invalid_spans = []

for doc_id, spans in saved_predictions.items():
    for span in spans:
        if not isinstance(span, (list, tuple)) or len(span) != 3:
            invalid_spans.append((doc_id, span, "expected [start, end, label]"))
            continue

        s, e, label = span
        label_counts[label] += 1
        doc_counts[label] += 1

        text = test_texts.get(doc_id)
        if text is None:
            continue
        if not (isinstance(s, int) and isinstance(e, int) and 0 <= s < e <= len(text)):
            invalid_spans.append((doc_id, span, "invalid character offsets"))
        elif label not in ENTITY_LABELS:
            invalid_spans.append((doc_id, span, "unknown label"))

print("\n=== ENTITY COUNTS ===")
for label in ENTITY_LABELS:
    print(f"{label:12} {label_counts[label]:5d}")

print("\nRESP spans:", label_counts["RESP"])
print("WIT spans:", label_counts["WIT"])
print("Invalid spans:", len(invalid_spans))
for item in invalid_spans[:10]:
    print("  ", item)

print("\n=== EXAMPLE RESP/WIT PREDICTIONS ===")
for label in ("RESP", "WIT"):
    examples = []
    for doc_id, spans in saved_predictions.items():
        for s, e, lab in spans:
            if lab == label:
                examples.append((doc_id, s, e, test_texts[doc_id][s:e]))
                if len(examples) >= 10:
                    break
        if len(examples) >= 10:
            break
    print(f"\n{label}:")
    print(examples if examples else "No predictions.")

print("\nRead-only audit complete; submission unchanged.")


In [ ]:

# CS781 — RESP/WIT raw-vs-thresholded audit on the actual test fold
# Does not write files or change the champion.

raw_test_audit = predict_texts(champion_model, test_texts)
filtered_test_audit = apply_thr(raw_test_audit, champ_thr)

print("=== TEST-FOLD RESP/WIT COUNTS ===")
for label in ("RESP", "WIT"):
    raw_spans = [
        (doc_id, s, e, conf)
        for doc_id, spans in raw_test_audit.items()
        for s, e, lab, conf in spans
        if lab == label
    ]

    kept_spans = [
        (doc_id, s, e)
        for doc_id, spans in filtered_test_audit.items()
        for s, e, lab in spans
        if lab == label
    ]

    print(f"\n{label}")
    print("Raw decoded spans:", len(raw_spans))
    print("Spans surviving current threshold:", len(kept_spans))
    print("Current threshold:", champ_thr[label])
    print("Highest raw confidence:", max((x[3] for x in raw_spans), default=None))
    print("Top 10 raw spans:", sorted(raw_spans, key=lambda x: x[3], reverse=True)[:10])

print("\nRead-only audit complete.")


In [ ]:

# CS781 — Controlled RESP/WIT recovery experiment
# Champion stays untouched. Candidate uses a separate model and optimizer.
# This cell does NOT write over any existing checkpoint or prediction file.

import copy
import gc
import os
import random
import torch
from torch.utils.data import Dataset, DataLoader

assert callable(make_unique_pilot_windows), "Corrected window builder missing."
assert "train_texts" in globals() and "train_gold" in globals()
assert "champion_model" in globals()
assert not champion_model.training, "Expected champion in eval mode."

SEED_RECOVERY = 781
RECOVERY_STEPS = 40
RECOVERY_ACCUM = 2
RECOVERY_LR = 1e-6
RECOVERY_BATCH_SIZE = 1
RECOVERY_DIR = "/kaggle/working/phase2_ner_resp_wit_recovery"
os.makedirs(RECOVERY_DIR, exist_ok=True)

random.seed(SEED_RECOVERY)
torch.manual_seed(SEED_RECOVERY)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_RECOVERY)

# Build corrected, uniquely supervised windows from fold 1 only.
recovery_windows, recovery_label_counts = make_unique_pilot_windows(
    train_texts, train_gold, tokenizer
)

print("Training windows:", len(recovery_windows))
for lab in ("RESP", "WIT"):
    n_b = recovery_label_counts.get(f"B-{lab}", 0)
    n_i = recovery_label_counts.get(f"I-{lab}", 0)
    print(f"{lab}: B={n_b}, I={n_i}, total={n_b+n_i}")

class RecoveryDataset(Dataset):
    def __init__(self, windows):
        self.windows = windows
    def __len__(self):
        return len(self.windows)
    def __getitem__(self, idx):
        return self.windows[idx]

def recovery_collate(batch):
    max_len = max(len(item["input_ids"]) for item in batch)
    ids_batch, masks_batch, labels_batch = [], [], []

    for item in batch:
        ids = item["input_ids"]
        labels = item["labels"]
        pad = max_len - len(ids)
        ids_batch.append(ids + [0] * pad)
        masks_batch.append([1] * len(ids) + [0] * pad)
        labels_batch.append(labels + [-100] * pad)

    return (
        torch.tensor(ids_batch, dtype=torch.long),
        torch.tensor(masks_batch, dtype=torch.long),
        torch.tensor(labels_batch, dtype=torch.long),
    )

recovery_generator = torch.Generator()
recovery_generator.manual_seed(SEED_RECOVERY)

recovery_loader = DataLoader(
    RecoveryDataset(recovery_windows),
    batch_size=RECOVERY_BATCH_SIZE,
    shuffle=True,
    collate_fn=recovery_collate,
    generator=recovery_generator,
)

# Independent model copy: never train champion_model directly.
recovery_model = copy.deepcopy(champion_model).to(device)
recovery_model.train()

recovery_optimizer = torch.optim.AdamW(
    recovery_model.parameters(),
    lr=RECOVERY_LR,
    weight_decay=0.01,
)

# Apply extra weight only to RESP/WIT; retain ordinary weights for other labels.
recovery_weights = torch.ones(len(BIO_LABELS), dtype=torch.float32, device=device)
recovery_weights[label2id["O"]] = 0.5

for lab in ("RESP", "WIT"):
    recovery_weights[label2id[f"B-{lab}"]] = 4.0
    recovery_weights[label2id[f"I-{lab}"]] = 4.0

recovery_criterion = torch.nn.CrossEntropyLoss(
    weight=recovery_weights,
    ignore_index=-100,
)

recovery_scaler = (
    torch.amp.GradScaler("cuda")
    if torch.cuda.is_available()
    else None
)

recovery_losses = []
recovery_optimizer_steps = 0
recovery_failed = False
recovery_iter = iter(recovery_loader)

try:
    while recovery_optimizer_steps < RECOVERY_STEPS:
        recovery_optimizer.zero_grad(set_to_none=True)
        batch_losses = []

        for _ in range(RECOVERY_ACCUM):
            try:
                batch = next(recovery_iter)
            except StopIteration:
                recovery_iter = iter(recovery_loader)
                batch = next(recovery_iter)

            input_ids, attention_mask, labels = [
                tensor.to(device) for tensor in batch
            ]

            if torch.cuda.is_available():
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    logits = recovery_model(input_ids, attention_mask)
                    loss = recovery_criterion(
                        logits.reshape(-1, len(BIO_LABELS)),
                        labels.reshape(-1),
                    )
            else:
                logits = recovery_model(input_ids, attention_mask)
                loss = recovery_criterion(
                    logits.reshape(-1, len(BIO_LABELS)),
                    labels.reshape(-1),
                )

            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite loss; aborting recovery run.")

            scaled_loss = loss / RECOVERY_ACCUM
            if recovery_scaler is not None:
                recovery_scaler.scale(scaled_loss).backward()
            else:
                scaled_loss.backward()

            batch_losses.append(float(loss.detach().cpu()))
            del input_ids, attention_mask, labels, logits, loss, scaled_loss

        if recovery_scaler is not None:
            recovery_scaler.unscale_(recovery_optimizer)

        grad_norm = torch.nn.utils.clip_grad_norm_(
            recovery_model.parameters(), max_norm=1.0
        )
        if not torch.isfinite(grad_norm):
            raise RuntimeError("Non-finite gradient norm; aborting recovery run.")

        if recovery_scaler is not None:
            recovery_scaler.step(recovery_optimizer)
            recovery_scaler.update()
        else:
            recovery_optimizer.step()

        recovery_optimizer_steps += 1
        mean_loss = sum(batch_losses) / len(batch_losses)
        recovery_losses.append(mean_loss)

        if recovery_optimizer_steps == 1 or recovery_optimizer_steps % 5 == 0:
            print(
                f"Step {recovery_optimizer_steps}/{RECOVERY_STEPS} | "
                f"loss={mean_loss:.4f} | "
                f"grad_norm={float(grad_norm):.3f}"
            )

except Exception as exc:
    recovery_failed = True
    print("RECOVERY TRAINING ABORTED:", repr(exc))

finally:
    recovery_optimizer.zero_grad(set_to_none=True)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

recovery_model.eval()

print("\n=== RECOVERY RUN SUMMARY ===")
print("Completed steps:", recovery_optimizer_steps)
print("Failed:", recovery_failed)
print("First loss:", recovery_losses[0] if recovery_losses else None)
print("Last loss:", recovery_losses[-1] if recovery_losses else None)
print("Champion still in eval mode:", not champion_model.training)
print("Candidate is separate:", recovery_model is not champion_model)

if recovery_failed or recovery_optimizer_steps != RECOVERY_STEPS:
    raise RuntimeError(
        "Recovery run did not complete. Do not evaluate or save this candidate."
    )

print("\nCandidate trained in memory only. No checkpoint saved yet.")


In [ ]:

# CS781 — Exact-span validation comparison
# Does not change either model or write prediction files.

def evaluate_champion_rules(model, texts, gold):
    raw = predict_texts(model, texts)
    filtered = apply_thr(raw, champ_thr)
    merged = merge_all(texts, filtered, keep)
    metrics = calculate_exact_span_metrics(gold, merged, ENTITY_LABELS)
    return metrics, merged

baseline_metrics, baseline_val_predictions = evaluate_champion_rules(
    champion_model, val_texts, val_gold
)
candidate_metrics, candidate_val_predictions = evaluate_champion_rules(
    recovery_model, val_texts, val_gold
)

print("=== MACRO F1 ===")
print("Champion:", round(baseline_metrics["MACRO_F1"], 6))
print("Candidate:", round(candidate_metrics["MACRO_F1"], 6))
print("Delta:", round(
    candidate_metrics["MACRO_F1"] - baseline_metrics["MACRO_F1"], 6
))

print("\n=== PER-LABEL COMPARISON ===")
for lab in ENTITY_LABELS:
    b = baseline_metrics[lab]
    c = candidate_metrics[lab]
    print(
        f"{lab:12} "
        f"champion={b['f1']:.4f} "
        f"candidate={c['f1']:.4f} "
        f"delta={c['f1']-b['f1']:+.4f} "
        f"candidate_TP={c['TP']} "
        f"FP={c['FP']} FN={c['FN']}"
    )

print("\n=== RESP/WIT RAW DECODE COUNTS ===")
candidate_raw_val = predict_texts(recovery_model, val_texts)
for lab in ("RESP", "WIT"):
    count = sum(
        1 for spans in candidate_raw_val.values()
        for _, _, predicted_lab, _ in spans
        if predicted_lab == lab
    )
    print(f"{lab}: {count} raw decoded spans")

print("\nExisting champion and submission remain untouched.")


In [ ]:

# CS781 — Conservative acceptance gate; never overwrite champion or submission.

from pathlib import Path
import torch

macro_delta = (
    candidate_metrics["MACRO_F1"] - baseline_metrics["MACRO_F1"]
)

target_deltas = {
    lab: candidate_metrics[lab]["f1"] - baseline_metrics[lab]["f1"]
    for lab in ("RESP", "WIT")
}

other_deltas = {
    lab: candidate_metrics[lab]["f1"] - baseline_metrics[lab]["f1"]
    for lab in ENTITY_LABELS if lab not in ("RESP", "WIT")
}

passes_gate = (
    macro_delta > 0
    and max(target_deltas.values()) > 0
    and min(other_deltas.values(), default=0) >= -0.03
)

candidate_path = Path(
    "/kaggle/working/phase2_ner_resp_wit_recovery/"
    "resp_wit_recovery_candidate.pt"
)

print("Macro-F1 delta:", round(macro_delta, 6))
print("RESP/WIT F1 deltas:", target_deltas)
print("Worst non-target label delta:", round(
    min(other_deltas.values(), default=0), 6
))
print("Acceptance gate:", "PASS" if passes_gate else "REJECT")

if passes_gate:
    if candidate_path.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing candidate: {candidate_path}"
        )

    torch.save({
        "model_state_dict": {
            k: v.detach().cpu()
            for k, v in recovery_model.state_dict().items()
        },
        "bio_labels": BIO_LABELS,
        "optimizer_steps": recovery_optimizer_steps,
        "loss_history": recovery_losses,
        "validation_macro_f1": candidate_metrics["MACRO_F1"],
        "baseline_macro_f1": baseline_metrics["MACRO_F1"],
        "validation_resp_wit_f1": target_deltas,
    }, candidate_path)

    print("Candidate saved separately:", candidate_path)
else:
    print("Candidate rejected. No checkpoint saved.")
    print("Keep the original champion and submission unchanged.")


In [ ]:

# CS781 — Read-only audit of RESP/WIT training-window labels

from collections import Counter, defaultdict

assert "recovery_windows" in globals()
assert "train_texts" in globals() and "train_gold" in globals()
assert "BIO_LABELS" in globals() and "label2id" in globals()

print("=== LABEL-ID CHECK ===")
for lab in ("RESP", "WIT", "PREC", "APP", "STAT"):
    print(
        f"{lab:8} B={label2id['B-' + lab]} "
        f"I={label2id['I-' + lab]}"
    )

# Count actual labels represented in the windows.
window_label_counts = Counter()
ignored_positions = 0
all_positions = 0

for win in recovery_windows:
    ids = win["input_ids"]
    labels = win["labels"]

    if len(ids) != len(labels):
        raise ValueError(
            f"Window length mismatch: ids={len(ids)}, labels={len(labels)}"
        )

    for tag_id in labels:
        all_positions += 1
        if int(tag_id) == -100:
            ignored_positions += 1
        elif 0 <= int(tag_id) < len(BIO_LABELS):
            window_label_counts[BIO_LABELS[int(tag_id)]] += 1
        else:
            raise ValueError(f"Unexpected target ID: {tag_id}")

print("\n=== TARGET COUNTS ACTUALLY STORED IN WINDOWS ===")
for tag in ("B-RESP", "I-RESP", "B-WIT", "I-WIT",
            "B-PREC", "I-PREC", "B-APP", "I-APP"):
    print(f"{tag:10} {window_label_counts[tag]}")

print("\nTotal positions:", all_positions)
print("Ignored (-100) positions:", ignored_positions)
print("Supervised positions:", all_positions - ignored_positions)

print("\n=== COUNTS REPORTED BY WINDOW BUILDER ===")
print(recovery_label_counts)

print("\n=== TRAINING GOLD COUNTS ===")
for lab in ("RESP", "WIT"):
    spans = [
        ent for doc_id in train_gold
        for ent in train_gold[doc_id]
        if len(ent) >= 3 and ent[2] == lab
    ]
    print(f"{lab}: {len(spans)} spans")

print("\nRead-only audit complete.")


In [ ]:

# CS781 — Compare expected gold BIO targets with stored window targets
# Read-only: does not alter recovery_windows or any model.

from collections import Counter, defaultdict
from bisect import bisect_left, bisect_right

assert "tokenizer" in globals()
assert "recovery_windows" in globals()

# Map each document's window token positions to stored labels.
windows_by_doc = defaultdict(list)
for win in recovery_windows:
    required_keys = {"doc_id", "start", "input_ids", "labels"}
    if not required_keys.issubset(win):
        raise RuntimeError(
            "Unexpected window format. Found keys: " + str(list(win.keys()))
        )
    windows_by_doc[win["doc_id"]].append(win)

results = Counter()
examples = []

for doc_id, text in train_texts.items():
    enc = tokenizer.encode(text, add_special_tokens=False)
    offsets = list(enc.offsets)
    starts = [s for s, e in offsets]
    ends = [e for s, e in offsets]

    # Rebuild the intended BIO tags exactly as in the training annotation
    # convention: first overlapping token gets B, remaining tokens get I.
    expected = {}
    for ent in train_gold[doc_id]:
        if len(ent) < 3:
            continue
        s, e, lab = int(ent[0]), int(ent[1]), str(ent[2])
        if lab not in ("RESP", "WIT"):
            continue

        lo = bisect_right(ends, s)
        hi = bisect_left(starts, e)
        overlap = [
            i for i in range(lo, hi)
            if ends[i] > s and starts[i] < e
        ]

        if not overlap:
            results[(lab, "no_token_overlap")] += 1
            continue

        for j, token_idx in enumerate(overlap):
            tag = ("B-" if j == 0 else "I-") + lab
            if token_idx in expected and expected[token_idx] != tag:
                results[(lab, "overlapping_gold_conflict")] += 1
            expected[token_idx] = tag

    # Reconstruct the global token labels from the unique windows.
    stored = {}
    for win in windows_by_doc.get(doc_id, []):
        start = int(win["start"])
        labels = win["labels"]
        for local_idx, raw_tag in enumerate(labels):
            token_idx = start + local_idx
            raw_tag = int(raw_tag)

            # Ignore special/padding positions and labels outside the
            # original document's tokenizer token sequence.
            if token_idx < 0 or token_idx >= len(offsets) or raw_tag == -100:
                continue

            tag = BIO_LABELS[raw_tag]
            if token_idx in stored and stored[token_idx] != tag:
                results[("ALL", "duplicate_conflicting_window_target")] += 1
            stored[token_idx] = tag

    for token_idx, expected_tag in expected.items():
        actual_tag = stored.get(token_idx)
        lab = expected_tag[2:]

        if actual_tag is None:
            results[(lab, "missing_stored_target")] += 1
            if len(examples) < 12:
                examples.append(
                    (doc_id, token_idx, expected_tag, actual_tag,
                     text[offsets[token_idx][0]:offsets[token_idx][1]])
                )
        elif actual_tag != expected_tag:
            results[(lab, "target_mismatch")] += 1
            if len(examples) < 12:
                s, e = offsets[token_idx]
                examples.append(
                    (doc_id, token_idx, expected_tag, actual_tag, text[s:e])
                )
        else:
            results[(lab, "target_match")] += 1

print("=== TARGET RECONSTRUCTION RESULTS ===")
for key, value in sorted(results.items()):
    print(f"{key}: {value}")

print("\n=== FIRST MISMATCH EXAMPLES ===")
for item in examples:
    print(item)

print("\nRead-only audit complete.")


In [ ]:

# CS781 — Read-only RESP/WIT gold-token probability comparison (FIXED)

import torch
import numpy as np
from collections import defaultdict

assert "champion_model" in globals()
assert "recovery_model" in globals()
assert "recovery_windows" in globals()
assert "BIO_LABELS" in globals()

target_ids = {
    "RESP": (BIO_LABELS.index("B-RESP"), BIO_LABELS.index("I-RESP")),
    "WIT":  (BIO_LABELS.index("B-WIT"),  BIO_LABELS.index("I-WIT")),
}

def _get_logits_tensor(model, input_ids):
    device = next(model.parameters()).device
    ids = torch.as_tensor(
        input_ids, dtype=torch.long, device=device
    ).unsqueeze(0)
    attention = torch.ones_like(ids)

    with torch.no_grad():
        out = model(input_ids=ids, attention_mask=attention)

    # Some custom models return logits directly as a Tensor.
    logits = out.logits if hasattr(out, "logits") else out

    if not isinstance(logits, torch.Tensor):
        raise TypeError(f"Unexpected model output type: {type(logits)}")

    if logits.ndim == 3:
        logits = logits[0]
    elif logits.ndim != 2:
        raise ValueError(f"Unexpected logits shape: {tuple(logits.shape)}")

    return logits.float().softmax(dim=-1).cpu().numpy()

def _compare_gold_token_probs(model, model_name):
    rows = defaultdict(list)
    model.eval()

    for win in recovery_windows:
        ids = win["input_ids"]
        labels = win["labels"]
        probs = _get_logits_tensor(model, ids)

        if len(probs) != len(labels):
            raise ValueError(
                f"Logit/label length mismatch: {len(probs)} vs {len(labels)}"
            )

        for i, raw_label in enumerate(labels):
            raw_label = int(raw_label)
            if raw_label < 0 or raw_label >= len(BIO_LABELS):
                continue

            tag = BIO_LABELS[raw_label]
            if tag not in ("B-RESP", "I-RESP", "B-WIT", "I-WIT"):
                continue

            lab = tag[2:]
            p = probs[i]
            b_id, i_id = target_ids[lab]

            rows[(lab, tag)].append((
                float(p[raw_label]),
                float(p[b_id] + p[i_id]),
                BIO_LABELS[int(np.argmax(p))] == tag
            ))

    print(f"\n=== {model_name} ===")
    for lab in ("RESP", "WIT"):
        for tag in ("B-" + lab, "I-" + lab):
            vals = rows[(lab, tag)]
            if not vals:
                print(f"{tag}: NO TARGETS FOUND")
                continue

            p_tag = np.array([x[0] for x in vals])
            p_entity = np.array([x[1] for x in vals])
            top1 = np.mean([x[2] for x in vals])

            print(
                f"{tag}: n={len(vals)}, "
                f"mean P(correct tag)={p_tag.mean():.4f}, "
                f"median={np.median(p_tag):.4f}, "
                f"mean P(B+I)={p_entity.mean():.4f}, "
                f"correct-tag top1={top1:.2%}"
            )

_compare_gold_token_probs(champion_model, "CHAMPION")
_compare_gold_token_probs(recovery_model, "RECOVERY CANDIDATE")


In [ ]:

# CS781 — WIT/RESP annotation distribution and surface-form audit
# Read-only. Does not alter models or files.

from collections import Counter, defaultdict

assert "train_texts" in globals() and "train_gold" in globals()

for target_lab in ("RESP", "WIT"):
    per_doc = Counter()
    surfaces = Counter()
    examples = defaultdict(list)

    for doc_id, entities in train_gold.items():
        text = train_texts[doc_id]

        for ent in entities:
            if len(ent) < 3:
                continue

            s, e, lab = int(ent[0]), int(ent[1]), str(ent[2])
            if lab != target_lab:
                continue

            per_doc[doc_id] += 1
            surface = text[s:e]
            surfaces[surface] += 1

            if len(examples[surface]) < 3:
                examples[surface].append((doc_id, s, e))

    total = sum(per_doc.values())
    print(f"\n{'=' * 18} {target_lab} {'=' * 18}")
    print("Total spans:", total)
    print("Documents containing label:", len(per_doc))
    print("Largest document share:", round(max(per_doc.values()) / total, 3) if total else 0)
    print("Most frequent surfaces:")
    for surface, count in surfaces.most_common(20):
        print(f"{count:4} | {surface!r} | examples={examples[surface][:2]}")

    print("Per-document span counts:")
    for doc_id, count in per_doc.most_common():
        print(f"{doc_id}: {count}")


In [ ]:

# CS781 — RESP/WIT threshold sweep on fold2
# Inference-only: no training, no checkpoint writes, no submission overwrite.

import copy
import numpy as np

assert "champion_model" in globals()
assert "val_texts" in globals() and "val_gold" in globals()
assert "champ_thr" in globals()
assert "predict_texts" in globals()
assert "apply_thr" in globals()
assert "merge_all" in globals()
assert "calculate_exact_span_metrics" in globals()
assert "ENTITY_LABELS" in globals()

# Predict once, then reuse raw predictions for every threshold combination.
sweep_raw_resp_wit = predict_texts(champion_model, val_texts)

sweep_grid = [0.01, 0.015, 0.02, 0.03, 0.04, 0.06, 0.08, 0.10, 0.15, 0.20]

sweep_rows = []

for resp_thr in sweep_grid:
    for wit_thr in sweep_grid:
        local_thr = copy.deepcopy(champ_thr)
        local_thr["RESP"] = resp_thr
        local_thr["WIT"] = wit_thr

        local_filtered = apply_thr(sweep_raw_resp_wit, local_thr)
        local_merged = merge_all(val_texts, local_filtered, keep)

        local_metrics = calculate_exact_span_metrics(
            val_gold, local_merged, ENTITY_LABELS
        )

        resp_f1 = local_metrics["RESP"]["F1"]
        wit_f1 = local_metrics["WIT"]["F1"]
        macro_f1 = local_metrics["MACRO_F1"]

        sweep_rows.append({
            "RESP_thr": resp_thr,
            "WIT_thr": wit_thr,
            "RESP_F1": resp_f1,
            "WIT_F1": wit_f1,
            "Target_avg_F1": (resp_f1 + wit_f1) / 2,
            "Macro_F1": macro_f1,
            "RESP_pred": sum(
                1 for spans in local_merged.values()
                for span in spans if span[2] == "RESP"
            ),
            "WIT_pred": sum(
                1 for spans in local_merged.values()
                for span in spans if span[2] == "WIT"
            ),
        })

sweep_rows.sort(
    key=lambda r: (r["Target_avg_F1"], r["Macro_F1"]),
    reverse=True
)

print("=== TOP 15 SETTINGS BY RESP/WIT AVERAGE F1 ===")
for row in sweep_rows[:15]:
    print(row)

print("\n=== BEST MACRO-F1 SETTINGS ===")
for row in sorted(sweep_rows, key=lambda r: r["Macro_F1"], reverse=True)[:5]:
    print(row)

print("\nBaseline thresholds:", champ_thr["RESP"], champ_thr["WIT"])


In [ ]:

# Inspect the actual metric dictionary schema from the existing evaluator
_sweep_probe_thr = dict(champ_thr)
_sweep_probe_filtered = apply_thr(sweep_raw_resp_wit, _sweep_probe_thr)
_sweep_probe_merged = merge_all(val_texts, _sweep_probe_filtered, keep)
_sweep_probe_metrics = calculate_exact_span_metrics(
    val_gold, _sweep_probe_merged, ENTITY_LABELS
)

print("Top-level keys:", list(_sweep_probe_metrics.keys()))
print("\nRESP metrics:", _sweep_probe_metrics.get("RESP"))
print("\nWIT metrics:", _sweep_probe_metrics.get("WIT"))
print("\nMacro metric:", _sweep_probe_metrics.get("MACRO_F1"))


In [ ]:

# CS781 — Corrected RESP/WIT threshold sweep
# Reuses cached raw predictions. No training or file writes.

import copy

sweep_grid_fixed = [
    0.001, 0.003, 0.005, 0.01, 0.015,
    0.02, 0.03, 0.04, 0.06, 0.08, 0.10,
    0.15, 0.20, 0.30, 0.40, 0.45
]

sweep_rows_fixed = []

for resp_thr in sweep_grid_fixed:
    for wit_thr in sweep_grid_fixed:
        local_thr = copy.deepcopy(champ_thr)
        local_thr["RESP"] = resp_thr
        local_thr["WIT"] = wit_thr

        local_filtered = apply_thr(sweep_raw_resp_wit, local_thr)
        local_merged = merge_all(val_texts, local_filtered, keep)

        local_metrics = calculate_exact_span_metrics(
            val_gold, local_merged, ENTITY_LABELS
        )

        resp_m = local_metrics["RESP"]
        wit_m = local_metrics["WIT"]

        sweep_rows_fixed.append({
            "RESP_thr": resp_thr,
            "WIT_thr": wit_thr,
            "RESP_f1": resp_m["f1"],
            "WIT_f1": wit_m["f1"],
            "target_avg_f1": (resp_m["f1"] + wit_m["f1"]) / 2,
            "macro_f1": local_metrics["MACRO_F1"],
            "RESP_pred": resp_m["pred"],
            "WIT_pred": wit_m["pred"],
            "RESP_FP": resp_m["FP"],
            "WIT_FP": wit_m["FP"],
        })

print("=== TOP SETTINGS BY RESP/WIT AVERAGE F1 ===")
for row in sorted(
    sweep_rows_fixed,
    key=lambda r: (r["target_avg_f1"], r["macro_f1"]),
    reverse=True
)[:15]:
    print(row)

print("\n=== TOP SETTINGS BY MACRO-F1 ===")
for row in sorted(
    sweep_rows_fixed,
    key=lambda r: r["macro_f1"],
    reverse=True
)[:10]:
    print(row)

print("\n=== SETTINGS THAT PRODUCE ANY RESP OR WIT SPANS ===")
nonzero = [
    r for r in sweep_rows_fixed
    if r["RESP_pred"] > 0 or r["WIT_pred"] > 0
]
print("Count:", len(nonzero))
for row in sorted(
    nonzero,
    key=lambda r: (r["target_avg_f1"], r["macro_f1"]),
    reverse=True
)[:20]:
    print(row)


In [ ]:

# CS781 — Raw decoder audit before thresholding
from collections import Counter, defaultdict

raw_counts = Counter()
raw_examples = defaultdict(list)

for doc_id, spans in sweep_raw_resp_wit.items():
    text = val_texts[doc_id]
    for item in spans:
        s, e, lab, conf = item[:4]
        if lab in ("RESP", "WIT"):
            raw_counts[lab] += 1
            if len(raw_examples[lab]) < 20:
                raw_examples[lab].append({
                    "doc_id": doc_id,
                    "span": text[int(s):int(e)],
                    "start": int(s),
                    "end": int(e),
                    "confidence": float(conf),
                })

print("=== RAW DECODED SPAN COUNTS ===")
for lab in ("RESP", "WIT"):
    print(f"{lab}: {raw_counts[lab]}")
    for ex in raw_examples[lab]:
        print(" ", ex)

print("\n=== RAW SPAN COUNTS FOR ALL LABELS ===")
print(Counter(
    item[2]
    for spans in sweep_raw_resp_wit.values()
    for item in spans
))


In [ ]:

# CS781 — Token-level argmax diagnostic on validation documents
# Uses the champion model; inference only.

import torch
from collections import Counter

def _get_logits_for_text(text, model):
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=False,
        return_offsets_mapping=True,
    )
    offsets = enc.pop("offset_mapping")[0].tolist()
    device = next(model.parameters()).device
    model_inputs = {
        k: v.to(device)
        for k, v in enc.items()
        if isinstance(v, torch.Tensor)
    }

    model.eval()
    with torch.no_grad():
        out = model(**model_inputs)

    logits = out.logits if hasattr(out, "logits") else out
    if logits.ndim == 3:
        logits = logits[0]

    probs = logits.float().softmax(-1).cpu()
    return offsets, probs

argmax_counts = Counter()
target_prob_samples = defaultdict(list)
top_competitors = defaultdict(Counter)

for doc_id, text in val_texts.items():
    offsets, probs = _get_logits_for_text(text, champion_model)

    for i, (s, e) in enumerate(offsets):
        if e <= s:
            continue

        row = probs[i]
        top_id = int(row.argmax())
        top_tag = BIO_LABELS[top_id]
        argmax_counts[top_tag] += 1

        for lab in ("RESP", "WIT"):
            b_id = BIO_LABELS.index("B-" + lab)
            i_id = BIO_LABELS.index("I-" + lab)
            target_prob_samples[lab].append(
                (float(row[b_id] + row[i_id]), top_tag)
            )

            if top_tag in ("B-" + lab, "I-" + lab):
                top_competitors[lab][top_tag] += 1

print("=== TOKEN ARGMAX TAG COUNTS ===")
for tag, count in argmax_counts.most_common():
    if tag in ("B-RESP", "I-RESP", "B-WIT", "I-WIT", "O",
               "B-PREC", "I-PREC", "B-STAT", "I-STAT", "B-APP", "I-APP"):
        print(tag, count)

print("\n=== ENTITY-TAG ARGMAX COUNTS ===")
for lab in ("RESP", "WIT"):
    print(lab, dict(top_competitors[lab]))
    vals = target_prob_samples[lab]
    print(
        "Mean B+I probability:",
        round(sum(x[0] for x in vals) / max(1, len(vals)), 5)
    )
    print("Most common competing tags:", Counter(x[1] for x in vals).most_common(8))


In [ ]:

# CS781 — Find tokenizer provenance and inference conventions
import inspect

print("=== TOKENIZER OBJECT ===")
print("Class:", type(tokenizer))
print("Vocab size:", tokenizer.get_vocab_size())
print("Padding:", tokenizer.padding)
print("Truncation:", tokenizer.truncation)

print("\n=== TOKENIZER-RELATED GLOBALS ===")
for name, obj in list(globals().items()):
    if "token" in name.lower() and not name.startswith("_"):
        if isinstance(obj, (str, int, bool, type(None))):
            print(name, repr(obj)[:250])
        elif hasattr(obj, "get_vocab_size"):
            try:
                print(name, type(obj), "vocab_size=", obj.get_vocab_size())
            except Exception:
                print(name, type(obj))

print("\n=== MODEL FORWARD SIGNATURE ===")
print(inspect.signature(champion_model.forward))

print("\n=== INFERENCE FUNCTIONS ===")
for name in ("predict_texts", "diag_doc_probs", "bio_viterbi_decode", "decode_spans"):
    fn = globals().get(name)
    print(f"\n--- {name} ---")
    if fn is None:
        print("Not found")
        continue
    try:
        print(inspect.getsource(fn))
    except (OSError, TypeError) as exc:
        print("Source unavailable:", repr(exc))

print("\n=== TOKENIZER VOCAB SAMPLE ===")
vocab = tokenizer.get_vocab()
for token, idx in sorted(vocab.items(), key=lambda kv: kv[1])[:30]:
    print(idx, repr(token))


In [ ]:

# CS781 — Gold RESP/WIT probabilities with the established sliding-window inference

import numpy as np
from collections import defaultdict
from bisect import bisect_right, bisect_left

assert "diag_doc_probs" in globals()
assert "val_texts" in globals() and "val_gold" in globals()

def _gold_span_token_diagnostic(model, model_name):
    per_label = defaultdict(list)
    results = defaultdict(lambda: {
        "spans": 0, "overlap_tokens": 0, "gold_B_probs": [],
        "gold_I_probs": [], "gold_B_top1": 0, "gold_I_top1": 0
    })

    for doc_id, text in val_texts.items():
        offsets, probs = diag_doc_probs(model, text)
        starts = [s for s, e in offsets]
        ends = [e for s, e in offsets]

        for ent in val_gold[doc_id]:
            if len(ent) < 3:
                continue
            s, e, lab = int(ent[0]), int(ent[1]), str(ent[2])
            if lab not in ("RESP", "WIT"):
                continue

            token_ids = [
                i for i in range(bisect_right(ends, s), bisect_left(starts, e))
                if ends[i] > s and starts[i] < e
            ]
            if not token_ids:
                continue

            results[lab]["spans"] += 1
            results[lab]["overlap_tokens"] += len(token_ids)

            for j, tok_i in enumerate(token_ids):
                tag = ("B-" if j == 0 else "I-") + lab
                tag_id = BIO_LABELS.index(tag)
                p = probs[tok_i]
                results[lab]["gold_B_probs" if j == 0 else "gold_I_probs"].append(
                    float(p[tag_id])
                )
                if int(p.argmax()) == tag_id:
                    results[lab]["gold_B_top1" if j == 0 else "gold_I_top1"] += 1

    print(f"\n=== {model_name} ===")
    for lab in ("RESP", "WIT"):
        r = results[lab]
        print(f"\n{lab}: spans={r['spans']}, overlapping tokens={r['overlap_tokens']}")
        for kind in ("B", "I"):
            vals = r[f"gold_{kind}_probs"]
            if vals:
                top1 = r[f"gold_{kind}_top1"] / len(vals)
                print(
                    f"  {kind}: n={len(vals)}, mean={np.mean(vals):.5f}, "
                    f"median={np.median(vals):.5f}, gold-tag top1={top1:.2%}"
                )
            else:
                print(f"  {kind}: no tokens")

_gold_span_token_diagnostic(champion_model, "CHAMPION")
_gold_span_token_diagnostic(recovery_model, "RECOVERY CANDIDATE")


In [ ]:

# CS781 — Read-only training-configuration audit
# Does not train models, alter checkpoints, or write files.

import inspect
import torch

print("=== TRAINING-RELATED GLOBALS ===")
keywords = (
    "optimizer", "criterion", "loss", "weight", "train",
    "batch", "window", "scheduler", "lr", "epoch", "step"
)

for name, obj in list(globals().items()):
    if name.startswith("_") or not any(k in name.lower() for k in keywords):
        continue

    if isinstance(obj, torch.optim.Optimizer):
        print(name, type(obj).__name__, "param_groups=", [
            {k: v for k, v in group.items() if k != "params"}
            for group in obj.param_groups
        ])
    elif isinstance(obj, torch.nn.Module):
        print(name, type(obj).__name__, repr(obj)[:500])
    elif isinstance(obj, (int, float, str, bool, type(None))):
        print(name, repr(obj)[:250])
    elif isinstance(obj, (list, tuple)) and len(obj) <= 20:
        print(name, repr(obj)[:500])

print("\n=== AVAILABLE TRAINING FUNCTIONS ===")
for name, obj in list(globals().items()):
    if callable(obj) and any(
        word in name.lower()
        for word in ("train", "fit", "loss", "window", "batch")
    ):
        print("\n---", name, "---")
        try:
            print(inspect.getsource(obj)[:5000])
        except (OSError, TypeError):
            print("Source unavailable")

print("\n=== CHAMPION CHECKPOINT METADATA ===")
checkpoint_path = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-working-backup-20261010-215243/"
    "cs781_recovery/champion_recovery.pt"
)

checkpoint = torch.load(
    checkpoint_path, map_location="cpu", weights_only=False
)

print("Checkpoint type:", type(checkpoint))
if isinstance(checkpoint, dict):
    print("Top-level keys:", list(checkpoint.keys()))
    for key, value in checkpoint.items():
        if key != "model_state_dict":
            if isinstance(value, torch.Tensor):
                print(key, "tensor", tuple(value.shape))
            else:
                print(key, repr(value)[:1000])
